# Record linkage [B]: extracted entity records against a watchlist

Two spreadsheets in (extracted entity records, and a watchlist), one Excel workbook out: for
each extracted party, how likely it is on the watchlist, which watchlist rows it could be,
field-by-field evidence, and what every number rests on. No text is read; relationships come
only from rows and shared details. Design: [DESIGN.md](DESIGN.md); build plan:
[PLAN.md](PLAN.md). Everything is in this notebook; data files stay files (`mappings/`,
`reference/`, and the gitignored `data/`, `out/`, `review/`).

**Run.** `RL_DATASET=synthetic|leie|scale|files` (default `synthetic`), then run all cells.
`python run_notebook_check.py` runs every cell on the synthetic set and exits non-zero on any
error or failed self-test.

**Sections.** Matching core v1.0 (shared with goko-v2-poc) · 0 Setup · 1 Mappings,
reference tables, inputs · 2 Normalize · 3 Breakdown · 4 Value index · 5 Category · 6 Outside
rarity and party frames · 7 Candidates · Noise engine · Synthetic data · LEIE exporter ·
8-11 Comparisons, u, m, prior · 12 Score · 13 Roll-up · 14 Evidence · 15 Workbook and manifest ·
16 Diagnostics · Edge-case check · Pipeline function · Run · 17 Self-tests · Acceptance ·
18 Review sample (later phase: skeleton only).

**Rules this notebook keeps** (project_v0.2/AGENTS.md): every link carries a score and a basis,
name-only links are visible and never upgraded, conflicting identifiers veto (rule 5); every
gate has an unresolved outcome: an extracted party with no candidate is shown as such, empty
rows and dropped oversized blocking keys are listed (rule 11); truth files are read only by
diagnostics and self-tests, never by the pipeline (rule 13).


## Matching core v1.0

**This section is the matching core.** It is identical code in [B] (this notebook) and,
later, in [A] (goko-v2-poc). The self-test section hashes every code cell between this
heading and the "End of the matching core" marker and compares the hash with the recorded
value, so any drift between the two systems is caught. Change it only deliberately: bump
`CORE_VERSION`, re-record the hash, and copy the section to [A].

**Contract.** Plain tables in, plain tables out. Nothing here reads a file, writes a
spreadsheet or knows about note text. The caller builds a *party frame* per side, one row per
party, with these columns (empty string when unknown):

| Column | Meaning |
|---|---|
| `party_id`, `part` | stable id; `person` or `business` |
| `first`, `middle`, `last` | cleaned name parts (`clean_person`) |
| `first_roots`, `last_nysiis`, `name_key` | nickname roots (`|`-joined), NYSIIS of the surname, `FIRST LAST` |
| `holder_key` | who holds a value, for single-holder counts: `LAST|F` (surname and first initial) for a person, the first alias for a business |
| `org_aliases`, `name_key` (business) | `|`-joined aliases from `org_aliases`; first alias is the name key |
| `dob`, `dob_int` | ISO date; `YYYYMMDD` as int (0 = none) |
| `addr_full`, `addr_street`, `zip`, `city_state`, `state` | address keys (`address_keys`) |
| `id_ssn` `id_npi` `id_dl` `id_tin` `id_license` `id_email` `id_vin` `id_plate` `id_cnpi` | valid, non-junk identifier values |
| `phone_own`, `phone_row` | a phone the party owns; a phone its row holds |
| `specialty`, `category`, `cat_strength` | canonical specialty; category; `strong` or `weak` |
| `tie` | `party_id` of the other part of the same row, or empty |

Pairs are positional: `il[i]` indexes the left frame, `ir[i]` the right one. The right frame
is the reference population whose value frequencies give per-value `u` (the watchlist in [B]).

**Where Splink would do better (whole core).** Splink compiles comparisons to SQL and runs
them in DuckDB or Spark, so 1M x 300k is routine and multi-core by default; here numpy and
per-unique-value Python loops do the work on one core. Splink's comparison library has
tested levels for names, dates and addresses; the levels below are hand-written. Splink
draws a waterfall chart per pair; the evidence table below is its tabular equivalent.

In [1]:
# ---- Matching core v1.0: parameters and field definitions ---------------------------------
# Ported pieces carry a header naming their source in goko-v2-poc/goko_v2_poc.ipynb.
import math, re, unicodedata, hashlib, json
from dataclasses import dataclass, field, asdict
from collections import defaultdict
import numpy as np
import pandas as pd
import jellyfish
import recordlinkage as rl

CORE_VERSION = "1.0"


@dataclass
class CoreParams:
    """Every number the core uses. The caller's run configuration embeds one of these and the
    manifest dumps it: there are no thresholds anywhere else in the core."""
    jw_close: float = 0.92          # Jaro-Winkler at or above: a spelling variant (goko cell 18)
    org_rare_idf: float = 10.0      # bits: a word fewer than ~1 in 1,000 organizations use
    org_distinct_min_len: int = 4   # a distinctive org word has at least this many letters
    surname_floor: int = 50         # count for a surname below the Census cutoff (goko cell 16)
    firstname_floor: int = 20       # count for a first name below the SSA cutoff
    org_df_floor: int = 1           # organizations using a word absent from NPPES
    own_org_min_count: int = 5      # the reference population's own word share counts from this many users
    flat_freq: float = 1e-3         # used only when an outside table is missing; stamped FLAT
    alpha: float = 5.0              # shrinkage of an m estimate toward the next source
    n_min: int = 50                 # informative pairs below which simulation joins the chain
    em_max_iter: int = 300
    em_tol: float = 1e-8
    u_pseudo: float = 0.5           # pseudo-count for a level never seen among random pairs
    prior_pseudo: float = 0.5       # pseudo-count for a prior group with no strict pair
    junk_holders: int = 25          # a value held under more names than this is junk
    u_floor: float = 1e-12
    dob_min_year: int = 1900
    dob_max_year: int = 2026        # dates after this year are invalid (fixed for determinism)
    coparty_min_p: float = 0.9      # a business link must reach this to anchor a co-party
    ci_z: float = 1.96              # 95% intervals


# Fields per part, and their levels best-first. Every field also has an EMPTY level (-1),
# worth 0 bits: one side has nothing to compare.
EMPTY = -1
FIELD_LEVELS = {
    "name": ["exact", "first_nick_or_close", "last_close_first_agrees", "initial_agrees",
             "swapped", "first_empty", "first_differs", "else"],
    "middle": ["exact", "initial", "differs"],
    "org": ["exact", "dba", "short_form", "rare_shared", "common_shared", "sibling", "none"],
    "dob": ["exact", "swap_or_typo", "year_month", "year", "differs"],
    "address": ["exact", "street", "zip", "city_state", "state", "differs"],
    "ssn": ["exact", "near", "differs"],
    "npi": ["exact", "near", "differs"],
    "dl": ["exact", "near", "differs"],
    "tin": ["exact", "near", "differs"],
    "license": ["exact", "differs"],
    "email": ["exact", "differs"],
    "vin": ["exact", "differs"],
    "plate": ["exact", "differs"],
    "cnpi": ["exact", "differs"],
    "phone": ["exact_owned_single", "exact_shared", "differs"],
    "spec_cat": ["specialty", "category_id", "category_weak", "differs"],
    "co_party": ["anchored", "not_anchored"],
}
PART_FIELDS = {
    "person": ["name", "middle", "dob", "address", "ssn", "npi", "dl", "license", "email",
               "vin", "plate", "phone", "spec_cat", "co_party"],
    "business": ["org", "address", "tin", "cnpi", "email", "phone", "spec_cat"],
}
COMPARED_FIELDS = {p: [f for f in fs if f != "co_party"] for p, fs in PART_FIELDS.items()}
# Levels that mean "agrees": their bits never go below 0, so an extra agreeing field can
# never lower p.
AGREEMENT_LEVELS = {
    "name": {"exact", "first_nick_or_close", "last_close_first_agrees", "initial_agrees", "swapped"},
    "middle": {"exact", "initial"}, "org": {"exact", "dba", "short_form", "rare_shared"},
    "dob": {"exact", "swap_or_typo"}, "address": {"exact", "street"},
    "ssn": {"exact", "near"}, "npi": {"exact", "near"}, "dl": {"exact", "near"},
    "tin": {"exact", "near"}, "license": {"exact"}, "email": {"exact"}, "vin": {"exact"},
    "plate": {"exact"}, "cnpi": {"exact"}, "phone": {"exact_owned_single", "exact_shared"},
    "spec_cat": {"specialty", "category_id", "category_weak"}, "co_party": {"anchored"},
}
# Levels that mean "disagrees": their bits never go above 0 (a sparse level can otherwise come
# out with m > u by chance, and a disagreement must never count for a match).
DISAGREEMENT_LEVELS = {
    "name": {"else"}, "middle": {"differs"}, "org": {"sibling", "none"}, "dob": {"differs"},
    "address": {"differs"}, "ssn": {"differs"}, "npi": {"differs"}, "dl": {"differs"},
    "tin": {"differs"}, "license": {"differs"}, "email": {"differs"}, "vin": {"differs"},
    "plate": {"differs"}, "cnpi": {"differs"}, "phone": {"differs"}, "spec_cat": {"differs"},
}
# Levels whose bits are forced to 0: no evidence either way, by declaration.
ZERO_LEVELS = {"co_party": {"not_anchored"}}
IDENTIFIER_FIELDS = ["ssn", "npi", "dl", "tin", "license", "email", "vin", "plate", "cnpi", "phone"]
VETO_FIELDS = ["ssn", "npi", "dl", "tin"]          # one per party: two single-holder values veto
NEAR_FIELDS = {"ssn", "npi", "dl", "tin"}
NAME_FIELDS = {"name", "org"}
BASIS_ORDER = ["identifier", "address", "dob", "co_party", "contextual", "name_only", "none"]


def level_code(field_, level):
    return FIELD_LEVELS[field_].index(level)

In [2]:
# ---- Matching core v1.0: normalizers ------------------------------------------------------
# Ported from goko-v2-poc/goko_v2_poc.ipynb cell "## 16" (parse_person, _undot, _words,
# org_aliases, ORG_ABBREV, ORG_LEGAL, TITLES, CREDENTIALS, name_like) and cell "## 18"
# (_rejoin, _org_alias_bits, _distinctive, declared d/b/a). Adapted: names arrive in parts.

TITLES = {"DR", "DOCTOR", "MR", "MRS", "MS", "MISS", "HON", "JUDGE", "PROF"}
CREDENTIALS = {"MD", "DO", "DC", "DPM", "DDS", "DMD", "PHD", "PT", "DPT", "NP", "PA", "RN",
               "LPN", "LAC", "ESQ", "JD", "CPA", "OT", "OTR", "FACS", "FACP", "MPH", "LCSW",
               "PSYD"}
BARE_CREDENTIALS = CREDENTIALS - {"DO", "PA", "PT", "OT"}
NAME_SUFFIXES = {"JR", "SR", "II", "III", "IV"}
ORG_LEGAL = {"PC", "PLLC", "LLC", "INC", "CORP", "CORPORATION", "CO", "LTD", "LLP", "LP",
             "PA", "COMPANY", "INCORPORATED", "THE", "AND", "OF"}
ORG_ABBREV = {"INS": "INSURANCE", "MED": "MEDICAL", "CTR": "CENTER", "CNTR": "CENTER",
              "SVCS": "SERVICES", "SVC": "SERVICES", "ASSOC": "ASSOCIATES", "ASSN": "ASSOCIATION",
              "HOSP": "HOSPITAL", "MGMT": "MANAGEMENT", "INTL": "INTERNATIONAL", "NATL": "NATIONAL",
              "DIAG": "DIAGNOSTIC", "REHAB": "REHABILITATION", "PHYS": "PHYSICAL", "THER": "THERAPY",
              "CHIRO": "CHIROPRACTIC", "GRP": "GROUP", "SVS": "SERVICES", "LAB": "LABORATORY",
              "LABS": "LABORATORIES", "PHARM": "PHARMACY", "TRANS": "TRANSPORTATION"}
ROLE_WORDS_UP = {"DEFENDANT", "DEFENDANTS", "PLAINTIFF", "PLAINTIFFS", "ANSWERING", "CLAIMANT",
                 "CLAIMANTS", "INSURED", "COUNSEL", "ATTORNEY", "RESPONDENT", "PETITIONER", "THE",
                 "HIS", "HER", "THEIR"}
_DBA_RE = re.compile(r"\b(?:d\s*/\s*b\s*/\s*a|dba|a\s*/\s*k\s*/\s*a|aka|f\s*/\s*k\s*/\s*a|fka|"
                     r"doing business as|trading as|t\s*/\s*a)\b", re.I)


def fold(s):
    """Uppercase ASCII: accents dropped ('JOSÉ' -> 'JOSE'), whitespace collapsed."""
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip().upper()


def _undot(s):
    """'M.D.' -> 'MD', 'P.C.' -> 'PC', then any remaining dot is a space ('A.' -> 'A')."""
    s = re.sub(r"\b((?:[A-Za-z]{1,2}\.){2,})", lambda m: m.group(1).replace(".", ""), s)
    return s.replace(".", " ")


def _words(s):
    return [w.upper() for w in re.findall(r"[A-Za-z][A-Za-z'\-]*", s)]


def parse_person(name):
    """goko cell 16: split a full name into first / middle initial / last, with credentials.
    Handles 'Dr. A. Monroe', 'WILLIAM A. WEINER, D.O.', 'Moy, Marvin' and 'Mr. Pierre'."""
    s = _undot(fold(name))
    head, _, tail = s.partition(",")
    hw, tw = _words(head), _words(tail)
    creds = {w for w in tw if w in CREDENTIALS} | {w for w in hw[1:] if w in BARE_CREDENTIALS}
    title = next((w for w in hw if w in TITLES), None)
    drop = TITLES | CREDENTIALS | NAME_SUFFIXES | ROLE_WORDS_UP
    core_h = [w.strip("'-") for w in hw if w not in drop]
    core_t = [w.strip("'-") for w in tw if w not in drop]
    core = core_t + core_h if (len(core_h) == 1 and core_t) else core_h
    core = [w for w in core if w]
    first = middle = last = ""
    if len(core) == 1:
        last = core[0]
    elif len(core) >= 2:
        first, last = core[0], core[-1]
        middle = core[1][0] if len(core) > 2 else ""
    return {"first": first, "middle": middle, "last": last, "creds": sorted(creds), "title": title}


def _clean_part(s, drop_creds=True):
    words = [w.replace("'", "").strip("-") for w in _words(_undot(fold(s)))]
    drop = TITLES | NAME_SUFFIXES | (BARE_CREDENTIALS if drop_creds else set())
    return " ".join(w for w in words if w and w not in drop)


def clean_person(first, middle, last):
    """Cleaned (first, middle, last) from name parts as a form holds them.

    'Dr. John' -> 'JOHN'; 'SMITH JR' -> 'SMITH'; "O'Brien" -> 'OBRIEN'. A full name typed into
    one box is split with parse_person: 'SMITH, JOHN' in last, or 'JOHN SMITH' in first with
    no last."""
    f, m, l = fold(first), fold(middle), fold(last)
    if not l and f and len(_words(f)) >= 2 and not m:
        p = parse_person(f)
        f, m, l = p["first"], p["middle"], p["last"]
    elif l and not f and "," in l:
        p = parse_person(l)
        f, m, l = p["first"], p["middle"], p["last"]
    return _clean_part(f), _clean_part(m), _clean_part(l)


def org_aliases(name):
    """goko cell 16: name variants of one organization: d/b/a parts split, legal suffixes
    stripped, abbreviations expanded. Returns a list of word lists."""
    out = []
    for p in _DBA_RE.split(fold(name)):
        toks = re.findall(r"[A-Z0-9]+", _undot(p).upper().replace("'", "").replace("&", " AND "))
        joined, i = [], 0
        while i < len(toks):                     # OCR splits a suffix ('COMP ANY'): rejoin it
            if i + 1 < len(toks) and toks[i] + toks[i + 1] in ORG_LEGAL:
                joined.append(toks[i] + toks[i + 1]); i += 2
            else:
                joined.append(toks[i]); i += 1
        toks = [ORG_ABBREV.get(t, t) for t in joined]
        toks = [t for t in toks if len(t) > 1 and t not in ORG_LEGAL and t not in ROLE_WORDS_UP]
        if toks and toks not in out:
            out.append(toks)
    # a short form that is a subset of a longer alias adds nothing (goko build_mention)
    keep = [a for a in out if not any(set(a) < set(b) for b in out)] or out
    return keep


def org_alias_string(name):
    """'|'-joined aliases, each alias its words joined by a space: the party-frame form."""
    return "|".join(" ".join(a) for a in org_aliases(name))


# ---- identifiers ---------------------------------------------------------------------------
def _digits(s):
    return re.sub(r"\D", "", fold(s))


def _all_same(d):
    return len(set(d)) <= 1


_SEQ = {"123456789", "987654321", "1234567890", "0123456789", "012345678"}


def norm_ssn(s):
    """(value, valid, reason). Invalid: not 9 digits, area 000/666/9xx, group 00, serial 0000,
    one repeated digit, or a sequence."""
    d = _digits(s)
    if not d:
        return "", False, ""
    if len(d) != 9:
        return d, False, "length"
    if d[:3] in ("000", "666") or d[0] == "9" or d[3:5] == "00" or d[5:] == "0000":
        return d, False, "range"
    if _all_same(d) or d in _SEQ:
        return d, False, "placeholder"
    return d, True, ""


_BAD_EIN_PREFIX = {"00", "07", "08", "09", "17", "18", "19", "28", "29", "49", "69", "70", "78",
                   "79", "89", "96", "97"}


def norm_tin(s):
    d = _digits(s)
    if not d:
        return "", False, ""
    if len(d) != 9:
        return d, False, "length"
    if _all_same(d) or d in _SEQ:
        return d, False, "placeholder"
    if d[:2] in _BAD_EIN_PREFIX:
        return d, False, "prefix"
    return d, True, ""


def npi_luhn_ok(d):
    """NPI check digit: Luhn over '80840' + the first nine digits."""
    if len(d) != 10 or not d.isdigit():
        return False
    total = 0
    for i, ch in enumerate(reversed("80840" + d[:9])):
        v = int(ch)
        if i % 2 == 0:
            v *= 2
            if v > 9:
                v -= 9
        total += v
    return (10 - total % 10) % 10 == int(d[9])


def norm_npi(s):
    d = _digits(s)
    if not d or not d.strip("0"):
        return "", False, ""                       # LEIE writes 0000000000 for "none"
    if len(d) != 10:
        return d, False, "length"
    if not npi_luhn_ok(d):
        return d, False, "check_digit"
    return d, True, ""


def norm_phone(s):
    d = _digits(s)
    if len(d) == 11 and d[0] == "1":
        d = d[1:]
    if not d:
        return "", False, ""
    if len(d) != 10:
        return d, False, "length"
    if d[0] in "01" or d[3] in "01":
        return d, False, "range"
    if _all_same(d) or d in _SEQ or (d[3:6] == "555" and d[6:8] == "01"):
        return d, False, "placeholder"
    return d, True, ""


_JUNK_EMAIL_LOCAL = {"none", "na", "n/a", "noemail", "no", "unknown", "test", "noreply",
                     "nomail", "declined", "null", "email"}


def norm_email(s):
    e = str(s or "").strip().lower()
    if not e or e in ("nan",):
        return "", False, ""
    if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}", e):
        return e, False, "format"
    local, dom = e.split("@", 1)
    if local in _JUNK_EMAIL_LOCAL or dom.split(".")[0] in ("none", "na", "unknown", "noemail"):
        return e, False, "placeholder"
    return e, True, ""


def _alnum(s):
    return re.sub(r"[^A-Z0-9]", "", fold(s))


US_STATES = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR", "CALIFORNIA": "CA",
    "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE", "DISTRICT OF COLUMBIA": "DC",
    "FLORIDA": "FL", "GEORGIA": "GA", "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL",
    "INDIANA": "IN", "IOWA": "IA", "KANSAS": "KS", "KENTUCKY": "KY", "LOUISIANA": "LA",
    "MAINE": "ME", "MARYLAND": "MD", "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN",
    "MISSISSIPPI": "MS", "MISSOURI": "MO", "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ", "NEW MEXICO": "NM", "NEW YORK": "NY",
    "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH", "OKLAHOMA": "OK", "OREGON": "OR",
    "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC", "SOUTH DAKOTA": "SD",
    "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT", "VERMONT": "VT", "VIRGINIA": "VA",
    "WASHINGTON": "WA", "WEST VIRGINIA": "WV", "WISCONSIN": "WI", "WYOMING": "WY",
    "PUERTO RICO": "PR", "GUAM": "GU", "VIRGIN ISLANDS": "VI", "AMERICAN SAMOA": "AS",
    "NORTHERN MARIANA ISLANDS": "MP"}
STATE_CODES = set(US_STATES.values()) | {"AA", "AE", "AP"}


def norm_state(s):
    t = re.sub(r"[^A-Z ]", "", fold(s)).strip()
    if t in STATE_CODES:
        return t
    return US_STATES.get(t, "")


def _qualified(number, state, min_len=4):
    """'STATE:NUMBER' (or ':NUMBER' when no state). Invalid when too short or one repeated
    character."""
    n = _alnum(number)
    if not n:
        return "", False, ""
    st = norm_state(state)
    v = f"{st}:{n}"
    if len(n) < min_len:
        return v, False, "length"
    if _all_same(n) or not n.strip("0"):
        return v, False, "placeholder"
    return v, True, ""


def norm_dl(number, state):
    return _qualified(number, state, 4)


def norm_license(number, state):
    return _qualified(number, state, 3)


def norm_plate(number, state):
    return _qualified(number, state, 2)


_VIN_MAP = {**{str(i): i for i in range(10)},
            "A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "H": 8, "J": 1, "K": 2,
            "L": 3, "M": 4, "N": 5, "P": 7, "R": 9, "S": 2, "T": 3, "U": 4, "V": 5, "W": 6,
            "X": 7, "Y": 8, "Z": 9}
_VIN_W = [8, 7, 6, 5, 4, 3, 2, 10, 0, 9, 8, 7, 6, 5, 4, 3, 2]


def vin_check_ok(v):
    if len(v) != 17 or any(c not in _VIN_MAP for c in v):
        return False
    r = sum(_VIN_MAP[c] * w for c, w in zip(v, _VIN_W)) % 11
    return v[8] == ("X" if r == 10 else str(r))


def norm_vin(s):
    v = _alnum(s)
    if not v:
        return "", False, ""
    if len(v) != 17 or set(v) & set("IOQ"):
        return v, False, "format"
    if not vin_check_ok(v):
        return v, False, "check_digit"
    return v, True, ""


_PLACEHOLDER_DOB = {"1900-01-01", "1901-01-01", "1899-12-31", "1800-01-01", "9999-12-31",
                    "1111-11-11", "1910-01-01"}


def norm_dob(s, params=None):
    """(ISO date, valid, reason). Accepts ISO, US m/d/y (2-digit years pivot at the max year),
    YYYYMMDD. Placeholders (1900-01-01 ...), impossible and future dates are invalid."""
    p = params or CoreParams()
    t = str(s or "").strip()
    if not t or t.lower() == "nan":
        return "", False, ""
    t = t.split("T")[0].split(" ")[0]
    y = m = d = None
    if re.fullmatch(r"\d{8}", t):
        y, m, d = int(t[:4]), int(t[4:6]), int(t[6:])
    elif re.fullmatch(r"\d{4}[-/.]\d{1,2}[-/.]\d{1,2}", t):
        y, m, d = (int(x) for x in re.split(r"[-/.]", t))
    elif re.fullmatch(r"\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}", t):
        m, d, y = (int(x) for x in re.split(r"[-/.]", t))
        if y < 100:
            y += 1900 if y > p.dob_max_year % 100 else 2000
    else:
        return t, False, "format"
    try:
        iso = pd.Timestamp(year=y, month=m, day=d).strftime("%Y-%m-%d")
    except (ValueError, OverflowError):
        return t, False, "impossible"
    if iso in _PLACEHOLDER_DOB:
        return iso, False, "placeholder"
    if y < p.dob_min_year or y > p.dob_max_year:
        return iso, False, "range"
    return iso, True, ""


# ---- addresses -----------------------------------------------------------------------------
STREET_TYPES = {"STREET": "ST", "STR": "ST", "ST": "ST", "AVENUE": "AVE", "AV": "AVE", "AVE": "AVE",
                "BOULEVARD": "BLVD", "BLVD": "BLVD", "ROAD": "RD", "RD": "RD", "DRIVE": "DR",
                "DR": "DR", "LANE": "LN", "LN": "LN", "COURT": "CT", "CT": "CT", "PLACE": "PL",
                "PL": "PL", "TERRACE": "TER", "TER": "TER", "PARKWAY": "PKWY", "PKWY": "PKWY",
                "HIGHWAY": "HWY", "HWY": "HWY", "CIRCLE": "CIR", "CIR": "CIR", "WAY": "WAY",
                "TRAIL": "TRL", "TRL": "TRL", "SQUARE": "SQ", "SQ": "SQ", "PLAZA": "PLZ",
                "PLZ": "PLZ", "COVE": "CV", "CV": "CV", "EXPRESSWAY": "EXPY", "EXPY": "EXPY",
                "TURNPIKE": "TPKE", "TPKE": "TPKE", "ALLEY": "ALY", "ALY": "ALY", "LOOP": "LOOP",
                "PIKE": "PIKE", "ROW": "ROW", "WALK": "WALK", "PATH": "PATH", "RUN": "RUN",
                "CROSSING": "XING", "XING": "XING", "POINT": "PT", "PT": "PT", "RIDGE": "RDG",
                "RDG": "RDG"}
DIRECTIONS = {"NORTH": "N", "SOUTH": "S", "EAST": "E", "WEST": "W", "NORTHEAST": "NE",
              "NORTHWEST": "NW", "SOUTHEAST": "SE", "SOUTHWEST": "SW", "N": "N", "S": "S",
              "E": "E", "W": "W", "NE": "NE", "NW": "NW", "SE": "SE", "SW": "SW"}
UNIT_WORDS = {"APT", "APARTMENT", "UNIT", "STE", "SUITE", "RM", "ROOM", "FL", "FLOOR", "BLDG",
              "LOT", "SPC", "SPACE", "TRLR", "DEPT", "#"}
_CITY_PREFIX = {"ST": "SAINT", "FT": "FORT", "MT": "MOUNT"}


def _addr_words(s):
    return re.sub(r"[^A-Z0-9# ]", " ", _undot(fold(s)).replace("-", " - ")).split()


def norm_street_parts(number, direction, name, stype, unit):
    """Normalized (number, direction, name, type, unit). A type or direction left inside the
    name ('MAIN STREET', 'W TAYLOR') is moved to its own part."""
    num = re.sub(r"[^A-Z0-9\-]", "", fold(number))
    words = [w for w in _addr_words(name) if w != "-"]
    d = DIRECTIONS.get(fold(direction).replace(".", ""), "")
    t = STREET_TYPES.get(fold(stype).replace(".", ""), fold(stype).replace(".", ""))
    if words and not d and words[0] in DIRECTIONS and len(words) > 1:
        d = DIRECTIONS[words[0]]; words = words[1:]
    if words and not t and words[-1] in STREET_TYPES and len(words) > 1:
        t = STREET_TYPES[words[-1]]; words = words[:-1]
    if words and words[-1] in DIRECTIONS and len(words) > 1:      # post-direction: 'AVE W'
        words = words[:-1]
    u = [w for w in _addr_words(unit) if w not in UNIT_WORDS and w != "-"]
    return num, d, " ".join(words), t, "".join(u)


def parse_street_line(line):
    """'2161 UNIVERSITY AVENUE W, STE 5' -> ('2161', '', 'UNIVERSITY', 'AVE', '5').
    'P O BOX 2161' -> ('2161', '', 'PO BOX', '', '')."""
    s = fold(line)
    if not s:
        return "", "", "", "", ""
    box = re.match(r"^(?:P\s*\.?\s*O\.?\s*BOX|POST OFFICE BOX|BOX)\s*#?\s*([A-Z0-9\-]+)", s)
    if box:
        return box.group(1), "", "PO BOX", "", ""
    main, _, rest = s.partition(",")
    unit = rest.strip()
    m = re.search(r"\s(?:APT|APARTMENT|UNIT|STE|SUITE|RM|ROOM|FL|FLOOR|BLDG|LOT|SPC|#)\s*\.?\s*#?\s*([A-Z0-9\-]+)\s*$", main)
    if m:
        unit = unit or m.group(1)
        main = main[:m.start()]
    elif "#" in main:
        main, _, u2 = main.partition("#")
        unit = unit or u2.strip()
    mnum = re.match(r"^\s*(\d+[A-Z]?(?:-\d+[A-Z]?)?)\s+(.*)$", main.strip())
    num, rest_name = (mnum.group(1), mnum.group(2)) if mnum else ("", main.strip())
    ws = rest_name.split()
    d = ""
    if len(ws) > 2 and ws[0] in ("N", "S") and ws[1] in ("E", "W"):      # 'N W 45TH' -> NW
        ws = [ws[0] + ws[1]] + ws[2:]
    if len(ws) > 1 and ws[0].replace(".", "") in DIRECTIONS:
        d = DIRECTIONS[ws[0].replace(".", "")]; ws = ws[1:]
    t = ""
    if len(ws) > 1 and ws[-1].replace(".", "") in DIRECTIONS:
        ws = ws[:-1]
    if len(ws) > 1 and ws[-1].replace(".", "") in STREET_TYPES:
        t = STREET_TYPES[ws[-1].replace(".", "")]; ws = ws[:-1]
    unit = " ".join(w for w in re.sub(r"[^A-Z0-9 ]", " ", unit).split() if w not in UNIT_WORDS)
    return num, d, " ".join(ws), t, unit.replace(" ", "")


def norm_city(s):
    ws = re.sub(r"[^A-Z ]", " ", _undot(fold(s))).split()
    if ws and ws[0] in _CITY_PREFIX:
        ws[0] = _CITY_PREFIX[ws[0]]
    return " ".join(ws)


def norm_zip(s):
    d = _digits(s)
    if len(d) in (3, 4):
        d = d.zfill(5)                             # a leading zero lost in a spreadsheet
    d = d[:5]
    return d if len(d) == 5 and d != "00000" else ""


def address_keys(number, name, unit, zip5, city, state):
    """The five keys the address levels compare, best first. Each is empty when a part it
    needs is missing."""
    street = f"{number}|{name}" if number and name else ""
    full = f"{street}|{unit}|{zip5}" if street and zip5 else ""
    city_state = f"{city}|{state}" if city and state else ""
    return full, street, zip5, city_state, state


# ---- nicknames and phonetics ---------------------------------------------------------------
class Nicknames:
    """Nickname roots from a (name1, relationship, name2) table (carltonnorthern/nicknames).
    The table lists relations both ways ('bill has_nickname robert'), so it is read as
    undirected, and a name's roots are itself plus every related name longer than it: BILL
    {BILL, ROBERT, WILLIAM} meets WILLIAM {WILLIAM}, and BILLY meets WILL through WILLIAM, but
    ROBERT {ROBERT} does not meet WILLIAM through BILL."""

    def __init__(self, table):
        nb = defaultdict(set)
        if table is not None and len(table):
            for a, b in zip(table["name1"].map(fold), table["name2"].map(fold)):
                nb[a].add(b)
                nb[b].add(a)
        self._roots = {n: {x for x in rel if len(x) > len(n)} for n, rel in nb.items()}
        self.size = len(table) if table is not None else 0

    def roots(self, first):
        cache = self.__dict__.setdefault("_cache", {})
        r = cache.get(first)
        if r is None:
            f = fold(first).split(" ")[0] if first else ""
            r = cache[first] = frozenset(self._roots.get(f, set()) | {f}) if f else frozenset()
        return r

    def roots_string(self, first):
        return "|".join(sorted(self.roots(first)))


def nysiis(s):
    s = re.sub(r"[^A-Z]", "", fold(s))
    return jellyfish.nysiis(s) if s else ""


def map_unique(values, func):
    """Apply func once per distinct value; values is a Series. Returns a Series."""
    s = pd.Series(values)
    uniq = pd.unique(s)
    table = {v: func(v) for v in uniq}
    return s.map(table)

In [3]:
# ---- Matching core v1.0: outside rarity -----------------------------------------------------
# Ported from goko-v2-poc/goko_v2_poc.ipynb cell "## 16" (surname_freq, first_freq,
# initial_share, org_idf). Change: org_idf takes the larger of NPPES's share and the reference
# population's own share of a word (PLAN.md open question 5), so AUTO or COLLISION, rare among
# NPPES health-care organizations, are not treated as rare in a list full of body shops.

class Rarity:
    """Value-specific chance agreement from outside tables. Tables arrive as dicts
    {value: count} plus the metadata totals; a missing table gives a flat rate that is stamped
    'FLAT' in the source so the run cannot be mistaken for a real one."""

    def __init__(self, surnames=None, first_names=None, org_df=None, meta=None, params=None,
                 own_org_counts=None, own_org_total=0):
        self.p = params or CoreParams()
        meta = meta or {}
        self.surnames, self.firsts, self.org_df = surnames or {}, first_names or {}, org_df or {}
        self.surname_total = meta.get("census", {}).get("people") or max(1, sum(self.surnames.values()))
        self.first_total = meta.get("ssa", {}).get("births") or max(1, sum(self.firsts.values()))
        self.org_n = meta.get("nppes", {}).get("organizations") or 1
        ini = defaultdict(int)
        for n, c in self.firsts.items():
            ini[n[0]] += c
        self.initial = {k: v / self.first_total for k, v in ini.items()}
        self.own_org = own_org_counts or {}
        self.own_org_total = own_org_total
        self.source = {"surnames": "census2010" if self.surnames else "FLAT",
                       "first_names": "ssa1930-2005" if self.firsts else "FLAT",
                       "org_tokens": ("nppes" if (self.org_df and self.org_n > 1) else "FLAT")
                                     + ("+reference_population" if self.own_org_total else "")}
        self._cache_org = {}

    def surname_freq(self, last):
        if not self.surnames:
            return self.p.flat_freq
        parts = [last] + [x for x in re.split(r"[ \-]", last) if x and x != last]
        return min(self.surnames.get(x, self.p.surname_floor) for x in parts) / self.surname_total

    def first_freq(self, first):
        if not self.firsts:
            return self.p.flat_freq
        f = first.split(" ")[0]
        return self.firsts.get(f, self.p.firstname_floor) / self.first_total

    def initial_share(self, letter):
        return self.initial.get(letter[:1], 1 / 26) if self.firsts else 1 / 26

    def org_idf(self, tok):
        """Bits of surprise in two organizations sharing this name word."""
        if tok in self._cache_org:
            return self._cache_org[tok]
        if not (self.org_df and self.org_n > 1):
            idf = 6.0
        else:
            idf = math.log2(self.org_n / self.org_df.get(tok, self.p.org_df_floor))
        if self.own_org_total:
            own = math.log2(self.own_org_total / max(1, self.own_org.get(tok, 0)))
            if self.own_org.get(tok, 0) >= self.p.own_org_min_count:
                idf = min(idf, own)
        self._cache_org[tok] = idf
        return idf


def own_org_word_counts(org_alias_strings):
    """How many parties of the reference population use each org word (for Rarity)."""
    counts = defaultdict(int)
    n = 0
    for s in org_alias_strings:
        if not s:
            continue
        n += 1
        for w in set(" ".join(s.split("|")).split()):
            counts[w] += 1
    return dict(counts), n

In [4]:
# ---- Matching core v1.0: value statistics for per-value u and vetoes -----------------------

VALUE_FIELDS = {  # field -> party-frame columns holding its values
    "ssn": ["id_ssn"], "npi": ["id_npi"], "dl": ["id_dl"], "tin": ["id_tin"],
    "license": ["id_license"], "email": ["id_email"], "vin": ["id_vin"], "plate": ["id_plate"],
    "cnpi": ["id_cnpi"], "phone": ["phone_own", "phone_row"],
}
KEY_FIELDS = ["dob", "addr_full", "addr_street", "zip", "city_state", "state", "specialty",
              "category"]


@dataclass
class ValueStats:
    """Per-value counts the comparisons read. `ref_counts[key][value]`: parties of the
    reference (right) population holding the value; `ref_n[key]`: those holding any value.
    `names[field][value]`: distinct names holding an identifier value across both sides.
    `single[field]`: values held under exactly one name."""
    ref_counts: dict
    ref_n: dict
    names: dict
    single: dict


def build_value_stats(left, right):
    """Counts from the two party frames. Identifier u uses how many distinct names hold the
    value across both sides (so a shared or placeholder SSN weighs little); dates, addresses,
    specialty and category use the reference side's own frequencies."""
    ref_counts, ref_n, names, single = {}, {}, {}, {}
    for k in KEY_FIELDS:
        v = right[k][right[k] != ""] if k in right else pd.Series([], dtype=object)
        if k == "category":
            v = v[v != "other"]
        ref_counts[k] = v.value_counts()
        ref_n[k] = int(len(v))
    both = pd.concat([left, right], ignore_index=True)
    for f, cols in VALUE_FIELDS.items():
        parts = [pd.DataFrame({"v": both[c], "n": both["holder_key"]}) for c in cols if c in both]
        long = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame({"v": [], "n": []})
        long = long[long["v"] != ""]
        nn = long.drop_duplicates().groupby("v")["n"].size()
        names[f] = nn
        single[f] = set(nn.index[nn == 1])
        rv = pd.concat([right[c] for c in cols if c in right], ignore_index=True) if cols else pd.Series([], dtype=object)
        rv = rv[rv != ""]
        ref_n[f] = int((pd.concat([right[c] != "" for c in cols], axis=1).any(axis=1)).sum()) if len(right) else 0
        ref_counts[f] = rv.value_counts()
    return ValueStats(ref_counts, ref_n, names, single)


def junk_values(stats, params):
    """Identifier values held under more names than params.junk_holders: {field: set}."""
    return {f: set(nn.index[nn > params.junk_holders]) for f, nn in stats.names.items()}

In [5]:
# ---- Matching core v1.0: comparisons (recordlinkage.Compare + custom features) -------------
# One level per field, the same code for candidates, random pairs, anchors and simulated
# pairs. Each feature returns its level codes and, where the level is value-specific, the
# value's own chance-agreement factor `uv` (NaN where the field-level u applies).


@dataclass
class CompareContext:
    params: CoreParams
    rarity: Rarity
    nick: Nicknames
    stats: ValueStats
    dba_pairs: list = field(default_factory=list)      # [(set, set)] declared as one org
    components: dict = field(default_factory=dict)    # random-pair rates for name sub-parts


def _unique_pairs(a, b):
    """Joint factorization of two aligned arrays: (inverse, unique a, unique b)."""
    ca, ua = pd.factorize(np.asarray(a, dtype=object))
    cb, ub = pd.factorize(np.asarray(b, dtype=object))
    nb = len(ub) + 1
    key = ca.astype(np.int64) * nb + cb.astype(np.int64)
    uk, inv = np.unique(key, return_inverse=True)
    return inv, np.asarray(ua, dtype=object)[uk // nb], np.asarray(ub, dtype=object)[uk % nb]


def _arr(s):
    return np.asarray(s.to_numpy() if hasattr(s, "to_numpy") else s, dtype=object)


def _lookup(counts, values):
    """counts: Series value->count. Returns float array (0 where absent)."""
    if counts is None or not len(counts):
        return np.zeros(len(values))
    return pd.Series(values).map(counts).fillna(0).to_numpy(dtype=float)


_JW_CACHE = {}


def _jw(a, b):
    """Jaro-Winkler, memoized: name and org words repeat across millions of pairs."""
    k = (a, b)
    v = _JW_CACHE.get(k)
    if v is None:
        if len(_JW_CACHE) > 5_000_000:
            _JW_CACHE.clear()
        v = _JW_CACHE[k] = jellyfish.jaro_winkler_similarity(a, b)
    return v


_NYSIIS_CACHE = {}


def _nysiis_c(s):
    v = _NYSIIS_CACHE.get(s)
    if v is None:
        v = _NYSIIS_CACHE[s] = nysiis(s)
    return v


def _last_cmp(a, b, p):
    """0 exact, 1 close (spelling, sound or one compound part), 2 differs, -1 empty."""
    if not a or not b:
        return -1
    if a == b:
        return 0
    ta, tb = set(re.split(r"[ \-]", a)), set(re.split(r"[ \-]", b))
    if (ta <= tb or tb <= ta) or _jw(a, b) >= p.jw_close or _nysiis_c(a) == _nysiis_c(b):
        return 1
    return 2


FIRST_CMP = ["exact", "nick_or_close", "initial", "empty", "differs"]


def _first_cmp(a, b, p, nick):
    """0 exact, 1 nickname or close spelling, 2 initial agrees, 3 empty, 4 differs."""
    if not a or not b:
        return 3
    if len(a) == 1 or len(b) == 1:
        return 2 if a[0] == b[0] else 4
    if a == b:
        return 0
    if (nick.roots(a) & nick.roots(b)) or _jw(a, b) >= p.jw_close:
        return 1
    return 4


def f_name(first_l, last_l, first_r, last_r, ctx):
    """Person name, one graded group (see FIELD_LEVELS['name'])."""
    p, R = ctx.params, ctx.rarity
    fl, ll, fr, lr = _arr(first_l), _arr(last_l), _arr(first_r), _arr(last_r)
    inv, ua, ub = _unique_pairs(ll, lr)
    lc = np.array([_last_cmp(a, b, p) for a, b in zip(ua, ub)], dtype=np.int8)[inv]
    inv, ua, ub = _unique_pairs(fl, fr)
    fc = np.array([_first_cmp(a, b, p, ctx.nick) for a, b in zip(ua, ub)], dtype=np.int8)[inv]
    swapped = (fl != "") & (ll != "") & (fl == lr) & (ll == fr) & (ll != lr)
    last_exact, last_close = lc == 0, lc == 1
    first_agree = (fc == 0) | (fc == 1)
    conds = [lc < 0, last_exact & (fc == 0), last_exact & (fc == 1),
             last_close & first_agree, last_exact & (fc == 2), swapped,
             last_exact & (fc == 3), last_exact & (fc == 4)]
    codes = [EMPTY, 0, 1, 2, 3, 4, 5, 6]
    level = np.select(conds, codes, default=7).astype(np.int8)
    # value-specific u: outside tables for the agreeing values, random-pair rates for the rest
    C = ctx.components or {}
    fL = map_unique(pd.Series(lr), R.surname_freq).to_numpy(dtype=float)
    fF = map_unique(pd.Series(fr), lambda v: R.first_freq(v) if v else 1.0).to_numpy(dtype=float)
    ini = map_unique(pd.Series(fr), lambda v: R.initial_share(v) if v else 1.0).to_numpy(dtype=float)
    fLsw = map_unique(pd.Series(fl), R.surname_freq).to_numpy(dtype=float)     # swapped: x.first is w.last
    uv = np.full(len(level), np.nan)
    uv = np.where(level == 0, fL * fF, uv)
    uv = np.where(level == 1, fL * C.get("first_nick_or_close", np.nan), uv)
    uv = np.where(level == 2, C.get("last_close", np.nan) *
                  np.where(fc == 0, fF, C.get("first_nick_or_close", np.nan)), uv)
    uv = np.where(level == 3, fL * ini, uv)
    uv = np.where(level == 4, fLsw * fF, uv)
    uv = np.where(level == 5, fL * C.get("first_empty", np.nan), uv)
    uv = np.where(level == 6, fL * C.get("first_differs", np.nan), uv)
    return level, uv, fc, lc


def f_middle(ml, mr, ctx):
    a, b = _arr(ml), _arr(mr)
    empty = (a == "") | (b == "")
    exact = (a == b) & ~empty & (np.char.str_len(a.astype(str)) > 1)
    ia = np.array([x[:1] for x in a], dtype=object)
    ib = np.array([x[:1] for x in b], dtype=object)
    initial = (ia == ib) & ~empty & ~exact
    level = np.select([empty, exact, initial], [EMPTY, 0, 1], default=2).astype(np.int8)
    return level, np.full(len(level), np.nan)


def _digit_arrays(v, width):
    return np.stack([(v // 10 ** k) % 10 for k in range(width)], axis=1)


def f_dob(dl, dr, ctx):
    a = np.asarray(dl, dtype=np.int64)
    b = np.asarray(dr, dtype=np.int64)
    empty = (a == 0) | (b == 0)
    ya, ma, da = a // 10000, (a // 100) % 100, a % 100
    yb, mb, db = b // 10000, (b // 100) % 100, b % 100
    exact = (a == b) & ~empty
    swap = (ya == yb) & (ma == db) & (da == mb) & ~exact
    hamming = (_digit_arrays(a, 8) != _digit_arrays(b, 8)).sum(axis=1)
    typo = (hamming == 1) & ~exact
    level = np.select([empty, exact, swap | typo, (ya == yb) & (ma == mb), ya == yb],
                      [EMPTY, 0, 1, 2, 3], default=4).astype(np.int8)
    st = ctx.stats
    uv = np.where(level == 0, _lookup(st.ref_counts.get("dob"),
                                      pd.Series(b).map(_dob_str).to_numpy()) / max(1, st.ref_n.get("dob", 0)),
                  np.nan)
    return level, uv


def _dob_str(v):
    v = int(v)
    return f"{v // 10000:04d}-{(v // 100) % 100:02d}-{v % 100:02d}" if v else ""


def f_address(full_l, street_l, zip_l, cs_l, st_l, full_r, street_r, zip_r, cs_r, st_r, ctx):
    fl, sl, zl, cl, tl = (_arr(x) for x in (full_l, street_l, zip_l, cs_l, st_l))
    fr, sr, zr, cr, tr = (_arr(x) for x in (full_r, street_r, zip_r, cs_r, st_r))
    eq = lambda x, y: (x == y) & (x != "")
    comparable = ((tl != "") & (tr != "")) | ((zl != "") & (zr != "")) | ((sl != "") & (sr != ""))
    conds = [eq(fl, fr), eq(sl, sr), eq(zl, zr), eq(cl, cr), eq(tl, tr), comparable]
    level = np.select(conds, [0, 1, 2, 3, 4, 5], default=EMPTY).astype(np.int8)
    st = ctx.stats
    uv = np.full(len(level), np.nan)
    for code, key, vals in ((0, "addr_full", fr), (1, "addr_street", sr), (2, "zip", zr),
                            (3, "city_state", cr), (4, "state", tr)):
        m = level == code
        if m.any():
            uv[m] = _lookup(st.ref_counts.get(key), vals[m]) / max(1, st.ref_n.get(key, 0))
    return level, uv


def _near(a, b):
    """One substituted character or two adjacent ones exchanged (same length)."""
    if len(a) != len(b) or a == b:
        return False
    diff = [i for i, (x, y) in enumerate(zip(a, b)) if x != y]
    if len(diff) == 1:
        return True
    return len(diff) == 2 and diff[1] == diff[0] + 1 and a[diff[0]] == b[diff[1]] and a[diff[1]] == b[diff[0]]


def _split_q(v):
    st, _, n = v.partition(":")
    return st, n


def f_identifier(vl, vr, ctx, field_):
    """Exact (value-specific u) / near (typo, for SSN, NPI, DL, TIN) / differs. Qualified
    values (DL, licence, plate: 'STATE:NUMBER') agree when the numbers match and the states
    match or one is unstated. Returns level, uv, veto (two different single-holder values of a
    one-per-party identifier; DL only within one state)."""
    a, b = _arr(vl), _arr(vr)
    n = len(a)
    empty = (a == "") | (b == "")
    qualified = field_ in ("dl", "license", "plate")
    level = np.full(n, EMPTY, dtype=np.int8)
    veto = np.zeros(n, dtype=bool)
    idx = np.flatnonzero(~empty)
    if len(idx):
        inv, ua, ub = _unique_pairs(a[idx], b[idx])
        near_ok = field_ in NEAR_FIELDS
        single = ctx.stats.single.get(field_, set())
        res = []
        for x, y in zip(ua, ub):
            if qualified:
                sx, nx = _split_q(x)
                sy, ny = _split_q(y)
                states_ok = (sx == sy) or not sx or not sy
                if nx == ny and states_ok:
                    res.append((0, False))
                elif near_ok and states_ok and _near(nx, ny):
                    res.append((1, False))
                else:
                    v = (field_ == "dl" and sx and sx == sy and x in single and y in single)
                    res.append((len(FIELD_LEVELS[field_]) - 1, bool(v)))
            else:
                if x == y:
                    res.append((0, False))
                elif near_ok and _near(x, y):
                    res.append((1, False))
                else:
                    v = field_ in VETO_FIELDS and x in single and y in single
                    res.append((len(FIELD_LEVELS[field_]) - 1, bool(v)))
        r = np.array(res, dtype=np.int64).reshape(-1, 2)[inv]
        level[idx] = r[:, 0]
        veto[idx] = r[:, 1].astype(bool)
    st = ctx.stats
    uv = np.full(n, np.nan)
    m = level == 0
    if m.any():
        names = _lookup(st.names.get(field_), b[m])
        refc = _lookup(st.ref_counts.get(field_), b[m])
        uv[m] = np.maximum(np.maximum(names, refc), 1.0) / max(1, st.ref_n.get(field_, 0))
    return level, uv, veto


def f_phone(own_l, row_l, own_r, row_r, ctx):
    """exact_owned_single: the agreeing number is owned on both sides and held under one name;
    exact_shared: agrees otherwise (row-level, or several holders); differs."""
    ol, rwl, orr, rwr = _arr(own_l), _arr(row_l), _arr(own_r), _arr(row_r)
    single = ctx.stats.single.get("phone", set())
    has_l = (ol != "") | (rwl != "")
    has_r = (orr != "") | (rwr != "")
    eq = lambda x, y: (x == y) & (x != "")
    own_own = eq(ol, orr)
    any_eq = own_own | eq(ol, rwr) | eq(rwl, orr) | eq(rwl, rwr)
    own_single = own_own & pd.Series(ol).isin(single).to_numpy()
    level = np.select([~(has_l & has_r), own_single, any_eq], [EMPTY, 0, 1], default=2).astype(np.int8)
    matched = np.where(own_own, ol, np.where(eq(ol, rwr), ol, np.where(eq(rwl, orr), rwl, rwl)))
    st = ctx.stats
    uv = np.full(len(level), np.nan)
    m = (level == 0) | (level == 1)
    if m.any():
        names = _lookup(st.names.get("phone"), matched[m])
        refc = _lookup(st.ref_counts.get("phone"), matched[m])
        uv[m] = np.maximum(np.maximum(names, refc), 1.0) / max(1, st.ref_n.get("phone", 0))
    return level, uv


def f_spec_cat(spec_l, cat_l, str_l, spec_r, cat_r, str_r, ctx):
    """Specialty and category, one correlated group: same specialty / same category, both
    identifier- or mapping-backed / same category, one side given or keyword / different."""
    sl, cl, gl, sr, cr, gr = (_arr(x) for x in (spec_l, cat_l, str_l, spec_r, cat_r, str_r))
    cl = np.where(cl == "other", "", cl)
    cr = np.where(cr == "other", "", cr)
    same_spec = (sl == sr) & (sl != "")
    same_cat = (cl == cr) & (cl != "")
    strong = (gl == "strong") & (gr == "strong")
    comparable = ((cl != "") & (cr != "")) | ((sl != "") & (sr != ""))
    level = np.select([same_spec, same_cat & strong, same_cat, comparable], [0, 1, 2, 3],
                      default=EMPTY).astype(np.int8)
    st = ctx.stats
    uv = np.full(len(level), np.nan)
    m = level == 0
    if m.any():
        uv[m] = _lookup(st.ref_counts.get("specialty"), sr[m]) / max(1, st.ref_n.get("specialty", 0))
    m = (level == 1) | (level == 2)
    if m.any():
        uv[m] = _lookup(st.ref_counts.get("category"), cr[m]) / max(1, st.ref_n.get("category", 0))
    return level, uv


# ---- organization names (goko cell 18, adapted to levels) ----------------------------------
def _rejoin(x, y):
    """goko cell 18: undo OCR splits: 'CASUAL','TY' -> 'CASUALTY' when the other has it."""
    ys, out, i = set(y), [], 0
    while i < len(x):
        if i + 1 < len(x) and x[i] + x[i + 1] in ys and x[i] not in ys:
            out.append(x[i] + x[i + 1]); i += 2
        else:
            out.append(x[i]); i += 1
    return out


def _match_words(x, y, p):
    """goko _org_alias_bits' word pairing: exact first, then Jaro-Winkler >= jw_close.
    Returns (matched words of x, words only in x, words only in y)."""
    x, y = _rejoin(x, y), _rejoin(y, x)
    left, matched = list(y), []
    for t in x:
        hit = t if t in left else max(left, key=lambda u: _jw(t, u), default=None)
        if hit is not None and (hit == t or _jw(t, hit) >= p.jw_close):
            matched.append(t); left.remove(hit)
    only_x = [t for t in x if t not in matched]
    return matched, only_x, left


def _fuzzy_family(a, b, p):
    """goko: one name's words all appear (spelling-tolerant) in the other."""
    small, big = (a, b) if len(a) <= len(b) else (b, a)
    return bool(small) and all(any(t == u or _jw(t, u) >= p.jw_close for u in big) for t in small)


def declared_dba_pairs(org_alias_strings):
    """goko cell 18 declared_dbas: alias pairs that one party declares to be one organization
    ('X d/b/a Y'). A trade name shares no words with the legal name; without this the sibling
    level would split a party from its own d/b/a."""
    pairs, seen = [], set()
    for s in org_alias_strings:
        al = [a for a in s.split("|") if a] if s else []
        for i in range(len(al)):
            for j in range(i + 1, len(al)):
                k = (al[i], al[j])
                if k not in seen:
                    seen.add(k)
                    pairs.append((frozenset(al[i].split()), frozenset(al[j].split())))
    return pairs


def _dba_index(ctx):
    """word -> indexes of declared d/b/a pairs using it (built once per context)."""
    idx = getattr(ctx, "_dba_idx", None)
    if idx is None or idx[0] is not ctx.dba_pairs:
        d = defaultdict(set)
        R = ctx.rarity
        for i, (q1, q2) in enumerate(ctx.dba_pairs):
            for q in (q1, q2):
                # index on the side's distinctive words (common ones like MEDICAL would put
                # every declaration in front of every pair); all words when none is distinctive
                rare = [w for w in q if R.org_idf(w) >= 6.0] or list(q)
                for w in rare:
                    d[w].add(i)
        idx = (ctx.dba_pairs, d)
        ctx._dba_idx = idx
    return idx[1]


def _declared_same(xs, ys, ctx):
    """Some record declares the two names to be one organization (goko _declared_same).
    Only declarations sharing a word with each side are checked."""
    if not ctx.dba_pairs:
        return False
    d = _dba_index(ctx)
    cx = set().union(*(d.get(w, set()) for a in xs for w in a))
    cy = set().union(*(d.get(w, set()) for b in ys for w in b))
    p = ctx.params
    for i in sorted(cx & cy):
        q1, q2 = ctx.dba_pairs[i]
        for a in xs:
            for b in ys:
                if (_fuzzy_family(set(a), q1, p) and _fuzzy_family(set(b), q2, p)) or                         (_fuzzy_family(set(a), q2, p) and _fuzzy_family(set(b), q1, p)):
                    return True
    return False


def _org_words_u(words, R):
    """Chance two organizations share these words: the rarest counts in full, the others at
    half, because the words of one name travel together (goko _org_alias_bits)."""
    idfs = sorted((R.org_idf(t) for t in words), reverse=True)
    if not idfs:
        return 1.0
    return 2.0 ** -(idfs[0] + 0.5 * sum(idfs[1:]))


def org_level(al, ar, ctx):
    """(level code, uv, detail) for two '|'-joined alias strings."""
    p, R = ctx.params, ctx.rarity
    if not al or not ar:
        return EMPTY, np.nan, ""
    xs = [a.split() for a in al.split("|") if a]
    ys = [a.split() for a in ar.split("|") if a]
    for x in xs:
        for y in ys:
            if x == y:
                return 0, max(p.u_floor, _org_words_u(y, R)), " ".join(y)
    best = None
    for x in xs:
        for y in ys:
            matched, ox, oy = _match_words(x, y, p)
            score = sum(R.org_idf(t) for t in matched)
            if best is None or score > best[0]:
                best = (score, matched, ox, oy, x, y)
    _, matched, ox, oy, x, y = best
    declared = _declared_same(xs, ys, ctx)
    if declared and not matched:
        return 1, max(p.u_floor, _org_words_u(y, R)), "declared d/b/a"
    if matched and (not ox or not oy):
        return 2, max(p.u_floor, _org_words_u(matched, R)), " ".join(matched)
    dist = lambda ws: [t for t in ws if len(t) >= p.org_distinct_min_len and R.org_idf(t) >= p.org_rare_idf]
    if matched and dist(ox) and dist(oy) and not declared:
        return 5, np.nan, f"{'+'.join(dist(ox))} vs {'+'.join(dist(oy))}"
    if declared:
        return 1, max(p.u_floor, _org_words_u(y, R)), "declared d/b/a"
    if matched:
        top = max(matched, key=R.org_idf)
        if R.org_idf(top) >= p.org_rare_idf:
            return 3, max(p.u_floor, 2.0 ** -R.org_idf(top)), top
        return 4, np.nan, " ".join(matched)
    return 6, np.nan, ""


def f_org(al, ar, ctx):
    a, b = _arr(al), _arr(ar)
    inv, ua, ub = _unique_pairs(a, b)
    res = [org_level(x, y, ctx) for x, y in zip(ua, ub)]
    level = np.array([r[0] for r in res], dtype=np.int8)[inv]
    uv = np.array([r[1] for r in res], dtype=float)[inv]
    return level, uv


# Which party-frame columns each feature reads.
FEATURE_COLUMNS = {
    "name": (["first", "last"], ["first", "last"]),
    "middle": (["middle"], ["middle"]),
    "dob": (["dob_int"], ["dob_int"]),
    "address": (["addr_full", "addr_street", "zip", "city_state", "state"],
                ["addr_full", "addr_street", "zip", "city_state", "state"]),
    "phone": (["phone_own", "phone_row"], ["phone_own", "phone_row"]),
    "spec_cat": (["specialty", "category", "cat_strength"], ["specialty", "category", "cat_strength"]),
    "org": (["org_aliases"], ["org_aliases"]),
    **{f: ([f"id_{f}"], [f"id_{f}"]) for f in ["ssn", "npi", "dl", "tin", "license", "email",
                                                "vin", "plate", "cnpi"]},
}


def _feature(field_, ctx):
    if field_ == "name":
        return lambda a, b, c, d: f_name(a, b, c, d, ctx), [field_, f"uv_{field_}", "name_first_cmp", "name_last_cmp"]
    if field_ == "middle":
        return lambda a, b: f_middle(a, b, ctx), [field_, f"uv_{field_}"]
    if field_ == "dob":
        return lambda a, b: f_dob(a, b, ctx), [field_, f"uv_{field_}"]
    if field_ == "address":
        return lambda *s: f_address(*s, ctx), [field_, f"uv_{field_}"]
    if field_ == "phone":
        return lambda a, b, c, d: f_phone(a, b, c, d, ctx), [field_, f"uv_{field_}"]
    if field_ == "spec_cat":
        return lambda *s: f_spec_cat(*s, ctx), [field_, f"uv_{field_}"]
    if field_ == "org":
        return lambda a, b: f_org(a, b, ctx), [field_, f"uv_{field_}"]
    return (lambda a, b, f=field_: f_identifier(a, b, ctx, f),
            [field_, f"uv_{field_}", f"veto_{field_}"])


def compare_pairs(left, right, il, ir, part, ctx):
    """Level per field for positional pairs (il into left, ir into right), computed through
    recordlinkage.Compare with one custom vectorized feature per field. Returns a DataFrame
    with one row per pair: `<field>` level codes, `uv_<field>` value-specific u factors,
    `veto_<field>` flags and the name sub-comparisons."""
    il = np.asarray(il, dtype=np.int64)
    ir = np.asarray(ir, dtype=np.int64)
    cols = ["l", "r"]
    if len(il) == 0:
        out = pd.DataFrame({"l": il, "r": ir})
        for f in COMPARED_FIELDS[part]:
            _, labels = _feature(f, ctx)
            for lab in labels:
                out[lab] = pd.Series([], dtype=float if lab.startswith("uv_") else
                                     (bool if lab.startswith("veto_") else np.int8))
        return out
    pairs = pd.MultiIndex.from_arrays([il, ir], names=cols)
    cmp = rl.Compare(indexing_type="position")
    for f in COMPARED_FIELDS[part]:
        func, labels = _feature(f, ctx)
        lcols, rcols = FEATURE_COLUMNS[f]
        cmp.compare_vectorized(func, lcols, rcols, label=labels)
    feats = cmp.compute(pairs, left, right)
    feats = feats.reset_index(drop=True)
    feats.insert(0, "r", ir)
    feats.insert(0, "l", il)
    for c in feats.columns:
        if c in FIELD_LEVELS or c in ("name_first_cmp", "name_last_cmp"):
            feats[c] = feats[c].astype(np.int8)
        elif c.startswith("veto_"):
            feats[c] = feats[c].astype(bool)
    return feats


def coparty_proxy_levels(left, right, il, ir):
    """Co-party level for pairs used in estimation, where business links are not scored yet:
    'anchored' when the two tied businesses share a TIN or clinic NPI. Scoring uses the real
    anchors (business links on an identifier at p >= coparty_min_p, no veto)."""
    tl = left["tie_pos"].to_numpy()[il]
    tr = right["tie_pos"].to_numpy()[ir]
    has = (tl >= 0) & (tr >= 0)
    level = np.full(len(il), EMPTY, dtype=np.int8)
    if has.any():
        bl, br = tl[has], tr[has]
        agree = np.zeros(len(bl), dtype=bool)
        for c in ("id_tin", "id_cnpi"):
            a = left[c].to_numpy()[bl]
            b = right[c].to_numpy()[br]
            agree |= (a == b) & (a != "")
        level[has] = np.where(agree, 0, 1)
    return level

In [6]:
# ---- Matching core v1.0: parameter estimation ------------------------------------------------
# u from random pairs (Splink's method), with the name sub-part rates the value-specific u
# needs; m from level counts per source, EM with u fixed, a shrinkage chain across sources,
# and the prior over all pairs.


def level_counts(levels, fields, weights=None, exclude=None):
    """Counts of each level among informative pairs (level != EMPTY). `exclude[field]` is a
    boolean mask of pairs whose value of that field must not count (the anchor's own field).
    Returns {field: (counts array per level, informative total)}."""
    out = {}
    w = np.ones(len(levels)) if weights is None else np.asarray(weights, dtype=float)
    for f in fields:
        if f not in levels:
            continue
        lv = levels[f].to_numpy()
        ok = lv >= 0
        if exclude is not None and f in exclude:
            ok &= ~exclude[f]
        k = len(FIELD_LEVELS[f])
        counts = np.bincount(lv[ok].astype(np.int64), weights=w[ok], minlength=k)[:k]
        out[f] = (counts, float(w[ok].sum()))
    return out


def estimate_u(levels_random, fields, params):
    """Field-level u per level from random pairs, smoothed by a pseudo-count."""
    rows = []
    cnt = level_counts(levels_random, fields)
    for f in fields:
        counts, n = cnt.get(f, (np.zeros(len(FIELD_LEVELS[f])), 0.0))
        k = len(counts)
        u = (counts + params.u_pseudo) / (n + params.u_pseudo * k)
        for i, lev in enumerate(FIELD_LEVELS[f]):
            rows.append({"field": f, "level": lev, "level_code": i, "u": float(u[i]),
                         "u_count": float(counts[i]), "u_pairs": float(n),
                         "u_source": f"random pairs ({int(n):,} informative)"})
    return pd.DataFrame(rows)


def name_components(levels_random):
    """Random-pair rates of the name sub-parts: the first-name part given informative names,
    and 'last name close'. Used to build value-specific u for the partial name levels."""
    ok = levels_random["name"].to_numpy() >= 0 if "name" in levels_random else np.zeros(0, bool)
    n = max(1, int(ok.sum()))
    fc = levels_random["name_first_cmp"].to_numpy()[ok] if ok.any() else np.array([], dtype=int)
    lc = levels_random["name_last_cmp"].to_numpy()[ok] if ok.any() else np.array([], dtype=int)
    rate = lambda mask: (float(mask.sum()) + 0.5) / (n + 1.0)
    return {"first_nick_or_close": rate(fc == 1), "first_initial": rate(fc == 2),
            "first_empty": rate(fc == 3), "first_differs": rate(fc == 4),
            "last_close": rate(lc == 1), "pairs": n}


def em_fixed_u(levels, free_fields, fixed_fields, m_fixed, u_table, params, exclude=None):
    """Expectation-maximization over the pairs in `levels`, with u fixed for every field and m
    fixed for `fixed_fields`; learns m for `free_fields` and the match share lambda.
    `m_fixed[f]` and `u_table[f]` are arrays per level. Returns ({field: (m array, n_eff)},
    lambda, iterations)."""
    n = len(levels)
    if n == 0:
        return {}, float("nan"), 0
    fields = [f for f in list(free_fields) + list(fixed_fields) if f in levels]
    lv = {f: levels[f].to_numpy().astype(np.int64) for f in fields}
    ok = {f: (lv[f] >= 0) & (~exclude[f] if exclude is not None and f in exclude else True)
          for f in fields}
    m = {f: (np.asarray(m_fixed[f], dtype=float) if f in fixed_fields
             else np.linspace(0.9, 0.1, len(FIELD_LEVELS[f])) / np.linspace(0.9, 0.1, len(FIELD_LEVELS[f])).sum())
         for f in fields}
    u = {f: np.asarray(u_table[f], dtype=float) for f in fields}
    lam, it = 0.5, 0
    for it in range(1, params.em_max_iter + 1):
        log_m = np.zeros(n)
        log_u = np.zeros(n)
        for f in fields:
            idx = np.where(ok[f], lv[f], 0)
            log_m += np.where(ok[f], np.log(np.clip(m[f][idx], 1e-12, 1)), 0.0)
            log_u += np.where(ok[f], np.log(np.clip(u[f][idx], 1e-12, 1)), 0.0)
        a = np.log(lam) + log_m
        b = np.log(1 - lam) + log_u
        w = 1.0 / (1.0 + np.exp(np.clip(b - a, -700, 700)))
        new_lam = float(np.clip(w.mean(), 1e-6, 1 - 1e-6))
        delta = abs(new_lam - lam)
        for f in free_fields:
            if f not in lv:
                continue
            k = len(FIELD_LEVELS[f])
            c = np.bincount(lv[f][ok[f]], weights=w[ok[f]], minlength=k)[:k]
            tot = c.sum()
            if tot > 0:
                newm = (c + 1e-3) / (tot + 1e-3 * k)
                delta = max(delta, float(np.abs(newm - m[f]).max()))
                m[f] = newm
        lam = new_lam
        if delta < params.em_tol:
            break
    out = {}
    for f in free_fields:
        if f in lv:
            out[f] = (m[f], float(w[ok[f]].sum()), np.bincount(lv[f][ok[f]], weights=w[ok[f]],
                                                             minlength=len(FIELD_LEVELS[f]))[:len(FIELD_LEVELS[f])])
    return out, lam, it


def combine_m(field_, sources, published, params):
    """One field's m per level from sources in preference order.

    `sources`: list of (name, counts array, informative n) best first. Each estimate is shrunk
    toward the next one (alpha pseudo-pairs), bottom-up from the published value. Simulation
    (a source named 'simulation') joins the chain only when the sources before it hold fewer
    than n_min informative pairs. Returns rows with m, the chain, pairs and a 95% interval."""
    k = len(FIELD_LEVELS[field_])
    pub = np.asarray(published, dtype=float)
    pub = pub / pub.sum() if pub.sum() > 0 else np.full(k, 1.0 / k)
    before_sim = 0.0
    used = []
    for name, counts, n in sources:
        if name == "simulation" and before_sim >= params.n_min:
            continue
        if n > 0:
            used.append((name, np.asarray(counts, dtype=float), float(n)))
        if name != "simulation":
            before_sim += n
    m = pub.copy()
    for name, counts, n in reversed(used):
        m = (counts + params.alpha * m) / (n + params.alpha)
    n_total = sum(n for _, _, n in used)
    n_eff = n_total + params.alpha
    se = np.sqrt(np.clip(m * (1 - m), 0, None) / n_eff)
    chain = " > ".join(f"{name}({n:,.0f})" for name, _, n in used) or "published only"
    primary = used[0][0] if used else "published"
    rows = []
    for i, lev in enumerate(FIELD_LEVELS[field_]):
        rows.append({"field": field_, "level": lev, "level_code": i, "m": float(m[i]),
                     "m_lo": float(max(0.0, m[i] - params.ci_z * se[i])),
                     "m_hi": float(min(1.0, m[i] + params.ci_z * se[i])),
                     "m_source": primary, "m_chain": chain, "m_pairs": float(n_total),
                     "m_published": float(pub[i])})
    return rows


def weights_table(m_rows, u_df, components):
    """Join m and u per field and level; field-level bits; flag agreement levels where m < u
    (those count 0 bits). Value-specific levels say what their u rests on."""
    w = pd.DataFrame(m_rows).merge(u_df, on=["field", "level", "level_code"], how="left")
    w["u"] = w["u"].fillna(1.0)
    w["agreement"] = [lev in AGREEMENT_LEVELS.get(f, set()) for f, lev in zip(w["field"], w["level"])]
    w["zero_by_rule"] = [lev in ZERO_LEVELS.get(f, set()) for f, lev in zip(w["field"], w["level"])]
    w["bits_field_level"] = np.log2(np.clip(w["m"], 1e-12, 1) / np.clip(w["u"], 1e-12, 1))
    w["disagreement"] = [lev in DISAGREEMENT_LEVELS.get(f, set()) for f, lev in zip(w["field"], w["level"])]
    w["flag"] = np.where(w["agreement"] & (w["m"] < w["u"]), "m<u: agreement counts 0 bits",
                         np.where(w["disagreement"] & (w["m"] > w["u"]), "m>u: disagreement counts 0 bits", ""))
    w["u_value_specific"] = [value_specific(f, lev) for f, lev in zip(w["field"], w["level"])]
    return w


VALUE_U_SOURCE = {
    ("name", "exact"): "census surname x SSA first name (per value)",
    ("name", "first_nick_or_close"): "census surname (per value) x random-pair rate of a nickname/close first name",
    ("name", "last_close_first_agrees"): "random-pair rate of a close surname x SSA first name (per value)",
    ("name", "initial_agrees"): "census surname x SSA share of the initial (per value)",
    ("name", "swapped"): "census x SSA for the swapped values",
    ("name", "first_empty"): "census surname (per value) x random-pair rate of an empty first name",
    ("name", "first_differs"): "census surname (per value) x random-pair rate of a differing first name",
    ("org", "exact"): "NPPES / reference-population word shares (per value)",
    ("org", "dba"): "NPPES / reference-population word shares (per value)",
    ("org", "short_form"): "NPPES / reference-population word shares of the shared words",
    ("org", "rare_shared"): "NPPES / reference-population share of the rarest shared word",
    ("dob", "exact"): "reference population: holders of the date",
    ("address", "exact"): "reference population: holders of the address",
    ("address", "street"): "reference population: holders of number + street",
    ("address", "zip"): "reference population: holders of the ZIP",
    ("address", "city_state"): "reference population: holders of city + state",
    ("address", "state"): "reference population: holders of the state",
    ("spec_cat", "specialty"): "reference population: holders of the specialty",
    ("spec_cat", "category_id"): "reference population: holders of the category",
    ("spec_cat", "category_weak"): "reference population: holders of the category",
}
for _f in IDENTIFIER_FIELDS:
    VALUE_U_SOURCE[(_f, "exact" if _f != "phone" else "exact_owned_single")] = "distinct names holding the value / reference holders of the field"
VALUE_U_SOURCE[("phone", "exact_shared")] = "distinct names holding the value / reference holders of the field"


def value_specific(f, lev):
    return VALUE_U_SOURCE.get((f, lev), "")


def strict_recall(fill, m_lookup):
    """How often a true match fires at least one strict rule, from field fill rates and the
    estimated m (rules treated as independent). `fill[key]`: share of pairs where both sides
    hold the field; `m_lookup(field, level)`: estimated m."""
    q = []
    ids = [f for f in ("ssn", "npi", "dl", "tin", "license", "email", "vin", "plate", "cnpi")]
    for f in ids:
        if fill.get(f, 0) > 0:
            q.append(fill[f] * m_lookup(f, "exact"))
    if fill.get("name", 0) > 0:
        q.append(fill["name"] * fill.get("dob", 0) * m_lookup("name", "exact") * m_lookup("dob", "exact"))
        q.append(fill["name"] * fill.get("street", 0) * m_lookup("name", "exact")
                 * (m_lookup("address", "exact") + m_lookup("address", "street")))
    if fill.get("org", 0) > 0:
        q.append(fill["org"] * fill.get("zip", 0) * m_lookup("org", "exact")
                 * (m_lookup("address", "exact") + m_lookup("address", "zip")))
        q.append(fill["org"] * fill.get("street", 0) * m_lookup("org", "exact")
                 * (m_lookup("address", "exact") + m_lookup("address", "street")))
    return float(1 - np.prod([1 - min(1.0, x) for x in q])) if q else 0.0


def prior_estimate(n_strict, n_pairs, recall, params):
    """P(a random pair of the group matches) = strict pairs / recall / all pairs. A zero count
    uses a pseudo-count and is flagged."""
    flagged = n_strict == 0 or recall <= 0
    num = (n_strict if n_strict > 0 else params.prior_pseudo) / max(recall, 1e-3)
    prior = float(min(0.5, num / max(1.0, float(n_pairs))))
    return prior, flagged

In [7]:
# ---- Matching core v1.0: scoring, vetoes, basis, co-party, evidence -------------------------


def _lookup_table(w, f, col):
    sub = w[w["field"] == f].sort_values("level_code")
    return sub[col].to_numpy(dtype=float)


def score_pairs(levels, part, weights, prior_logit):
    """Fellegi-Sunter in bits. For each field: log2(m / u), where u is the value-specific u on
    levels that have one and the field-level u otherwise; an empty field is 0 bits; an
    agreement level never goes below 0; levels declared 'no evidence' are 0. Vetoes set p = 0
    and stay visible. Returns a DataFrame: bits_<field>, bits, logit, p, veto, basis."""
    n = len(levels)
    out = pd.DataFrame(index=levels.index)
    total = np.zeros(n)
    for f in PART_FIELDS[part]:
        if f not in levels:
            out[f"bits_{f}"] = 0.0
            continue
        lv = levels[f].to_numpy().astype(np.int64)
        mt = _lookup_table(weights, f, "m")
        ut = _lookup_table(weights, f, "u")
        agree = np.array([lev in AGREEMENT_LEVELS.get(f, set()) for lev in FIELD_LEVELS[f]])
        zero = np.array([lev in ZERO_LEVELS.get(f, set()) for lev in FIELD_LEVELS[f]])
        vspec = np.array([bool(value_specific(f, lev)) for lev in FIELD_LEVELS[f]])
        idx = np.where(lv >= 0, lv, 0)
        uv = levels[f"uv_{f}"].to_numpy(dtype=float) if f"uv_{f}" in levels else np.full(n, np.nan)
        use_v = vspec[idx] & ~np.isnan(uv)
        u = np.where(use_v, uv, ut[idx])
        u = np.clip(u, 1e-15, 1.0)
        bits = np.log2(np.clip(mt[idx], 1e-15, 1.0) / u)
        disagree = np.array([lev in DISAGREEMENT_LEVELS.get(f, set()) for lev in FIELD_LEVELS[f]])
        bits = np.where(agree[idx], np.maximum(bits, 0.0), bits)
        bits = np.where(disagree[idx], np.minimum(bits, 0.0), bits)
        bits = np.where(zero[idx] | (lv < 0), 0.0, bits)
        out[f"bits_{f}"] = bits
        total += bits
    veto = np.full(n, "", dtype=object)
    for f in VETO_FIELDS:
        c = f"veto_{f}"
        if c in levels:
            v = levels[c].to_numpy(dtype=bool)
            veto = np.where(v, np.where(veto == "", f, veto + "+" + f), veto)
    out["bits"] = total
    out["logit"] = np.asarray(prior_logit, dtype=float) + total
    p = 1.0 / (1.0 + np.exp2(-np.clip(out["logit"].to_numpy(), -1000, 1000)))
    out["p"] = np.where(veto != "", 0.0, p)
    out["veto"] = veto
    out["basis"] = basis_of(levels, out, part)
    return out


def basis_of(levels, bits, part):
    """What the probability rests on, never upgraded by the score:
    identifier > address > dob > co_party > contextual (name + location/specialty/category)
    > name_only > none."""
    n = len(bits)
    pos = lambda f: (bits[f"bits_{f}"].to_numpy() > 0) if f"bits_{f}" in bits else np.zeros(n, bool)
    lv = lambda f: levels[f].to_numpy() if f in levels else np.full(n, EMPTY)
    ident = np.zeros(n, bool)
    for f in IDENTIFIER_FIELDS:
        if f in PART_FIELDS[part] and f in levels:
            exact = lv(f) == 0        # phone: only an owned, single-holder number
            ident |= exact & pos(f)
    addr = pos("address") & np.isin(lv("address"), [0, 1])
    location = pos("address") & np.isin(lv("address"), [2, 3, 4])
    dob = pos("dob") & np.isin(lv("dob"), [0, 1]) if part == "person" else np.zeros(n, bool)
    cop = pos("co_party")
    name = pos("name") | pos("org")
    context = name & (location | pos("spec_cat"))
    return np.select([ident, addr, dob & name, cop & name, context, name],
                     ["identifier", "address", "dob", "co_party", "contextual", "name_only"],
                     default="none").astype(object)


def apply_coparty(levels, scored, weights, prior_logit, anchored):
    """One-way co-party evidence for person pairs. `anchored`: boolean array, True where the
    two persons' tied businesses are linked on an identifier (p >= coparty_min_p, no veto),
    decided on business scores alone. Bits are added only where the names already agree, so
    a co-party can strengthen a name but never make one, and name-only links never vouch for
    each other. Returns re-scored pairs."""
    lev = levels.copy()
    has_tie = lev["co_party"].to_numpy() >= 0 if "co_party" in lev else np.zeros(len(lev), bool)
    name_agrees = scored["bits_name"].to_numpy() > 0
    code = np.where(has_tie, 1, EMPTY)
    code = np.where(has_tie & anchored & name_agrees, 0, code)
    lev["co_party"] = code.astype(np.int8)
    lev["uv_co_party"] = np.nan
    return lev, score_pairs(lev, "person", weights, prior_logit)


def evidence_rows(pair_ids, levels, scored, part, weights, values_l, values_r):
    """Long evidence: one row per pair and non-empty field (plus vetoing fields), with both
    values, the level, m and u with their sources and pair counts, and the bits. The rows of
    a pair sum to its total bits (the self-tests check it)."""
    rows = []
    w = weights.set_index(["field", "level_code"])
    for f in PART_FIELDS[part]:
        if f not in levels:
            continue
        lv = levels[f].to_numpy()
        keep = lv >= 0
        if not keep.any():
            continue
        idx = np.flatnonzero(keep)
        codes = lv[idx]
        sub = w.loc[[(f, int(c)) for c in codes]]
        uv = levels[f"uv_{f}"].to_numpy(dtype=float)[idx] if f"uv_{f}" in levels else np.full(len(idx), np.nan)
        vs = np.array([bool(x) for x in sub["u_value_specific"].to_numpy()])
        u_used = np.where(vs & ~np.isnan(uv), uv, sub["u"].to_numpy())
        rows.append(pd.DataFrame({
            "pair_id": np.asarray(pair_ids)[idx], "field": f,
            "value_extracted": values_l.get(f, np.full(len(lv), ""))[idx] if f in values_l else "",
            "value_watchlist": values_r.get(f, np.full(len(lv), ""))[idx] if f in values_r else "",
            "level": [FIELD_LEVELS[f][c] for c in codes],
            "m": sub["m"].to_numpy(), "m_source": sub["m_chain"].to_numpy(),
            "m_pairs": sub["m_pairs"].to_numpy(),
            "u": u_used,
            "u_source": np.where(vs & ~np.isnan(uv), sub["u_value_specific"].to_numpy(), sub["u_source"].to_numpy()),
            "u_pairs": sub["u_pairs"].to_numpy(),
            "bits": scored[f"bits_{f}"].to_numpy()[idx],
            "veto": levels[f"veto_{f}"].to_numpy()[idx] if f"veto_{f}" in levels else False,
        }))
    if not rows:
        return pd.DataFrame(columns=["pair_id", "field", "value_extracted", "value_watchlist", "level",
                                     "m", "m_source", "m_pairs", "u", "u_source", "u_pairs", "bits", "veto"])
    ev = pd.concat(rows, ignore_index=True)
    return ev.sort_values(["pair_id", "field"], kind="stable").reset_index(drop=True)

*End of the matching core.*

## 0 · Setup

One `RunConfig` holds every path, seed, sample size, chunk size, cap, floor and K; the
matching core's own numbers sit inside it as `core` (`CoreParams`). The manifest dumps both.
No threshold appears anywhere else.

The dataset is chosen with the environment variable `RL_DATASET`:

| Value | Watchlist | Extracted |
|---|---|---|
| `synthetic` (default) | fictional, ~5,000 rows | fictional, ~3,000 rows with ~45 hand-written edge cases |
| `leie` | OIG LEIE (84,001 rows), exported locally into gitignored `data/` | noisy copies of 3,000 LEIE rows + 3,000 fictional parties |
| `scale` | 1,000,000 fictional rows | 300,000 fictional rows |
| `files` | `RL_WATCHLIST` | `RL_EXTRACTED` |

**Where Splink would do better.** Splink's settings dictionary is a declarative, versioned
model: blocking rules, comparisons and trained parameters serialize to one JSON and reload.
Here the configuration is a dataclass and the trained parameters live in the manifest; a
saved model cannot yet be reloaded to score new data without re-estimating.

In [8]:
import os, sys, time, json, hashlib, platform, gc
from dataclasses import dataclass, field, asdict
from pathlib import Path


def _find_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "mappings" / "columns.csv").exists() and (p / "reference").exists():
            return p
    return here


@dataclass
class RunConfig:
    dataset: str = "synthetic"
    root: str = ""
    seed: int = 20260925
    # sample sizes
    random_pairs: int = 1_000_000        # random extracted x watchlist pairs for u (per part, split by share)
    sim_records: int = 50_000            # watchlist records copied with noise for simulated m
    anchor_pairs_per_value: int = 50     # cap per anchoring value (strict anchors)
    loose_holders: tuple = (2, 20)       # holders of a value for the loose anchor set
    # candidates
    chunk_size: int = 20_000             # extracted parties per indexing chunk
    block_pair_cap: int = 200_000        # a blocking key proposing more pairs is refined by state
    sn_window: int = 5                   # sorted-neighbourhood window
    # keeping and output
    keep_p_floor: float = 1e-3           # keep a pair at or above this p ...
    top_k: int = 3                       # ... or among its entity's top K (ties kept) ...
    #                                      ... or when it rests on an identifier or is vetoed
    excel_row_limit: int = 1_048_575     # data rows per sheet before a continuation sheet
    evidence_in_workbook: bool = True
    sensitivity_prior_mult: tuple = (0.1, 1.0, 10.0)
    sensitivity_recall: tuple = (0.5, 1.0)   # multiplied into the estimated recall, capped at 1
    p_bands: tuple = (0.5, 0.8, 0.9)
    # synthetic and benchmark sizes
    synth_persons: int = 2_000
    synth_businesses: int = 500
    leie_noisy_copies: int = 3_000
    leie_fictional: int = 3_000
    scale_watchlist: int = 1_000_000
    scale_extracted: int = 300_000
    core: CoreParams = field(default_factory=CoreParams)

    @property
    def paths(self):
        r = Path(self.root)
        return {"root": r, "mappings": r / "mappings", "reference": r / "reference",
                "data": r / "data", "out": r / "out" / self.dataset, "review": r / "review",
                "notebook": r / "record_linkage.ipynb"}


def make_config(dataset=None, **overrides):
    ds = dataset or os.environ.get("RL_DATASET", "synthetic")
    cfg = RunConfig(dataset=ds, root=str(_find_root()))
    if ds == "leie":
        cfg.random_pairs = 2_000_000
    elif ds == "scale":
        cfg.random_pairs = 5_000_000
    for k, v in overrides.items():
        setattr(cfg, k, v)
    for key in ("data", "out", "review"):
        cfg.paths[key].mkdir(parents=True, exist_ok=True)
    return cfg


def config_records(cfg):
    """The run configuration as manifest rows."""
    d = asdict(cfg)
    core = d.pop("core")
    rows = [{"section": "config", "key": k, "value": json.dumps(v) if not isinstance(v, str) else v,
             "source": "RunConfig"} for k, v in d.items()]
    rows += [{"section": "config.core", "key": k, "value": json.dumps(v), "source": "CoreParams"}
             for k, v in core.items()]
    return rows


class Stopwatch:
    """Wall time and peak resident memory per step (peak sampled after each step)."""

    def __init__(self):
        self.t0 = time.time()
        self.rows = []
        self._last = self.t0
        try:
            import psutil
            self._proc = psutil.Process()
        except ImportError:
            self._proc = None
        self.peak = 0

    def rss(self):
        if self._proc is None:
            return 0
        try:
            info = self._proc.memory_info()
            peak = getattr(info, "peak_wset", None) or info.rss   # Windows: peak working set
            return max(info.rss, peak)
        except Exception:
            return 0

    def mark(self, step):
        now = time.time()
        r = self.rss()
        self.peak = max(self.peak, r)
        self.rows.append({"step": step, "seconds": round(now - self._last, 2),
                          "elapsed": round(now - self.t0, 2), "peak_rss_gb": round(self.peak / 1e9, 3)})
        self._last = now
        print(f"[{now - self.t0:7.1f}s  peak {self.peak / 1e9:5.2f} GB] {step}")

## 1 · Mappings, reference tables and inputs

**Mappings** (`mappings/*.csv`) are the reviewable tables: column routing, category maps,
keyword rules, specialty synonyms, the simulation noise table and the published starting
values of m. **Reference tables** (`reference/`) are copied outside tables, checked against
the SHA-256 recorded in `reference/SOURCES.md`. **Inputs** are two CSVs with the schema in
DESIGN.md: `record_id` is required and unique per file, every other column may be empty,
missing schema columns are added empty, and unknown columns are kept as `x_<name>`
(displayed, never compared). All-empty columns are reported.

**Where Splink would do better.** Splink reads Parquet or a database table straight into
DuckDB without a pandas copy, so a 1M-row input costs little memory. Here pandas holds the
rows as text.

In [9]:
import csv, gzip, re

SCHEMA = ["record_id", "claim_id", "note_id", "category", "first_name", "middle_name",
          "last_name", "dob", "ssn", "driver_license_number", "driver_license_state",
          "provider_npi", "professional_license_number", "professional_license_state",
          "professional_license_type", "provider_specialty", "business_name", "tin", "clinic_npi",
          "street_number", "street_direction", "street_name", "street_type", "unit", "city",
          "state", "zip", "home_phone", "work_phone", "email", "vin", "plate_number", "plate_state"]


class InputError(ValueError):
    pass


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_mappings(mdir):
    """The reviewable tables, validated: known categories only, rates in [0, 1], m per field
    summing to 1."""
    mdir = Path(mdir)
    read = lambda n: pd.read_csv(mdir / n, dtype=str, keep_default_na=False)
    maps = {"columns": read("columns.csv"), "category_map": read("category_map.csv"),
            "category_keywords": read("category_keywords.csv"),
            "specialty_canon": read("specialty_canon.csv"),
            "simulation_noise": read("simulation_noise.csv"), "published_m": read("published_m.csv")}
    cats = {"medical", "legal", "repair shop", "witness", "claimant", "other"}
    for n in ("category_map", "category_keywords"):
        bad = set(maps[n]["category"]) - cats
        if bad:
            raise InputError(f"{n}.csv: unknown categories {sorted(bad)}")
    missing = set(SCHEMA) - set(maps["columns"]["column"])
    if missing:
        raise InputError(f"columns.csv does not route {sorted(missing)}")
    rates = maps["simulation_noise"]["rate"].astype(float)
    if ((rates < 0) | (rates > 1)).any():
        raise InputError("simulation_noise.csv: a rate outside [0, 1]")
    pm = maps["published_m"].assign(m=lambda d: d["m"].astype(float))
    for f, levels in FIELD_LEVELS.items():
        sub = pm[pm["field"] == f]
        if set(sub["level"]) != set(levels):
            raise InputError(f"published_m.csv: field {f} needs levels {levels}")
        if abs(sub["m"].sum() - 1) > 1e-6:
            raise InputError(f"published_m.csv: m for {f} sums to {sub['m'].sum():.4f}, not 1")
    maps["published_m"] = pm
    return maps


def load_reference(rdir, params):
    """Outside tables as plain dicts, plus a hash check against reference/SOURCES.md."""
    rdir = Path(rdir)

    def counts(fname):
        p = rdir / fname
        if not p.exists():
            return None
        with gzip.open(p, "rt", encoding="utf-8") as fh:
            rd = csv.reader(fh); next(rd)
            return {k: int(v) for k, v in rd}

    meta_p = rdir / "reference_meta.json"
    meta = json.loads(meta_p.read_text(encoding="utf-8")) if meta_p.exists() else {}
    nick_p = rdir / "nicknames.csv"
    nick = pd.read_csv(nick_p, dtype=str, keep_default_na=False) if nick_p.exists() else None
    recorded = {}
    src = rdir / "SOURCES.md"
    if src.exists():
        for line in src.read_text(encoding="utf-8").splitlines():
            m = re.match(r"\|\s*`([^`]+)`\s*\|.*`([0-9a-f]{64})`\s*\|\s*$", line)
            if m:
                recorded[m.group(1)] = m.group(2)
    checks = []
    for name, want in sorted(recorded.items()):
        p = rdir / name
        got = sha256_file(p) if p.exists() else ""
        checks.append({"file": name, "recorded": want, "actual": got,
                       "status": "ok" if got == want else ("missing" if not got else "CHANGED")})
    return {"surnames": counts("surnames.csv.gz"), "first_names": counts("first_names.csv.gz"),
            "org_tokens": counts("org_tokens.csv.gz"), "meta": meta, "nicknames": nick,
            "hash_checks": pd.DataFrame(checks)}


def load_input(path, source):
    """One input CSV as text. Returns (frame, report)."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_filter=False, encoding="utf-8")
    df.columns = [c.strip() for c in df.columns]
    return validate_input(df, source)


def validate_input(df, source):
    if "record_id" not in df.columns:
        raise InputError(f"{source}: record_id column is required")
    df = df.copy()
    df["record_id"] = df["record_id"].astype(str).str.strip()
    empty_ids = int((df["record_id"] == "").sum())
    if empty_ids:
        raise InputError(f"{source}: {empty_ids} rows have no record_id")
    dup = df["record_id"][df["record_id"].duplicated(keep=False)]
    if len(dup):
        raise InputError(f"{source}: record_id not unique, e.g. {sorted(set(dup))[:5]}")
    unknown = [c for c in df.columns if c not in SCHEMA]
    df = df.rename(columns={c: (c if c.startswith("x_") else f"x_{c}") for c in unknown})
    added = [c for c in SCHEMA if c not in df.columns]
    for c in added:
        df[c] = ""
    for c in df.columns:
        df[c] = df[c].fillna("").astype(str)
    all_null = [c for c in SCHEMA if c != "record_id" and (df[c].str.strip() == "").all()]
    report = {"source": source, "rows": len(df), "added_columns": added,
              "passthrough_columns": [c for c in df.columns if c.startswith("x_")],
              "all_empty_columns": all_null}
    return df[SCHEMA + [c for c in df.columns if c.startswith("x_")]], report

## 2 · Normalize

Every column is normalized per `mappings/columns.csv` with the core's normalizers, once per
distinct value (a name that appears 3,000 times is parsed once). Each identifier gets a
validity flag and a reason: NPI Luhn check, VIN check digit, SSN ranges, EIN prefixes,
placeholder values such as 000000000, 123456789 or 1900-01-01. Invalid values stay visible in
the details table and are never compared.

**Where Splink would do better.** Splink leaves cleaning to the user but its comparison
library normalizes inside SQL, in parallel. Here name and address parsing are Python loops
over distinct values; at 1.3M rows they are the second-largest cost after comparisons.

In [10]:
ID_TYPES = ["ssn", "npi", "dl", "license", "tin", "cnpi", "email", "vin", "plate"]


def _norm3(series, func):
    """Apply a (value, valid, reason) normalizer once per distinct value."""
    res = map_unique(series, func)
    return (res.map(lambda t: t[0]), res.map(lambda t: t[1]), res.map(lambda t: t[2]))


def normalize_rows(df, maps, nick, params, source):
    """Row-level normalized frame (same index as df)."""
    out = pd.DataFrame({"record_id": df["record_id"], "source": source,
                        "claim_id": df["claim_id"].str.strip(), "note_id": df["note_id"].str.strip()})
    key = df["first_name"] + "\x1f" + df["middle_name"] + "\x1f" + df["last_name"]
    parts = map_unique(key, lambda k: clean_person(*k.split("\x1f")))
    out["first"] = parts.map(lambda t: t[0])
    out["middle"] = parts.map(lambda t: t[1])
    out["last"] = parts.map(lambda t: t[2])
    out["business_raw"] = df["business_name"].map(fold)
    out["org_aliases"] = map_unique(df["business_name"], org_alias_string)
    out["dob"], out["dob_valid"], out["dob_reason"] = _norm3(df["dob"], lambda v: norm_dob(v, params))
    out["ssn"], out["ssn_valid"], out["ssn_reason"] = _norm3(df["ssn"], norm_ssn)
    out["npi"], out["npi_valid"], out["npi_reason"] = _norm3(df["provider_npi"], norm_npi)
    out["tin"], out["tin_valid"], out["tin_reason"] = _norm3(df["tin"], norm_tin)
    out["cnpi"], out["cnpi_valid"], out["cnpi_reason"] = _norm3(df["clinic_npi"], norm_npi)
    out["email"], out["email_valid"], out["email_reason"] = _norm3(df["email"], norm_email)
    out["vin"], out["vin_valid"], out["vin_reason"] = _norm3(df["vin"], norm_vin)
    q = lambda a, b: map_unique(df[a] + "\x1f" + df[b], lambda k: k.split("\x1f"))
    out["dl"], out["dl_valid"], out["dl_reason"] = _norm3(
        df["driver_license_number"] + "\x1f" + df["driver_license_state"], lambda k: norm_dl(*k.split("\x1f")))
    out["license"], out["license_valid"], out["license_reason"] = _norm3(
        df["professional_license_number"] + "\x1f" + df["professional_license_state"],
        lambda k: norm_license(*k.split("\x1f")))
    out["plate"], out["plate_valid"], out["plate_reason"] = _norm3(
        df["plate_number"] + "\x1f" + df["plate_state"], lambda k: norm_plate(*k.split("\x1f")))
    out["home_phone"], out["home_phone_valid"], out["home_phone_reason"] = _norm3(df["home_phone"], norm_phone)
    out["work_phone"], out["work_phone_valid"], out["work_phone_reason"] = _norm3(df["work_phone"], norm_phone)
    out["license_type"] = df["professional_license_type"].map(fold)
    canon = dict(zip(maps["specialty_canon"]["raw"].map(fold), maps["specialty_canon"]["canonical"].map(fold)))
    out["specialty_raw"] = df["provider_specialty"].map(fold)
    out["specialty"] = out["specialty_raw"].map(lambda s: canon.get(s, s))
    out["category_given"] = df["category"].str.strip().str.lower()
    akey = (df["street_number"] + "\x1f" + df["street_direction"] + "\x1f" + df["street_name"] + "\x1f"
            + df["street_type"] + "\x1f" + df["unit"])
    a = map_unique(akey, lambda k: norm_street_parts(*k.split("\x1f")))
    out["addr_number"] = a.map(lambda t: t[0]); out["addr_dir"] = a.map(lambda t: t[1])
    out["addr_name"] = a.map(lambda t: t[2]); out["addr_type"] = a.map(lambda t: t[3])
    out["addr_unit"] = a.map(lambda t: t[4])
    out["city"] = map_unique(df["city"], norm_city)
    out["state"] = map_unique(df["state"], norm_state)
    out["zip"] = map_unique(df["zip"], norm_zip)
    keys = map_unique(out["addr_number"] + "\x1f" + out["addr_name"] + "\x1f" + out["addr_unit"] + "\x1f"
                      + out["zip"] + "\x1f" + out["city"] + "\x1f" + out["state"],
                      lambda k: address_keys(*k.split("\x1f")))
    out["addr_full"] = keys.map(lambda t: t[0]); out["addr_street"] = keys.map(lambda t: t[1])
    out["city_state"] = keys.map(lambda t: t[3])
    xcols = [c for c in df.columns if c.startswith("x_")]
    for c in xcols:
        out[c] = df[c]
    # raw display values
    out["raw_name"] = (df["first_name"] + " " + df["middle_name"] + " " + df["last_name"]).str.split().str.join(" ")
    out["raw_business"] = df["business_name"].str.strip()
    out["raw_address"] = (df["street_number"] + " " + df["street_direction"] + " " + df["street_name"] + " "
                          + df["street_type"] + " " + df["unit"] + ", " + df["city"] + " " + df["state"]
                          + " " + df["zip"]).str.split().str.join(" ").str.strip(", ")
    return out.reset_index(drop=True)

## 3 · Breakdown

Each row yields up to two parties, tied by the row: a **person part** when there is a first
or last name, and a **business part** when there is a business name, or a TIN or clinic NPI
with no business name. Details go to their owner per `mappings/columns.csv`: address and work
phone are held by both parts with ownership `row`; the email goes to the person, or to the
business when the row has no person. Rows that yield no party are reported, never dropped
silently.

**Where Splink would do better.** Splink links records, not parts of records; the split is
ours. What Splink would add is its array comparisons for multi-valued fields (a party with
several phones), which this schema does not need because a row holds one value per column.

In [11]:
def breakdown(rows, source, nick):
    """Rows -> (parties, details, ties, empty_rows)."""
    has_person = (rows["first"] != "") | (rows["last"] != "")
    raw_tin_or_cnpi = (rows["tin"] != "") | (rows["cnpi"] != "")
    has_business = (rows["org_aliases"] != "") | raw_tin_or_cnpi
    pi = np.flatnonzero(has_person.to_numpy())
    bi = np.flatnonzero(has_business.to_numpy())
    empty = rows.loc[~(has_person | has_business), ["record_id"]]

    def base(idx, part):
        r = rows.iloc[idx]
        tag = "P" if part == "person" else "B"
        p = pd.DataFrame({"party_id": source + ":" + r["record_id"] + ":" + tag,
                          "source": source, "record_id": r["record_id"].to_numpy(), "part": part,
                          "row": idx, "claim_id": r["claim_id"].to_numpy(),
                          "note_id": r["note_id"].to_numpy()})
        return p

    P = base(pi, "person")
    B = base(bi, "business")
    rp, rb = rows.iloc[pi], rows.iloc[bi]
    for c in ("first", "middle", "last"):
        P[c] = rp[c].to_numpy()
        B[c] = ""
    P["org_aliases"] = ""
    B["org_aliases"] = rb["org_aliases"].to_numpy()
    P["name_key"] = np.where(P["first"] != "", P["first"] + " " + P["last"], P["last"])
    B["name_key"] = [a.split("|")[0] if a else "" for a in B["org_aliases"]]
    # who holds a value: surname + first initial, so 'R. Smith' and 'Robert Smith' are one holder
    P["holder_key"] = P["last"] + "|" + P["first"].str[:1]
    B["holder_key"] = B["name_key"]
    P["display_name"] = rp["raw_name"].to_numpy()
    B["display_name"] = np.where(rb["raw_business"].to_numpy() != "", rb["raw_business"].to_numpy(),
                                 "(no name: " + np.where(rb["tin"].to_numpy() != "", "TIN " + rb["tin"].to_numpy(),
                                                         "clinic NPI " + rb["cnpi"].to_numpy()) + ")")
    P["first_roots"] = map_unique(P["first"], nick.roots_string).to_numpy()
    B["first_roots"] = ""
    P["last_nysiis"] = map_unique(P["last"], nysiis).to_numpy()
    B["last_nysiis"] = ""
    # details owned by the person only
    for c, col in (("dob", "dob"),):
        P[c] = np.where(rp["dob_valid"].to_numpy(dtype=bool), rp["dob"].to_numpy(), "")
        B[c] = ""
    P["dob_int"] = [int(d.replace("-", "")) if d else 0 for d in P["dob"]]
    B["dob_int"] = 0
    valid = lambda r, t: np.where(r[f"{t}_valid"].to_numpy(dtype=bool), r[t].to_numpy(), "")
    for t in ("ssn", "npi", "dl", "license", "vin", "plate"):
        P[f"id_{t}"] = valid(rp, t)
        B[f"id_{t}"] = ""
    for t in ("tin", "cnpi"):
        B[f"id_{t}"] = valid(rb, t)
        P[f"id_{t}"] = ""
    # email: the person's, or the business's when the row has no person
    P["id_email"] = valid(rp, "email")
    B["id_email"] = np.where(has_person.to_numpy()[bi], "", valid(rb, "email"))
    P["phone_own"] = valid(rp, "home_phone")
    B["phone_own"] = ""
    P["phone_row"] = valid(rp, "work_phone")
    B["phone_row"] = valid(rb, "work_phone")
    for c in ("addr_full", "addr_street", "zip", "city_state", "state", "city", "addr_number",
              "addr_name", "addr_unit"):
        P[c] = rp[c].to_numpy()
        B[c] = rb[c].to_numpy()
    P["specialty"] = rp["specialty"].to_numpy()
    B["specialty"] = ""                        # provider_specialty is the person's
    P["specialty_raw_row"] = rp["specialty_raw"].to_numpy()
    B["specialty_raw_row"] = rb["specialty_raw"].to_numpy()   # category inference is row-level
    for c in ("license_type", "category_given", "raw_address"):
        P[c] = rp[c].to_numpy()
        B[c] = rb[c].to_numpy()
    P["business_raw"] = rp["business_raw"].to_numpy()
    B["business_raw"] = rb["business_raw"].to_numpy()
    parties = pd.concat([P, B], ignore_index=True)
    # ties: the other part of the same row
    tie = parties.groupby("row")["party_id"].agg(list)
    other = {}
    for ids in tie:
        if len(ids) == 2:
            other[ids[0]], other[ids[1]] = ids[1], ids[0]
    parties["tie"] = parties["party_id"].map(other).fillna("")
    parties = parties.sort_values(["row", "part"], kind="stable").reset_index(drop=True)
    ties = parties[(parties["part"] == "person") & (parties["tie"] != "")][["record_id", "party_id", "tie"]]
    ties = ties.rename(columns={"party_id": "person_party", "tie": "business_party"}).reset_index(drop=True)
    details = build_details(rows, parties, pi, bi, has_person.to_numpy())
    return parties, details, ties, empty


DETAIL_SPEC = [  # (type, row column, owner, ownership, source column)
    ("dob", "dob", "person", "own", "dob"),
    ("ssn", "ssn", "person", "own", "ssn"),
    ("npi", "npi", "person", "own", "provider_npi"),
    ("dl", "dl", "person", "own", "driver_license_number"),
    ("license", "license", "person", "own", "professional_license_number"),
    ("vin", "vin", "person", "own", "vin"),
    ("plate", "plate", "person", "own", "plate_number"),
    ("phone", "home_phone", "person", "own", "home_phone"),
    ("phone", "work_phone", "both", "row", "work_phone"),
    ("email", "email", "person_else_business", "own", "email"),
    ("tin", "tin", "business", "own", "tin"),
    ("cnpi", "cnpi", "business", "own", "clinic_npi"),
    ("address", "addr_full", "both", "row", "street_number..zip"),
]


def build_details(rows, parties, pi, bi, has_person):
    """Long details table: party, type, normalized value, validity, ownership, source column."""
    pid_p = parties.set_index(["row", "part"])["party_id"]
    person_of = pd.Series(pid_p.xs("person", level="part")) if len(pi) else pd.Series(dtype=object)
    business_of = pd.Series(pid_p.xs("business", level="part")) if len(bi) else pd.Series(dtype=object)
    out = []
    for typ, col, owner, own, src in DETAIL_SPEC:
        vals = rows[col]
        present = vals != ""
        vflag = rows[f"{col}_valid"] if f"{col}_valid" in rows else pd.Series(True, index=rows.index)
        reason = rows[f"{col}_reason"] if f"{col}_reason" in rows else pd.Series("", index=rows.index)
        targets = []
        if owner in ("person", "both", "person_else_business"):
            targets.append(person_of)
        if owner in ("business", "both"):
            targets.append(business_of)
        if owner == "person_else_business":
            targets.append(business_of[~pd.Series(has_person)[business_of.index].to_numpy()])
        for tgt in targets:
            idx = tgt.index[present.to_numpy()[tgt.index]] if len(tgt) else []
            if len(idx) == 0:
                continue
            out.append(pd.DataFrame({"party_id": tgt.loc[idx].to_numpy(), "type": typ,
                                     "value": vals.to_numpy()[idx], "valid": vflag.to_numpy()[idx].astype(bool),
                                     "reason": reason.to_numpy()[idx], "ownership": own,
                                     "source_column": src}))
    if not out:
        return pd.DataFrame(columns=["party_id", "type", "value", "valid", "reason", "ownership", "source_column"])
    return pd.concat(out, ignore_index=True)

## 4 · Value index

For every (type, value): how many parties hold it in each file, how many distinct names hold
it, whether one name holds it (single-holder: the only kind that can veto or anchor), and
whether it is junk (fails validation, or is held under more than `junk_holders` names). It
drives shared-identifier counts, anchors, per-value u and the row-level evidence. Junk values
are blanked from the party frames before any comparison and listed in the manifest.

**Where Splink would do better.** Splink computes term frequencies per column in SQL and
joins them onto pairs; the idea is the same, and Splink's version is lazier with memory.

In [12]:
def value_index(parties, details):
    """(type, value) -> holders per file, distinct names, single-holder and junk flags."""
    d = details.merge(parties[["party_id", "source", "name_key"]], on="party_id", how="left")
    g = d.groupby(["type", "value"], sort=True)
    vi = pd.DataFrame({
        "holders_extracted": g["source"].agg(lambda s: int((s == "X").sum())),
        "holders_watchlist": g["source"].agg(lambda s: int((s == "W").sum())),
        "distinct_names": g["name_key"].nunique(),
        "valid": g["valid"].all(),
        "invalid_reason": g["reason"].agg(lambda s: ",".join(sorted({x for x in s if x}))),
    }).reset_index()
    vi["single_holder"] = vi["valid"] & (vi["distinct_names"] == 1)
    return vi


def value_index_fast(parties, details):
    """Same as value_index, vectorized for large inputs."""
    d = details[["party_id", "type", "value", "valid", "reason"]].merge(
        parties[["party_id", "source", "holder_key"]].rename(columns={"holder_key": "name_key"}), on="party_id", how="left")
    d["is_x"] = (d["source"] == "X").astype(np.int64)
    d["is_w"] = (d["source"] == "W").astype(np.int64)
    g = d.groupby(["type", "value"], sort=True)
    vi = g.agg(holders_extracted=("is_x", "sum"), holders_watchlist=("is_w", "sum"),
               distinct_names=("name_key", "nunique"), valid=("valid", "all")).reset_index()
    bad = d[d["reason"] != ""].groupby(["type", "value"])["reason"].first()
    vi["invalid_reason"] = pd.MultiIndex.from_frame(vi[["type", "value"]]).map(bad.to_dict()).fillna("")
    vi["single_holder"] = vi["valid"] & (vi["distinct_names"] == 1)
    return vi


def mark_junk(vi, params):
    vi = vi.copy()
    many = vi["valid"] & (vi["distinct_names"] > params.junk_holders) & (vi["type"] != "address") & \
        (vi["type"] != "dob")
    vi["junk"] = ~vi["valid"] | many
    vi["junk_reason"] = np.where(~vi["valid"], "invalid:" + vi["invalid_reason"],
                                 np.where(many, "held under " + vi["distinct_names"].astype(str) + " names", ""))
    return vi


def blank_junk(parties, vi):
    """Blank identifier values held under too many names (invalid ones are already blank)."""
    many = vi[vi["junk"] & vi["valid"]]
    col = {"ssn": ["id_ssn"], "npi": ["id_npi"], "dl": ["id_dl"], "license": ["id_license"],
           "tin": ["id_tin"], "cnpi": ["id_cnpi"], "email": ["id_email"], "vin": ["id_vin"],
           "plate": ["id_plate"], "phone": ["phone_own", "phone_row"]}
    n = 0
    for t, cols in col.items():
        bad = set(many.loc[many["type"] == t, "value"])
        if not bad:
            continue
        for c in cols:
            m = parties[c].isin(bad)
            n += int(m.sum())
            parties.loc[m, c] = ""
    return n

## 5 · Category

**Extracted rows.** The given category is kept. An inferred category is added beside it,
with the rule that set it: a valid provider or clinic NPI → medical, a licence type that
`category_map.csv` maps to medical or legal → that category (both *identifier-backed*); a
business-name phrase from `category_keywords.csv` → its category (*keyword*, weaker, applied
to both parts of the row). Witness and claimant are never inferred; "other" counts as no
information. The category used in matching is identifier-backed → given → keyword.
A given category that disagrees with an inferred one is a data-quality finding; both stay
visible.

**Watchlist rows.** No category is given: it is inferred from the specialty, then the licence
type, through `category_map.csv`; unmapped values → unknown, listed in the manifest.

**Prior group** (from the extracted party): professional (medical, legal), business (repair
shop, or a business part unless identifier-backed medical/legal), private (witness,
claimant), unknown.

**Where Splink would do better.** Nothing specific: Splink has no category concept. Its
closest tool is blocking or training separately per group, which the prior groups mimic.

In [13]:
GROUPS = ["professional", "business", "private", "unknown"]


def _keyword_rules(kw):
    rules = []
    for phrase, cat in zip(kw["phrase"].map(fold), kw["category"]):
        rules.append((re.compile(r"(?<![A-Z0-9])" + re.escape(phrase) + r"(?![A-Z0-9])"), cat, phrase))
    return rules


def keyword_category(names, kw):
    """First matching phrase wins (file order). Returns (category, phrase) per name."""
    rules = _keyword_rules(kw)

    def one(s):
        s = _undot(fold(s))
        for rx, cat, phrase in rules:
            if rx.search(s):
                return cat, phrase
        return "", ""
    return map_unique(pd.Series(names), one)


def assign_category_extracted(parties, maps):
    cm = maps["category_map"]
    lic = dict(zip(cm.loc[cm["source_field"] == "license_type", "value"].map(fold),
                   cm.loc[cm["source_field"] == "license_type", "category"]))
    p = parties
    given = p["category_given"].where(p["category_given"].isin(
        ["medical", "legal", "repair shop", "witness", "claimant"]), "")
    npi_backed = (p["id_npi"] != "") | (p["id_cnpi"] != "")
    lic_cat = p["license_type"].map(lambda v: lic.get(v, "")).where(p["part"] == "person", "")
    lic_cat = lic_cat.where(lic_cat.isin(["medical", "legal"]), "")
    id_cat = np.where(npi_backed, "medical", lic_cat)
    id_rule = np.where(npi_backed, np.where(p["id_npi"] != "", "provider_npi", "clinic_npi"),
                       np.where(lic_cat != "", "license_type:" + p["license_type"], ""))
    kwc = keyword_category(p["business_raw"], maps["category_keywords"])
    kw_cat = kwc.map(lambda t: t[0]).to_numpy()
    kw_rule = np.where(kw_cat != "", "keyword:" + kwc.map(lambda t: t[1]).to_numpy(), "")
    inferred = np.where(id_cat != "", id_cat, kw_cat)
    inferred_rule = np.where(id_cat != "", id_rule, kw_rule)
    used = np.where(id_cat != "", id_cat, np.where(given != "", given, kw_cat))
    strength = np.where(id_cat != "", "strong", np.where(used != "", "weak", ""))
    used_rule = np.where(id_cat != "", id_rule, np.where(given != "", "given", kw_rule))
    p["category_inferred"] = inferred
    p["category_rule"] = inferred_rule
    p["category"] = used
    p["category_used_rule"] = used_rule
    p["cat_strength"] = strength
    p["category_mismatch"] = (given != "") & (inferred != "") & (given != inferred)
    person = p["part"] == "person"
    group = np.select(
        [person & np.isin(used, ["medical", "legal"]),
         person & np.isin(used, ["witness", "claimant"]),
         person & (used == "repair shop"),
         ~person & (id_cat != "")],
        ["professional", "private", "business", "professional"],
        default=np.where(person, "unknown", "business"))
    p["prior_group"] = group
    return p


def assign_category_watchlist(parties, maps):
    cm = maps["category_map"]
    spec = dict(zip(cm.loc[cm["source_field"] == "specialty", "value"].map(fold),
                    cm.loc[cm["source_field"] == "specialty", "category"]))
    lic = dict(zip(cm.loc[cm["source_field"] == "license_type", "value"].map(fold),
                   cm.loc[cm["source_field"] == "license_type", "category"]))
    p = parties
    raw_spec = p["specialty_raw_row"] if "specialty_raw_row" in p else p["specialty"]
    s_cat = raw_spec.map(lambda v: spec.get(v, ""))
    l_cat = p["license_type"].map(lambda v: lic.get(v, ""))
    cat = np.where(s_cat != "", s_cat, l_cat)
    rule = np.where(s_cat != "", "specialty_map:" + raw_spec, np.where(l_cat != "", "license_type_map:" + p["license_type"], ""))
    p["category_inferred"] = cat
    p["category_rule"] = rule
    p["category"] = cat
    p["category_used_rule"] = rule
    p["cat_strength"] = np.where(cat != "", "strong", "")
    p["category_mismatch"] = False
    p["prior_group"] = ""
    unmapped = pd.concat([
        pd.DataFrame({"source_field": "specialty", "value": raw_spec[(raw_spec != "") & (s_cat == "")]}),
        pd.DataFrame({"source_field": "license_type", "value": p["license_type"][(p["license_type"] != "") & (l_cat == "") & (s_cat == "")]}),
    ]).value_counts().rename("parties").reset_index()
    return p, unmapped

## 6 · Outside rarity and party frames

The outside tables become a `Rarity` (Census surnames, SSA first names and initials, NPPES
organization words, with the watchlist's own business-word shares as the larger of the two
where it is larger) and a `Nicknames` index. Parties are split by part into the positional
frames the matching core compares, junk identifier values are blanked, and the comparison
context (value statistics, declared d/b/a pairs) is built.

**Where Splink would do better.** Splink's term-frequency tables come from the data being
linked; ours come from outside, which is deliberate (a fraud file over-represents exactly the
names in question) but means rarity is only as good as the Census/SSA/NPPES coverage.

In [14]:
@dataclass
class Prepared:
    X: dict            # part -> positional party frame (extracted)
    W: dict            # part -> positional party frame (watchlist)
    parties: pd.DataFrame
    details: pd.DataFrame
    ties: pd.DataFrame
    vi: pd.DataFrame
    ctx: object
    reports: dict


def split_parts(parties):
    """part -> positional frame, with tie_pos: the tied party's position in the other part's
    frame of the same side (-1 when none)."""
    fr = {part: parties[parties["part"] == part].reset_index(drop=True) for part in ("person", "business")}
    pos = {part: pd.Series(np.arange(len(f)), index=f["party_id"].to_numpy()) for part, f in fr.items()}
    for part, other in (("person", "business"), ("business", "person")):
        t = fr[part]["tie"]
        fr[part]["tie_pos"] = t.map(pos[other]).fillna(-1).astype(np.int64).to_numpy()
    return fr


def rows_to_parties(df, source, maps, nick, params, watchlist):
    rows = normalize_rows(df, maps, nick, params, source)
    parties, details, ties, empty = breakdown(rows, source, nick)
    if watchlist:
        parties, unmapped = assign_category_watchlist(parties, maps)
    else:
        parties = assign_category_extracted(parties, maps)
        unmapped = None
    return rows, parties, details, ties, empty, unmapped


def prepare(xdf, wdf, maps, ref, cfg, log=print):
    """Inputs (validated frames) -> Prepared."""
    p = cfg.core
    nick = Nicknames(ref["nicknames"])
    xrows, xpar, xdet, xtie, xempty, _ = rows_to_parties(xdf, "X", maps, nick, p, False)
    wrows, wpar, wdet, wtie, wempty, unmapped = rows_to_parties(wdf, "W", maps, nick, p, True)
    parties = pd.concat([xpar, wpar], ignore_index=True)
    details = pd.concat([xdet, wdet], ignore_index=True)
    vi = mark_junk(value_index_fast(parties, details), p)
    blanked_x = blank_junk(xpar, vi)
    blanked_w = blank_junk(wpar, vi)
    counts, total = own_org_word_counts(wpar.loc[wpar["part"] == "business", "org_aliases"])
    rarity = Rarity(ref["surnames"], ref["first_names"], ref["org_tokens"], ref["meta"], p,
                    own_org_counts=counts, own_org_total=total)
    stats = build_value_stats(xpar, wpar)
    dba = declared_dba_pairs(pd.concat([xpar["org_aliases"], wpar["org_aliases"]]))
    ctx = CompareContext(params=p, rarity=rarity, nick=nick, stats=stats, dba_pairs=dba)
    X, W = split_parts(xpar), split_parts(wpar)
    dq = details[~details["valid"]].groupby(["type", "reason"]).size().rename("values").reset_index()
    reports = {"empty_rows_extracted": xempty, "empty_rows_watchlist": wempty, "unmapped": unmapped,
               "invalid_values": dq, "junk_blanked": blanked_x + blanked_w,
               "category_mismatch": int(xpar["category_mismatch"].sum()),
               "rarity_source": rarity.source, "nickname_rows": nick.size, "dba_pairs": len(dba),
               "rows": {"extracted": len(xdf), "watchlist": len(wdf)},
               "parties": {f"{s}_{part}": len(fr) for s, d in (("extracted", X), ("watchlist", W))
                           for part, fr in d.items()}}
    log(f"parties: {reports['parties']}  empty rows: extracted {len(xempty)}, watchlist {len(wempty)}")
    log(f"value index: {len(vi):,} values, {int(vi['single_holder'].sum()):,} single-holder, "
        f"{int(vi['junk'].sum()):,} junk; junk values blanked in {reports['junk_blanked']} party fields")
    log(f"rarity: {rarity.source}; nicknames: {nick.size} rows; declared d/b/a pairs: {len(dba)}")
    return Prepared(X=X, W=W, parties=parties, details=details, ties=pd.concat([xtie, wtie]), vi=vi,
                    ctx=ctx, reports=reports)

## 7 · Candidates

Only extracted × watchlist pairs of the same part type are ever proposed. The union of these
rules, each pair remembering which rules proposed it:

| Part | Rules |
|---|---|
| Person | exact SSN, provider NPI, DL (state:number), licence (state:number), email, VIN, plate; any phone; NYSIIS(last) + first initial; NYSIIS(last) + canonical first initial (Bill meets William); exact DOB + first initial (a surname change); swapped first/last; sorted neighbourhood on "last first" (window 5) |
| Business | exact TIN, clinic NPI, email, work phone; rarest word of each alias (d/b/a included); leading word + state; sorted neighbourhood on the sorted-word name |

**Before indexing**, every rule's pair count is computed from key counts on both sides and
printed. A key that would propose more than `block_pair_cap` pairs is refined with the state;
a refined key still over the cap is dropped and listed in the manifest with its pair count
(an unresolved outcome, never a silent loss). Keys shared by more than the cap in the sorted
neighbourhood are left to the refined blocks. Indexing runs in chunks of `chunk_size`
extracted parties through `recordlinkage.Index` (`Block`, `SortedNeighbourhood` with the
global sorting-key values, so a chunked run proposes exactly the pairs of an unchunked one).

**Where Splink would do better.** Splink's blocking runs as SQL joins in DuckDB, parallel
and out of core, and its `cumulative_comparisons_to_be_scored_from_blocking_rules_chart`
shows each rule's marginal pairs. recordlinkage's `Block` is a pandas merge per chunk.

In [15]:
PERSON_RULES = ["ssn", "npi", "dl", "license", "email", "vin", "plate", "phone",
                "nysiis_initial", "nysiis_canon_initial", "nysiis_first_missing", "dob_initial",
                "swapped", "sn_last_first", "street"]
BUSINESS_RULES = ["tin", "cnpi", "email", "phone", "rare_word", "lead_word_state", "sn_sorted_name",
                  "street"]
IDENTIFIER_RULES = {"ssn", "npi", "dl", "license", "email", "vin", "plate", "phone", "tin", "cnpi"}
SN_RULES = {"sn_last_first", "sn_sorted_name"}


def rule_keys(parties, part, rule, rarity, side="x", dba_pairs=None):
    """Long frame (pos, key, state) of one rule's keys; a party can have several keys."""
    p = parties
    pos = np.arange(len(p))
    st = p["state"].to_numpy(dtype=object)
    if rule in ("ssn", "npi", "dl", "license", "email", "vin", "plate", "tin", "cnpi"):
        k = p[f"id_{rule}"].to_numpy(dtype=object)
        if rule in ("dl", "license", "plate"):                   # number only: states may be unstated
            k = np.array([v.partition(":")[2] if v else "" for v in k], dtype=object)
        return _frame(pos, k, st)
    if rule == "phone":
        if part == "person":
            return pd.concat([_frame(pos, p["phone_own"].to_numpy(dtype=object), st),
                              _frame(pos, p["phone_row"].to_numpy(dtype=object), st)], ignore_index=True)
        return _frame(pos, p["phone_row"].to_numpy(dtype=object), st)
    if rule == "street":
        return _frame(pos, p["addr_street"].to_numpy(dtype=object), st)
    first = p["first"].to_numpy(dtype=object)
    last = p["last"].to_numpy(dtype=object)
    ny = p["last_nysiis"].to_numpy(dtype=object)
    ini = np.array([f[:1] for f in first], dtype=object)
    if rule == "nysiis_initial":
        k = np.where((ny != "") & (ini != ""), ny + "|" + ini, "")
        return _frame(pos, k, st)
    if rule == "nysiis_canon_initial":
        rows_p, rows_k = [], []
        for i, (n, f, roots) in enumerate(zip(ny, first, p["first_roots"].to_numpy(dtype=object))):
            if not n or not f:
                continue
            for letter in sorted({r[:1] for r in roots.split("|") if r}):
                rows_p.append(i); rows_k.append(f"{n}|{letter}")
        return _frame(np.array(rows_p, dtype=np.int64), np.array(rows_k, dtype=object), st[rows_p] if rows_p else np.array([], dtype=object))
    if rule == "nysiis_first_missing":
        # added by the build: a surname with no first name meets every holder of the surname
        k = np.where((ny != "") & ((ini == "") if side == "x" else True), ny, "")
        return _frame(pos, k, st)
    if rule == "street":
        # added by the build: exact number + street (a changed surname at the same address)
        return _frame(pos, p["addr_street"].to_numpy(dtype=object), st)
    if rule == "dob_initial":
        dob = p["dob"].to_numpy(dtype=object)
        k = np.where((dob != "") & (ini != ""), dob + "|" + ini, "")
        return _frame(pos, k, st)
    if rule == "swapped":
        k = np.where((first != "") & (last != ""), (first + "|" + last) if side == "x" else (last + "|" + first), "")
        return _frame(pos, k, st)
    if rule == "sn_last_first":
        k = np.where(last != "", last + " " + first, "")
        return _frame(pos, k, st)
    al = p["org_aliases"].to_numpy(dtype=object)
    if rule == "rare_word":
        # the rarest word of each alias, and of every alias another record declares as its
        # d/b/a partner (so a trade name meets the legal name)
        partners = defaultdict(set)
        for q1, q2 in (dba_pairs or []):
            a1, a2 = " ".join(sorted(q1)), " ".join(sorted(q2))
            partners[a1].add(q2); partners[a2].add(q1)
        rows_p, rows_k = [], []
        for i, s in enumerate(al):
            for a in (s.split("|") if s else []):
                ws = a.split()
                sets = [ws] + [sorted(q) for q in partners.get(" ".join(sorted(ws)), ())]
                for w_ in sets:
                    if w_:
                        rows_p.append(i); rows_k.append(min(w_, key=lambda t: (-rarity.org_idf(t), t)))
        return _frame(np.array(rows_p, dtype=np.int64), np.array(rows_k, dtype=object), st[rows_p] if rows_p else np.array([], dtype=object))
    if rule == "lead_word_state":
        k = np.array([(s.split("|")[0].split()[0] + "|" + t) if s and t else "" for s, t in zip(al, st)], dtype=object)
        return _frame(pos, k, st)
    if rule == "sn_sorted_name":
        k = np.array([" ".join(sorted(s.split("|")[0].split())) if s else "" for s in al], dtype=object)
        return _frame(pos, k, st)
    raise ValueError(rule)


def _frame(pos, key, state):
    f = pd.DataFrame({"pos": np.asarray(pos, dtype=np.int64), "key": np.asarray(key, dtype=object),
                      "state": np.asarray(state, dtype=object)})
    return f[f["key"] != ""].drop_duplicates(["pos", "key"]).reset_index(drop=True)


def plan_rule(xk, wk, rule, cap):
    """Effective keys after refining oversized keys by state. Returns (xk, wk, report rows)."""
    cx = xk["key"].value_counts()
    cw = wk["key"].value_counts()
    both = cx.index.intersection(cw.index)
    prod = cx[both] * cw[both]
    total = int(prod.sum())
    big = set(prod.index[prod > cap]) if rule not in SN_RULES else set()
    rep = {"rule": rule, "pairs_before_refine": total, "oversized_keys": len(big),
           "dropped_keys": 0, "dropped_pairs": 0, "pairs_after_refine": total, "dropped_examples": ""}
    if big:
        def refine(f):
            f = f.copy()
            m = f["key"].isin(big)
            f.loc[m, "key"] = np.where(f.loc[m, "state"] != "", f.loc[m, "key"] + "|" + f.loc[m, "state"], "")
            return f[f["key"] != ""]
        xk, wk = refine(xk), refine(wk)
        cx2, cw2 = xk["key"].value_counts(), wk["key"].value_counts()
        b2 = cx2.index.intersection(cw2.index)
        prod2 = cx2[b2] * cw2[b2]
        drop = prod2[prod2 > cap]
        if len(drop):
            xk = xk[~xk["key"].isin(drop.index)]
            wk = wk[~wk["key"].isin(drop.index)]
        rep.update(dropped_keys=int(len(drop)), dropped_pairs=int(drop.sum()),
                   pairs_after_refine=int(prod2.sum() - drop.sum()),
                   dropped_examples="; ".join(f"{k} ({v:,})" for k, v in drop.sort_values(ascending=False).head(5).items()))
    elif rule in SN_RULES:
        # exact-duplicate keys over the cap stay with the refined blocks
        over = set(prod.index[prod > cap])
        if over:
            xk = xk[~xk["key"].isin(over)]
            wk = wk[~wk["key"].isin(over)]
            rep.update(dropped_keys=len(over), dropped_pairs=int(prod[list(over)].sum()),
                       dropped_examples="left to the name blocks: " + "; ".join(sorted(over)[:5]))
    return xk.reset_index(drop=True), wk.reset_index(drop=True), rep


class CandidatePlan:
    """Keys of every rule for one part, prepared once for all chunks."""

    def __init__(self, xp, wp, part, rarity, cfg, dba_pairs=None):
        self.part, self.cfg = part, cfg
        self.rules = PERSON_RULES if part == "person" else BUSINESS_RULES
        self.bit = {r: 1 << i for i, r in enumerate(self.rules)}
        self.keys, self.report = {}, []
        for r in self.rules:
            xk, wk, rep = plan_rule(rule_keys(xp, part, r, rarity, dba_pairs=dba_pairs),
                                    rule_keys(wp, part, r, rarity, side="w", dba_pairs=dba_pairs), r,
                                    cfg.block_pair_cap)
            rep["part"] = part
            self.report.append(rep)
            sk = None
            if r in SN_RULES:
                sk = np.sort(pd.unique(np.concatenate([xk["key"].to_numpy(dtype=object),
                                                       wk["key"].to_numpy(dtype=object)])).astype(str))
            wkf = pd.DataFrame({"key": wk["key"].to_numpy(dtype=object)})
            self.keys[r] = (xk, wkf, wk["pos"].to_numpy(), sk)
        self.n_x = len(xp)

    def chunk(self, lo, hi):
        """Candidate pairs for extracted parties with positions in [lo, hi): (l, r, rules)."""
        ls, rs, bs = [], [], []
        for r in self.rules:
            xk, wkf, wpos, sk = self.keys[r]
            xc = xk[(xk["pos"] >= lo) & (xk["pos"] < hi)]
            if not len(xc) or not len(wkf):
                continue
            xf = pd.DataFrame({"key": xc["key"].to_numpy(dtype=object)})
            if r in SN_RULES:
                idx = rl.Index()
                idx.add(rl.index.SortedNeighbourhood("key", "key", window=self.cfg.sn_window,
                                                     sorting_key_values=sk))
            else:
                idx = rl.Index()
                idx.add(rl.index.Block("key", "key"))
            mi = idx.index(xf, wkf)
            if len(mi) == 0:
                continue
            li = xc["pos"].to_numpy()[mi.get_level_values(0).to_numpy()]
            ri = wpos[mi.get_level_values(1).to_numpy()]
            ls.append(li); rs.append(ri); bs.append(np.full(len(li), self.bit[r], dtype=np.int64))
        if not ls:
            return pd.DataFrame({"l": np.array([], np.int64), "r": np.array([], np.int64),
                                 "rules": np.array([], np.int64)})
        l = np.concatenate(ls); r_ = np.concatenate(rs); b = np.concatenate(bs)
        key = l * np.int64(1 << 32) + r_
        order = np.argsort(key, kind="stable")
        key, b = key[order], b[order]
        starts = np.flatnonzero(np.r_[True, key[1:] != key[:-1]])
        bits = np.bitwise_or.reduceat(b, starts)
        uk = key[starts]
        return pd.DataFrame({"l": uk >> 32, "r": uk & np.int64((1 << 32) - 1), "rules": bits})

    def rule_names(self, bits):
        return [",".join(r for r in self.rules if b & self.bit[r]) for b in bits]

## Noise engine

Noisy copies of input rows, driven by `mappings/simulation_noise.csv` (a pessimistic table,
for review). The same engine makes the simulated true-match pairs for m (step 10), the
synthetic extracted rows and the LEIE test set. Each copy records which noises hit it.

**Where Splink would do better.** Splink has no simulator; its answer to missing labels is
EM plus the user's own labels. Simulation here is a last-resort source of m and is flagged
as such wherever it is used.

In [16]:
_TYPO_ALPHA = "ABCDEFGHIJKLMNOPRSTUVWY"
SUFFIX_WORDS = {"INC", "LLC", "PC", "PLLC", "CORP", "CO", "LTD", "LLP", "PA", "INCORPORATED",
                "CORPORATION", "COMPANY", "P.C.", "L.L.C.", "INC."}
ABBREV_OF = {v: k for k, v in ORG_ABBREV.items() if len(k) >= 3}

# A small fictional-use geography: real city / state / ZIP-prefix triples, fictional streets.
GEOGRAPHY = [
    ("BROOKLYN", "NY", "112"), ("QUEENS", "NY", "113"), ("BRONX", "NY", "104"), ("ALBANY", "NY", "122"),
    ("NEWARK", "NJ", "071"), ("PATERSON", "NJ", "075"), ("PHILADELPHIA", "PA", "191"), ("PITTSBURGH", "PA", "152"),
    ("BOSTON", "MA", "021"), ("WORCESTER", "MA", "016"), ("HARTFORD", "CT", "061"), ("PROVIDENCE", "RI", "029"),
    ("BALTIMORE", "MD", "212"), ("RICHMOND", "VA", "232"), ("CHARLOTTE", "NC", "282"), ("ATLANTA", "GA", "303"),
    ("MIAMI", "FL", "331"), ("ORLANDO", "FL", "328"), ("TAMPA", "FL", "336"), ("HIALEAH", "FL", "330"),
    ("DETROIT", "MI", "482"), ("CLEVELAND", "OH", "441"), ("COLUMBUS", "OH", "432"), ("CHICAGO", "IL", "606"),
    ("MILWAUKEE", "WI", "532"), ("MINNEAPOLIS", "MN", "554"), ("SAINT LOUIS", "MO", "631"), ("MEMPHIS", "TN", "381"),
    ("NEW ORLEANS", "LA", "701"), ("HOUSTON", "TX", "770"), ("DALLAS", "TX", "752"), ("SAN ANTONIO", "TX", "782"),
    ("PHOENIX", "AZ", "850"), ("DENVER", "CO", "802"), ("LAS VEGAS", "NV", "891"), ("LOS ANGELES", "CA", "900"),
    ("SAN DIEGO", "CA", "921"), ("FRESNO", "CA", "937"), ("SEATTLE", "WA", "981"), ("PORTLAND", "OR", "972"),
]
STREET_NAMES = ["MAPLE", "OAK", "CEDAR", "PINE", "ELM", "WILLOW", "BIRCH", "SPRUCE", "CHESTNUT", "WALNUT",
                "HICKORY", "MAGNOLIA", "DOGWOOD", "SYCAMORE", "LAUREL", "JUNIPER", "HAWTHORN", "ASPEN",
                "LINDEN", "POPLAR", "CYPRESS", "HOLLY", "IVY", "ORCHARD", "MEADOW", "PRAIRIE", "RIDGE",
                "SUMMIT", "VALLEY", "HILLSIDE", "LAKEVIEW", "RIVERSIDE", "BAYVIEW", "HARBOR", "SUNSET",
                "SUNRISE", "PARK", "GARDEN", "CHURCH", "SCHOOL", "MILL", "BRIDGE", "MARKET", "UNION",
                "LIBERTY", "FRANKLIN", "WASHINGTON", "JEFFERSON", "MADISON", "MONROE", "JACKSON", "LINCOLN",
                "GRANT", "HAMILTON", "ADAMS", "CLINTON", "KENNEDY", "HIGHLAND", "FOREST", "GREEN", "SPRING",
                "CENTER", "MAIN", "BROAD", "HIGH", "WATER", "FRONT", "CANAL", "STATION", "RAILROAD", "COLONIAL",
                "EUCLID", "KINGS", "QUEENS", "WINDSOR", "OXFORD", "CAMBRIDGE", "STRATFORD", "ASHFORD",
                "BRADFORD", "CLIFTON", "FAIRVIEW", "GLENWOOD", "KENWOOD", "LAKEWOOD", "MAPLEWOOD", "OAKWOOD",
                "ROSEWOOD", "WOODLAND", "WESTGATE", "EASTGATE", "NORTHGATE", "SOUTHGATE", "CROSSWIND",
                "WINDMILL", "BLUEBIRD", "CARDINAL", "ORIOLE", "HERON", "FALCON", "EAGLE", "HAWK", "SPARROW",
                "1ST", "2ND", "3RD", "4TH", "5TH", "6TH", "7TH", "8TH", "9TH", "10TH", "12TH", "14TH", "21ST",
                "34TH", "45TH", "79TH", "110TH"]
STREET_SUFFIXES = ["ST", "AVE", "RD", "DR", "LN", "CT", "BLVD", "PL", "WAY", "TER"]
SPECIALTIES = ["CHIROPRACTIC", "PHYSICAL THERAPY", "ACUPUNCTURE", "ORTHOPEDICS", "GENERAL PRACTICE",
               "PAIN MANAGEMENT", "RADIOLOGY", "NEUROLOGY", "PSYCHOLOGY", "PODIATRY", "INTERNAL MEDICINE",
               "OSTEOPATHY", "DENTISTRY", "PHARMACY", "NURSING"]


def luhn_npi(prefix9):
    """A valid NPI from nine digits."""
    total = 0
    for i, ch in enumerate(reversed("80840" + prefix9)):
        v = int(ch)
        if i % 2 == 0:
            v *= 2
            if v > 9:
                v -= 9
        total += v
    return prefix9 + str((10 - total % 10) % 10)


def vin_with_check(body16, rng):
    """A VIN with a correct check digit from 16 characters (position 9 is replaced)."""
    v = list(body16[:8] + "0" + body16[8:16])
    r = sum(_VIN_MAP[c] * w for c, w in zip(v, _VIN_W)) % 11
    v[8] = "X" if r == 10 else str(r)
    return "".join(v)


class Fake:
    """Fictional values with realistic frequencies (names drawn from the Census and SSA
    tables), deterministic under one numpy Generator."""

    def __init__(self, ref, rng, n_surnames=30000, n_firsts=3000):
        self.rng = rng
        sn = sorted((ref.get("surnames") or {"SMITH": 1}).items(), key=lambda kv: -kv[1])[:n_surnames]
        fn = sorted((ref.get("first_names") or {"JOHN": 1}).items(), key=lambda kv: -kv[1])[:n_firsts]
        self.surnames = np.array([k for k, _ in sn], dtype=object)
        self.sw = np.array([v for _, v in sn], dtype=float); self.sw /= self.sw.sum()
        self.firsts = np.array([k for k, _ in fn], dtype=object)
        self.fw = np.array([v for _, v in fn], dtype=float); self.fw /= self.fw.sum()
        nick = ref.get("nicknames")
        self.nick_of = defaultdict(list)
        if nick is not None:
            for a, b in zip(nick["name1"].map(fold), nick["name2"].map(fold)):
                self.nick_of[a].append(b)

    def _draw(self, values, cum, n):
        """Weighted draws by inverse CDF (rng.choice with p re-sums the weights on every call)."""
        k = 1 if n is None else n
        idx = np.searchsorted(cum, self.rng.random(k) * cum[-1], side="right")
        out = values[np.minimum(idx, len(values) - 1)]
        return out[0] if n is None else out

    def surname(self, n=None):
        if not hasattr(self, "_scum"):
            self._scum = np.cumsum(self.sw)
        return self._draw(self.surnames, self._scum, n)

    def first(self, n=None):
        if not hasattr(self, "_fcum"):
            self._fcum = np.cumsum(self.fw)
        return self._draw(self.firsts, self._fcum, n)

    def digits(self, k):
        return "".join(str(x) for x in self.rng.integers(0, 10, k))

    def ssn(self):
        while True:
            s = f"{self.rng.integers(1, 899):03d}{self.rng.integers(1, 100):02d}{self.rng.integers(1, 10000):04d}"
            if norm_ssn(s)[1]:
                return s

    def tin(self):
        while True:
            s = f"{self.rng.integers(10, 99):02d}{self.rng.integers(0, 10 ** 7):07d}"
            if norm_tin(s)[1]:
                return s

    def npi(self):
        return luhn_npi(str(self.rng.integers(1, 3)) + self.digits(8))

    def phone(self):
        while True:
            s = f"{self.rng.integers(201, 990)}{self.rng.integers(200, 999)}{self.rng.integers(0, 10000):04d}"
            if norm_phone(s)[1]:
                return s

    def dob(self, lo=1940, hi=2002):
        y = int(self.rng.integers(lo, hi + 1)); m = int(self.rng.integers(1, 13)); d = int(self.rng.integers(1, 29))
        return f"{y:04d}-{m:02d}-{d:02d}"

    def address(self, state=None):
        g = [x for x in GEOGRAPHY if x[1] == state] if state else GEOGRAPHY
        if not g:
            g = GEOGRAPHY
        city, st, z3 = g[int(self.rng.integers(0, len(g)))]
        return {"street_number": str(int(self.rng.integers(1, 9999))), "street_direction": "",
                "street_name": str(self.rng.choice(STREET_NAMES)), "street_type": str(self.rng.choice(STREET_SUFFIXES)),
                "unit": (str(int(self.rng.integers(1, 40))) if self.rng.random() < 0.3 else ""),
                "city": city, "state": st, "zip": z3 + f"{int(self.rng.integers(0, 100)):02d}"}

    def email(self, first, last):
        dom = str(self.rng.choice(["mailbox.test", "post.test", "inbox.test", "webmail.test"]))
        return f"{first.lower()}.{last.lower()}{int(self.rng.integers(1, 999))}@{dom}".replace(" ", "")

    def dl(self):
        return str(self.rng.choice(list("ABCDEFGHJKLMNPRSTWY"))) + self.digits(7)

    def vin(self):
        chars = "ABCDEFGHJKLMNPRSTUVWXYZ0123456789"
        body = "".join(self.rng.choice(list(chars), 16))
        return vin_with_check(body, self.rng)

    def plate(self):
        return "".join(self.rng.choice(list("ABCDEFGHJKLMNPRSTUVWXYZ"), 3)) + self.digits(4)


def _typo(s, rng):
    s = str(s)
    if len(s) < 3:
        return s
    i = int(rng.integers(1, len(s) - 1))
    op = int(rng.integers(0, 4))
    c = _TYPO_ALPHA[int(rng.integers(0, len(_TYPO_ALPHA)))]
    if op == 0:
        return s[:i] + c + s[i + 1:]
    if op == 1:
        return s[:i] + s[i + 1:]
    if op == 2:
        return s[:i] + c + s[i:]
    return s[:i - 1] + s[i] + s[i - 1] + s[i + 1:]


def _digit_typo(s, rng):
    d = [i for i, ch in enumerate(s) if ch.isdigit()]
    if not d:
        return s
    i = d[int(rng.integers(0, len(d)))]
    new = str((int(s[i]) + int(rng.integers(1, 10))) % 10)
    return s[:i] + new + s[i + 1:]


def _transpose(s, rng):
    d = [i for i in range(len(s) - 1) if s[i].isdigit() and s[i + 1].isdigit() and s[i] != s[i + 1]]
    if not d:
        return s
    i = d[int(rng.integers(0, len(d)))]
    return s[:i] + s[i + 1] + s[i] + s[i + 2:]


def apply_noise(rows, noise, fake, rng, scale=1.0, id_prefix="sim"):
    """Noisy copies of input-schema rows. `noise`: the simulation_noise table; `scale`
    multiplies every rate. Returns (copies, log) where log lists the noises per copy."""
    rate = {n: float(r) * scale for n, r in zip(noise["noise"], noise["rate"])}
    hit = lambda n: rng.random() < rate.get(n, 0.0)
    cats = ["medical", "legal", "repair shop", "witness", "claimant", "other"]
    out, log = [], []
    for i, r in enumerate(rows.to_dict("records")):
        r = dict(r)
        applied = []

        def did(n):
            applied.append(n)
        has_person = bool(r.get("first_name") or r.get("last_name"))
        if has_person:
            last, first = r.get("last_name", ""), r.get("first_name", "")
            if last and hit("surname_typo"):
                last = _typo(last, rng); did("surname_typo")
            if last and re.search(r"[ \-]", last) and hit("surname_compound_dropped"):
                last = re.split(r"[ \-]", last)[0]; did("surname_compound_dropped")
            if last and hit("surname_change"):
                last = str(fake.surname()).title(); did("surname_change")
            f = fold(first)
            if f and hit("first_nickname") and fake.nick_of.get(f):
                opts = fake.nick_of[f]
                first = opts[int(rng.integers(0, len(opts)))].title(); did("first_nickname")
            elif f and hit("first_initial_only"):
                first = f[0]; did("first_initial_only")
            elif f and hit("first_typo"):
                first = _typo(first, rng); did("first_typo")
            elif f and hit("first_missing"):
                first = ""; did("first_missing")
            if first and last and hit("names_swapped"):
                first, last = last, first; did("names_swapped")
            r["first_name"], r["last_name"] = first, last
            mid = r.get("middle_name", "")
            if mid:
                if hit("middle_dropped"):
                    mid = ""; did("middle_dropped")
                elif hit("middle_initial"):
                    mid = mid[0]; did("middle_initial")
            r["middle_name"] = mid
            dob = r.get("dob", "")
            if dob:
                y, m, d = dob[:4], dob[5:7], dob[8:10]
                if hit("dob_missing"):
                    dob = ""; did("dob_missing")
                elif hit("dob_placeholder"):
                    dob = "1900-01-01"; did("dob_placeholder")
                elif int(d) <= 12 and d != m and hit("dob_day_month_swap"):
                    dob = f"{y}-{d}-{m}"; did("dob_day_month_swap")
                elif hit("dob_digit_typo"):
                    for _ in range(10):
                        cand = _digit_typo(dob.replace("-", ""), rng)
                        iso, ok, _r = norm_dob(cand)
                        if ok:
                            dob = iso; break
                    did("dob_digit_typo")
                elif hit("dob_year_off"):
                    dob = f"{int(y) + (1 if rng.random() < 0.5 else -1):04d}-{m}-{d}"; did("dob_year_off")
                r["dob"] = dob
            if r.get("provider_specialty") and hit("specialty_coarser"):
                r["provider_specialty"] = "PHYSICIAN"; did("specialty_coarser")
        for col in ("ssn", "provider_npi", "driver_license_number", "tin"):
            v = r.get(col, "")
            if v:
                if hit("id_digit_typo"):
                    r[col] = _digit_typo(v, rng); did(f"id_digit_typo:{col}")
                elif hit("id_transposition"):
                    r[col] = _transpose(v, rng); did(f"id_transposition:{col}")
        if r.get("street_name") or r.get("zip") or r.get("state"):
            if hit("moved_within_state"):
                r.update(fake.address(r.get("state") or None)); did("moved_within_state")
            elif hit("moved_other_state"):
                others = [g[1] for g in GEOGRAPHY if g[1] != r.get("state")]
                r.update(fake.address(others[int(rng.integers(0, len(others)))])); did("moved_other_state")
            if r.get("unit") and hit("unit_dropped"):
                r["unit"] = ""; did("unit_dropped")
            if r.get("zip") and hit("zip_typo"):
                r["zip"] = _digit_typo(r["zip"], rng); did("zip_typo")
        for col in ("home_phone", "work_phone"):
            if r.get(col) and hit("phone_changed"):
                r[col] = fake.phone(); did(f"phone_changed:{col}")
        if r.get("email") and hit("email_changed"):
            r["email"] = fake.email(fold(r.get("first_name") or "info") or "info",
                                    fold(r.get("last_name") or "office") or "office"); did("email_changed")
        b = r.get("business_name", "")
        if b:
            if _DBA_RE.search(b) and hit("business_dba_used"):
                b = _DBA_RE.split(b)[-1].strip(" ,"); did("business_dba_used")
            words = b.split()
            if len(words) > 1 and words[-1].upper().strip(",.") in SUFFIX_WORDS and hit("business_suffix_dropped"):
                b = " ".join(words[:-1]).strip(" ,"); did("business_suffix_dropped")
            words = b.split()
            ab = [j for j, w in enumerate(words) if w.upper() in ABBREV_OF]
            if ab and hit("business_abbreviated"):
                j = ab[int(rng.integers(0, len(ab)))]
                words[j] = ABBREV_OF[words[j].upper()].title(); b = " ".join(words); did("business_abbreviated")
            if hit("business_typo"):
                words = b.split()
                j = int(rng.integers(0, len(words)))
                if len(words[j]) >= 5:
                    words[j] = _typo(words[j], rng); b = " ".join(words); did("business_typo")
            r["business_name"] = b
        if r.get("category") and hit("category_disagrees"):
            r["category"] = str(rng.choice([c for c in cats if c != r["category"]])); did("category_disagrees")
        r["record_id"] = f"{id_prefix}:{r['record_id']}"
        out.append(r)
        log.append(";".join(applied))
    return pd.DataFrame(out, columns=list(rows.columns)), log

## Synthetic data

A fictional test set with a truth file, deterministic under the run seed.

* **Universe**: persons and businesses with canonical details; names drawn with Census and
  SSA frequencies (so "Smith" is as common as it is), every identifier fictional and valid
  (NPIs pass Luhn, VINs their check digit).
* **Watchlist rows**: 60% of the universe, lightly noised, ~10% listed twice (watchlist
  duplicates), plus watchlist-only parties up to the requested size.
* **Extracted rows**: 70% of the universe (so 40% of them are not listed), null-heavy, noised
  with the simulation table at half its rates, many parties in several claims (unmerged
  duplicates), grouped into claims and notes.
* **Edge cases**: ~45 hand-written cases with the expected best watchlist row, basis, veto
  and levels, listed in the table `EDGE_CASES`.

Truth is written beside the inputs and read only by diagnostics and self-tests, never by the
pipeline (AGENTS rule 13).

**Where Splink would do better.** Splink ships labelled demo datasets (febrl, historical
persons) with known truth; they are real benchmarks. This set is ours and shaped by our own
noise table, so results on it show that the code does what it says, not how well it would do
on client data.

In [17]:
def blank_row(rid, **kw):
    r = {c: "" for c in SCHEMA}
    r["record_id"] = rid
    r.update({k: ("" if v is None else str(v)) for k, v in kw.items()})
    return r


def _npi(n):
    return luhn_npi(f"19{n:07d}")


def _ssn(n):
    return f"{200 + n % 600:03d}{10 + n % 80:02d}{1000 + n:04d}"


def _tin(n):
    return f"45{n:07d}"


def _addr(num, street, stype, city, state, zp, unit=""):
    return {"street_number": num, "street_name": street, "street_type": stype, "unit": unit,
            "city": city, "state": state, "zip": zp}


A_NY = _addr("118", "ORCHARD", "ST", "BROOKLYN", "NY", "11211")
A_NY2 = _addr("42", "HERON", "AVE", "QUEENS", "NY", "11354")
A_CA = _addr("901", "SUNSET", "BLVD", "LOS ANGELES", "CA", "90026")
A_IL = _addr("77", "CANAL", "ST", "CHICAGO", "IL", "60606", "4")
A_FL = _addr("3100", "BAYVIEW", "DR", "MIAMI", "FL", "33133")


def edge_cases():
    """Hand-written cases: (case, description, extracted rows, watchlist rows, expectations).
    An expectation names the extracted party (record_id, part), the watchlist record expected
    to be its best candidate (rank 1) or merely visible, and the basis / veto / levels."""
    C = []

    def case(cid, desc, x, w, expect):
        C.append({"case": cid, "desc": desc, "x": x, "w": w, "expect": expect})

    E = lambda **k: k
    case("E01", "person + business on one row, both identifier-linked",
         [blank_row("E01X", claim_id="CE01", note_id="NE01", first_name="Harold", last_name="Vantongeren",
                    provider_npi=_npi(101), business_name="Vantongeren Chiropractic PC", tin=_tin(101), **A_NY)],
         [blank_row("E01W1", first_name="Harold", last_name="Vantongeren", provider_npi=_npi(101), dob="1961-03-14", **A_NY),
          blank_row("E01W2", business_name="Vantongeren Chiropractic P.C.", tin=_tin(101), **A_NY)],
         [E(x="E01X", part="person", w="E01W1", basis="identifier", rank=1),
          E(x="E01X", part="business", w="E01W2", basis="identifier", rank=1)])
    case("E02", "a TIN shared by two businesses in the extracted file",
         [blank_row("E02X1", claim_id="CE02", note_id="NE02", business_name="Silverline Medical Supply", tin=_tin(102)),
          blank_row("E02X2", claim_id="CE02", note_id="NE02b", business_name="Quillfeather Diagnostics", tin=_tin(102))],
         [blank_row("E02W", business_name="Silverline Medical Supply Inc", tin=_tin(102), **A_FL)],
         [E(x="E02X1", part="business", w="E02W", basis="identifier", rank=1),
          E(x="E02X2", part="business", w="E02W", basis="identifier")])
    case("E03", "sibling companies: a shared head word, distinctive words on both sides",
         [blank_row("E03X", claim_id="CE03", note_id="NE03", business_name="Allport Indemnity Company", **A_NY2)],
         [blank_row("E03W1", business_name="Allport Indemnity Co"),
          blank_row("E03W2", business_name="Allport Casualty Surety Company", **A_NY)],
         [E(x="E03X", part="business", w="E03W1", rank=1, levels={"org": "exact"}),
          E(x="E03X", part="business", w="E03W2", levels={"org": "sibling"}, if_visible=True)])
    s4 = _ssn(104)
    case("E04", "one SSN under three names in the extracted file",
         [blank_row("E04X1", claim_id="CE04", note_id="NE04", first_name="Rosalind", last_name="Achterberg", ssn=s4),
          blank_row("E04X2", claim_id="CE04", note_id="NE04", first_name="Tobias", last_name="Wenceslas", ssn=s4),
          blank_row("E04X3", claim_id="CE04b", note_id="NE04b", first_name="Imelda", last_name="Fairweather", ssn=s4)],
         [blank_row("E04W", first_name="Rosalind", last_name="Achterberg", ssn=s4, dob="1970-02-02")],
         [E(x="E04X1", part="person", w="E04W", basis="identifier", rank=1),
          E(x="E04X2", part="person", w="E04W", basis="identifier")])
    case("E05", "junk SSN on both sides is never an identifier",
         [blank_row("E05X", claim_id="CE05", note_id="NE05", first_name="Gertrude", last_name="Oyelaran", ssn="123-45-6789", dob="1958-07-21")],
         [blank_row("E05W", first_name="Gertrude", last_name="Oyelaran", ssn="123456789", dob="1958-07-21")],
         [E(x="E05X", part="person", w="E05W", basis="dob", rank=1, levels={"ssn": "empty"})])
    case("E06", "two different single-holder SSNs veto",
         [blank_row("E06X", claim_id="CE06", note_id="NE06", first_name="Mortimer", last_name="Quackenbush", ssn=_ssn(106), dob="1966-11-02")],
         [blank_row("E06W", first_name="Mortimer", last_name="Quackenbush", ssn=_ssn(206), dob="1966-11-02")],
         [E(x="E06X", part="person", w="E06W", veto="ssn")])
    case("E07", "Bill meets William",
         [blank_row("E07X", claim_id="CE07", note_id="NE07", first_name="Bill", last_name="Szczepanski", dob="1972-05-30")],
         [blank_row("E07W", first_name="William", last_name="Szczepanski", dob="1972-05-30")],
         [E(x="E07X", part="person", w="E07W", basis="dob", rank=1, levels={"name": "first_nick_or_close"})])
    case("E08", "first-name typo, same address",
         [blank_row("E08X", claim_id="CE08", note_id="NE08", first_name="Jonathon", last_name="Kowalczyk", **A_NY2)],
         [blank_row("E08W", first_name="Jonathan", last_name="Kowalczyk", **A_NY2)],
         [E(x="E08X", part="person", w="E08W", basis="address", rank=1, levels={"address": "exact"})])
    case("E09", "first and last swapped",
         [blank_row("E09X", claim_id="CE09", note_id="NE09", first_name="Nwachukwu", last_name="Chidiebere", dob="1980-01-19")],
         [blank_row("E09W", first_name="Chidiebere", last_name="Nwachukwu", dob="1980-01-19")],
         [E(x="E09X", part="person", w="E09W", basis="dob", rank=1, levels={"name": "swapped"})])
    case("E10", "initial only, owned phone agrees",
         [blank_row("E10X", claim_id="CE10", note_id="NE10", first_name="R.", last_name="Featherstonehaugh", home_phone="(718) 392-4410")],
         [blank_row("E10W", first_name="Rupert", last_name="Featherstonehaugh", home_phone="7183924410")],
         [E(x="E10X", part="person", w="E10W", basis="identifier", rank=1, levels={"name": "initial_agrees", "phone": "exact_owned_single"})])
    case("E11", "a claim row with only a TIN",
         [blank_row("E11X", claim_id="CE11", note_id="NE11", tin=_tin(111))],
         [blank_row("E11W", business_name="Marchetti Towing LLC", tin=_tin(111), **A_NY)],
         [E(x="E11X", part="business", w="E11W", basis="identifier", rank=1)])
    case("E12", "a rare exact name and nothing else: visible as name_only",
         [blank_row("E12X", claim_id="CE12", note_id="NE12", first_name="Evangelina", last_name="Throckmorton")],
         [blank_row("E12W", first_name="Evangelina", last_name="Throckmorton", dob="1949-09-09", **A_CA)],
         [E(x="E12X", part="person", w="E12W", basis="name_only", rank=1)])
    case("E13", "John Smith and nothing else: every John Smith visible, name_only",
         [blank_row("E13X", claim_id="CE13", note_id="NE13", first_name="John", last_name="Smith")],
         [blank_row(f"E13W{i}", first_name="John", last_name="Smith", dob=f"19{50 + i}-01-0{i}") for i in (1, 2, 3)],
         [E(x="E13X", part="person", w=f"E13W{i}", basis="name_only") for i in (1, 2, 3)])
    case("E14", "day and month swapped",
         [blank_row("E14X", claim_id="CE14", note_id="NE14", first_name="Philippa", last_name="Gundersen", dob="1975-04-09")],
         [blank_row("E14W", first_name="Philippa", last_name="Gundersen", dob="1975-09-04")],
         [E(x="E14X", part="person", w="E14W", basis="dob", rank=1, levels={"dob": "swap_or_typo"})])
    case("E15", "moved to another state, SSN agrees",
         [blank_row("E15X", claim_id="CE15", note_id="NE15", first_name="Cornelius", last_name="Abernethy", ssn=_ssn(115), **A_NY)],
         [blank_row("E15W", first_name="Cornelius", last_name="Abernethy", ssn=_ssn(115), **A_CA)],
         [E(x="E15X", part="person", w="E15W", basis="identifier", rank=1, levels={"address": "differs"})])
    case("E16", "given category conflicts with an NPI",
         [blank_row("E16X", claim_id="CE16", note_id="NE16", category="legal", first_name="Winifred", last_name="Castellano", provider_npi=_npi(116))],
         [blank_row("E16W", first_name="Winifred", last_name="Castellano", provider_npi=_npi(116))],
         [E(x="E16X", part="person", w="E16W", basis="identifier", rank=1, party={"category_mismatch": True, "category": "medical"})])
    case("E17", "keyword business: repair shop",
         [blank_row("E17X", claim_id="CE17", note_id="NE17", business_name="Ramirez Auto Body & Collision")],
         [blank_row("E17W", business_name="Ramirez Auto Body and Collision Inc")],
         [E(x="E17X", part="business", w="E17W", rank=1, levels={"org": "exact"},
            party={"category_inferred": "repair shop"})])
    v18 = vin_with_check("1FTFW1E5JFA12345", None)
    case("E18", "VIN agrees",
         [blank_row("E18X", claim_id="CE18", note_id="NE18", first_name="Desmond", last_name="Okonkwo-Lind", vin=v18)],
         [blank_row("E18W", first_name="Desmond", last_name="Okonkwo-Lind", vin=v18)],
         [E(x="E18X", part="person", w="E18W", basis="identifier", rank=1)])
    case("E19", "a clinic phone shared by two doctors is not an identifier",
         [blank_row("E19X1", claim_id="CE19", note_id="NE19", first_name="Anneliese", last_name="Borowczyk", work_phone="3124640199", business_name="Northgate Spine Center"),
          blank_row("E19X2", claim_id="CE19", note_id="NE19", first_name="Tiberius", last_name="Maclean", work_phone="312-464-0199", business_name="Northgate Spine Center")],
         [blank_row("E19W", first_name="Anneliese", last_name="Borowczyk", work_phone="3124640199")],
         [E(x="E19X2", part="person", w="E19W", basis_not="identifier", levels={"phone": "exact_shared"})])
    case("E20", "co-party: the tied businesses share a TIN",
         [blank_row("E20X", claim_id="CE20", note_id="NE20", first_name="Leopold", last_name="Anyanwu", business_name="Anyanwu Family Medicine PC", tin=_tin(120))],
         [blank_row("E20W", first_name="Leopold", last_name="Anyanwu", business_name="Anyanwu Family Medicine PC", tin=_tin(120))],
         [E(x="E20X", part="business", w="E20W", basis="identifier", rank=1),
          E(x="E20X", part="person", w="E20W", basis="co_party", rank=1, levels={"co_party": "anchored"})])
    case("E21", "co-party needs an anchor: name-only businesses vouch for nothing",
         [blank_row("E21X", claim_id="CE21", note_id="NE21", first_name="Octavia", last_name="Pellegrini", business_name="Pellegrini Wellness")],
         [blank_row("E21W", first_name="Octavia", last_name="Pellegrini", business_name="Pellegrini Wellness")],
         [E(x="E21X", part="person", w="E21W", basis="name_only", rank=1)])
    case("E22", "watchlist duplicates: the same person listed twice",
         [blank_row("E22X", claim_id="CE22", note_id="NE22", first_name="Ingrid", last_name="Solberg-Haas", provider_npi=_npi(122))],
         [blank_row("E22W1", first_name="Ingrid", last_name="Solberg-Haas", provider_npi=_npi(122), **A_IL),
          blank_row("E22W2", first_name="Ingrid", last_name="Solberg Haas", provider_npi=_npi(122), **A_FL)],
         [E(x="E22X", part="person", w="E22W1", basis="identifier", true=True),
          E(x="E22X", part="person", w="E22W2", basis="identifier", true=True)])
    case("E23", "an invalid NPI is never an identifier",
         [blank_row("E23X", claim_id="CE23", note_id="NE23", first_name="Fenwick", last_name="Oduya", provider_npi="1234567890")],
         [blank_row("E23W", first_name="Fenwick", last_name="Oduya", provider_npi="1234567890")],
         [E(x="E23X", part="person", w="E23W", basis="name_only", rank=1, levels={"npi": "empty"})])
    case("E24", "an empty row yields no party and is reported",
         [blank_row("E24X", claim_id="CE24", note_id="NE24", city="BROOKLYN")], [],
         [E(x="E24X", part="none", empty_row=True)])
    case("E25", "one part of a compound surname dropped",
         [blank_row("E25X", claim_id="CE25", note_id="NE25", first_name="Maria", last_name="Garcia-Villanueva", dob="1983-06-17")],
         [blank_row("E25W", first_name="Maria", last_name="Garcia", dob="1983-06-17")],
         [E(x="E25X", part="person", w="E25W", basis="dob", rank=1, levels={"name": "last_close_first_agrees"})])
    case("E26", "surname changed, SSN and DOB agree",
         [blank_row("E26X", claim_id="CE26", note_id="NE26", first_name="Beatrix", last_name="Holloway", ssn=_ssn(126), dob="1979-12-01")],
         [blank_row("E26W", first_name="Beatrix", last_name="Lindqvist", ssn=_ssn(126), dob="1979-12-01")],
         [E(x="E26X", part="person", w="E26W", basis="identifier", rank=1, levels={"name": "else"})])
    case("E27", "two driver licences of one state veto",
         [blank_row("E27X", claim_id="CE27", note_id="NE27", first_name="Casimir", last_name="Vandermeer", driver_license_number="D2700001", driver_license_state="NY")],
         [blank_row("E27W", first_name="Casimir", last_name="Vandermeer", driver_license_number="K5839204", driver_license_state="NY")],
         [E(x="E27X", part="person", w="E27W", veto="dl")])
    case("E28", "driver licences of two states do not veto",
         [blank_row("E28X", claim_id="CE28", note_id="NE28", first_name="Rosamund", last_name="Ekwueme", driver_license_number="D2800001", driver_license_state="NY")],
         [blank_row("E28W", first_name="Rosamund", last_name="Ekwueme", driver_license_number="E2800002", driver_license_state="NJ")],
         [E(x="E28X", part="person", w="E28W", veto="", rank=1)])
    case("E29", "email agrees",
         [blank_row("E29X", claim_id="CE29", note_id="NE29", first_name="Thaddeus", last_name="Blackwood", email="T.Blackwood@Mailbox.test")],
         [blank_row("E29W", first_name="Thaddeus", last_name="Blackwood", email="t.blackwood@mailbox.test")],
         [E(x="E29X", part="person", w="E29W", basis="identifier", rank=1)])
    case("E30", "a placeholder DOB is no evidence",
         [blank_row("E30X", claim_id="CE30", note_id="NE30", first_name="Ottoline", last_name="Ferrante", dob="1900-01-01")],
         [blank_row("E30W", first_name="Ottoline", last_name="Ferrante", dob="01/01/1900")],
         [E(x="E30X", part="person", w="E30W", basis="name_only", rank=1, levels={"dob": "empty"})])
    case("E31", "middle initials differ, DOB agrees",
         [blank_row("E31X", claim_id="CE31", note_id="NE31", first_name="Lucius", middle_name="J", last_name="Harrowgate", dob="1964-08-08")],
         [blank_row("E31W", first_name="Lucius", middle_name="K", last_name="Harrowgate", dob="1964-08-08")],
         [E(x="E31X", part="person", w="E31W", basis="dob", rank=1, levels={"middle": "differs"})])
    case("E32", "the trade name of a declared d/b/a",
         [blank_row("E32X", claim_id="CE32", note_id="NE32", business_name="Kestrel Imaging LLC d/b/a Harborview Radiology")],
         [blank_row("E32W", business_name="Harborview Radiology")],
         [E(x="E32X", part="business", w="E32W", rank=1, levels={"org": "exact"})])
    case("E33", "a d/b/a declared by another record",
         [blank_row("E33X", claim_id="CE33", note_id="NE33", business_name="Tamarind Wellness Group"),
          blank_row("E33D", claim_id="CE33b", note_id="NE33b", business_name="Juniper Holistic Center d/b/a Tamarind Wellness Group")],
         [blank_row("E33W", business_name="Juniper Holistic Center")],
         [E(x="E33X", part="business", w="E33W", levels={"org": "dba"}, true=True)])
    case("E34", "abbreviations expanded",
         [blank_row("E34X", claim_id="CE34", note_id="NE34", business_name="Lakeshore Med Ctr")],
         [blank_row("E34W", business_name="Lakeshore Medical Center Inc")],
         [E(x="E34X", part="business", w="E34W", rank=1, levels={"org": "exact"})])
    case("E35", "a short form of the name",
         [blank_row("E35X", claim_id="CE35", note_id="NE35", business_name="Pinecrest Orthopedic")],
         [blank_row("E35W", business_name="Pinecrest Orthopedic Rehabilitation")],
         [E(x="E35X", part="business", w="E35W", rank=1, levels={"org": "short_form"})])
    case("E36", "name plus city and specialty: contextual",
         [blank_row("E36X", claim_id="CE36", note_id="NE36", first_name="Ignatius", last_name="Wojciechowski", provider_specialty="Chiropractor", city="Chicago", state="IL")],
         [blank_row("E36W", first_name="Ignatius", last_name="Wojciechowski", provider_specialty="CHIROPRACTIC", city="CHICAGO", state="IL")],
         [E(x="E36X", part="person", w="E36W", basis="contextual", rank=1)])
    s37 = _ssn(137)
    s37n = s37[:4] + str((int(s37[4]) + 1) % 10) + s37[5:]
    case("E37", "one SSN digit mistyped: near, no veto",
         [blank_row("E37X", claim_id="CE37", note_id="NE37", first_name="Serafina", last_name="Lindgren", ssn=s37n, dob="1990-03-03")],
         [blank_row("E37W", first_name="Serafina", last_name="Lindgren", ssn=s37, dob="1990-03-03")],
         [E(x="E37X", part="person", w="E37W", veto="", rank=1, levels={"ssn": "near"})])
    s38 = _ssn(138)
    s38t = s38[:6] + s38[7] + s38[6] + s38[8:]
    case("E38", "two SSN digits transposed: near, no veto",
         [blank_row("E38X", claim_id="CE38", note_id="NE38", first_name="Augustin", last_name="Merriweather", ssn=s38t, dob="1955-10-10")],
         [blank_row("E38W", first_name="Augustin", last_name="Merriweather", ssn=s38, dob="1955-10-10")],
         [E(x="E38X", part="person", w="E38W", veto="", rank=1, levels={"ssn": "near"})])
    case("E39", "a claim row with only a clinic NPI",
         [blank_row("E39X", claim_id="CE39", note_id="NE39", clinic_npi=_npi(139))],
         [blank_row("E39W", business_name="Harbor Point Physical Therapy", clinic_npi=_npi(139))],
         [E(x="E39X", part="business", w="E39W", basis="identifier", rank=1)])
    case("E40", "surname only, DOB agrees",
         [blank_row("E40X", claim_id="CE40", note_id="NE40", last_name="Quintanilla-Obi", dob="1987-02-25")],
         [blank_row("E40W", first_name="Esperanza", last_name="Quintanilla-Obi", dob="1987-02-25")],
         [E(x="E40X", part="person", w="E40W", basis="dob", rank=1, levels={"name": "first_empty"})])
    case("E41", "LLP: keyword legal",
         [blank_row("E41X", claim_id="CE41", note_id="NE41", business_name="Pemberton & Vasquez LLP")], [],
         [E(x="E41X", part="business", party={"category_inferred": "legal"})])
    case("E42", "a witness is a private party",
         [blank_row("E42X", claim_id="CE42", note_id="NE42", category="witness", first_name="Jeremiah", last_name="Tolliver")], [],
         [E(x="E42X", part="person", party={"prior_group": "private"})])
    case("E43", "one NPI under two different names: identifier basis, the names count against",
         [blank_row("E43X", claim_id="CE43", note_id="NE43", first_name="Alice", last_name="Brennan-Yoo", provider_npi=_npi(143))],
         [blank_row("E43W", first_name="Marcus", last_name="Delacroix-Ibe", provider_npi=_npi(143))],
         [E(x="E43X", part="person", w="E43W", basis="identifier", veto="")])
    case("E44", "licence plate agrees",
         [blank_row("E44X", claim_id="CE44", note_id="NE44", first_name="Horatio", last_name="Villalobos", plate_number="HVX 4412", plate_state="NY")],
         [blank_row("E44W", first_name="Horatio", last_name="Villalobos", plate_number="HVX4412", plate_state="NY")],
         [E(x="E44X", part="person", w="E44W", basis="identifier", rank=1)])
    case("E45", "professional licence agrees; licence type makes it medical",
         [blank_row("E45X", claim_id="CE45", note_id="NE45", first_name="Clementine", last_name="Arbuthnot", professional_license_number="045678", professional_license_state="NY", professional_license_type="DC")],
         [blank_row("E45W", first_name="Clementine", last_name="Arbuthnot", professional_license_number="45678", professional_license_state="NY")],
         [E(x="E45X", part="person", party={"category": "medical", "prior_group": "professional"})])
    return C


def _person_entity(fake, rng, eid, professional):
    first, last = str(fake.first()), str(fake.surname())
    if rng.random() < 0.08:
        last = f"{last}-{fake.surname()}"
    e = {"eid": eid, "kind": "person", "first_name": first.title(), "last_name": last.title(),
         "middle_name": str(fake.first()).title() if rng.random() < 0.6 else "",
         "dob": fake.dob(), "ssn": fake.ssn(), "home_phone": fake.phone(),
         "driver_license_number": fake.dl(), "email": fake.email(first, last)}
    a = fake.address()
    e.update(a)
    e["driver_license_state"] = a["state"]
    if rng.random() < 0.2:
        e["vin"] = fake.vin(); e["plate_number"] = fake.plate(); e["plate_state"] = a["state"]
    if professional:
        spec = str(rng.choice(SPECIALTIES))
        e.update({"provider_npi": fake.npi(), "professional_license_number": fake.digits(6),
                  "professional_license_state": a["state"], "provider_specialty": spec,
                  "category": "medical", "work_phone": fake.phone()})
        if rng.random() < 0.08:
            e.update({"category": "legal", "provider_npi": "", "provider_specialty": "",
                      "professional_license_type": "ATTORNEY"})
    else:
        e["category"] = str(rng.choice(["claimant", "witness", "claimant"]))
    return e


BUSINESS_KINDS = [("Medical", "PC", "medical"), ("Chiropractic", "PC", "medical"),
                  ("Physical Therapy", "PLLC", "medical"), ("Imaging", "LLC", "medical"),
                  ("Acupuncture", "PC", "medical"), ("Rehabilitation Center", "Inc", "medical"),
                  ("Pharmacy", "Inc", "medical"), ("Auto Body", "Inc", "repair shop"),
                  ("Collision", "LLC", "repair shop"), ("Towing", "Corp", "repair shop"),
                  ("Law Offices", "PC", "legal"), ("Legal Group", "LLP", "legal"),
                  ("Diagnostics", "LLC", "medical"), ("Orthopedic Associates", "PC", "medical")]


def _business_entity(fake, rng, eid):
    kind, suffix, cat = BUSINESS_KINDS[int(rng.integers(0, len(BUSINESS_KINDS)))]
    head = str(fake.surname()).title()
    if rng.random() < 0.3:                       # 'Maple Ridge Kowalski ...': not only a surname
        head = f"{str(rng.choice(STREET_NAMES)).title()} {head}"
    name = f"{head} {kind} {suffix}"
    if kind == "Law Offices":
        name = f"Law Offices of {str(fake.first()).title()} {head}"
    if rng.random() < 0.08:
        name = f"{name} d/b/a {str(rng.choice(STREET_NAMES)).title()} {kind}"
    e = {"eid": eid, "kind": "business", "business_name": name, "tin": fake.tin(),
         "work_phone": fake.phone(), "email": f"office{int(rng.integers(1, 99999))}@{head.lower()}.test",
         "category": cat}
    if cat == "medical":
        e["clinic_npi"] = fake.npi()
    e.update(fake.address())
    return e


W_KEEP = {"dob": 0.9, "ssn": 0.3, "provider_npi": 0.9, "clinic_npi": 0.8, "tin": 0.5, "home_phone": 0.4,
          "work_phone": 0.5, "email": 0.2, "driver_license_number": 0.2, "professional_license_number": 0.5,
          "vin": 0.3, "street_name": 0.95, "provider_specialty": 0.9, "category": 0.0, "middle_name": 0.7}
X_KEEP = {"dob": 0.55, "ssn": 0.25, "provider_npi": 0.6, "clinic_npi": 0.5, "tin": 0.5, "home_phone": 0.4,
          "work_phone": 0.3, "email": 0.25, "driver_license_number": 0.2, "professional_license_number": 0.3,
          "vin": 0.3, "street_name": 0.6, "provider_specialty": 0.5, "category": 0.7, "middle_name": 0.5}
_GROUPS = {"street_name": ["street_number", "street_direction", "street_name", "street_type", "unit", "zip"],
           "driver_license_number": ["driver_license_number", "driver_license_state"],
           "professional_license_number": ["professional_license_number", "professional_license_state"],
           "vin": ["vin", "plate_number", "plate_state"]}


def _thin(r, keep, rng):
    for k, p in keep.items():
        if rng.random() >= p:
            for c in _GROUPS.get(k, [k]):
                r[c] = ""
    return r


def _row_from(entities, rid, keep, rng):
    r = blank_row(rid)
    for e in entities:
        for k, v in e.items():
            if k in r and v and (not r[k] or k not in ("category",)):
                r[k] = v
    return _thin(r, keep, rng)


def make_dataset(n_persons, n_businesses, n_watchlist, ref, noise, seed, x_share=0.75, w_share=0.6,
                 w_dup=0.1, x_repeat=0.5, x_noise_scale=0.5, w_noise_scale=0.15, include_edges=True,
                 prefix="S"):
    """(extracted rows, watchlist rows, truth pairs, edge cases). Truth pairs are
    (extracted record_id, part, watchlist record_id) for the same entity and part."""
    rng = np.random.default_rng(seed)
    fake = Fake(ref, rng)
    persons = [_person_entity(fake, rng, f"{prefix}P{i}", rng.random() < 0.35) for i in range(n_persons)]
    biz = [_business_entity(fake, rng, f"{prefix}B{i}") for i in range(n_businesses)]
    owner = {}                                  # business -> professional person on the same rows
    profs = [p for p in persons if p.get("provider_npi") or p.get("professional_license_type")]
    for b in biz:
        if profs and rng.random() < 0.4:
            owner[b["eid"]] = profs[int(rng.integers(0, len(profs)))]
    owners = {o["eid"] for o in owner.values()}
    units = [[p] for p in persons if p["eid"] not in owners]
    units += [[b, owner[b["eid"]]] if b["eid"] in owner else [b] for b in biz]
    w_rows, x_rows, w_ent, x_ent = [], [], [], []
    for u in units:
        if rng.random() < w_share:
            for k in range(2 if rng.random() < w_dup else 1):
                rid = f"{prefix}W{len(w_rows):06d}"
                r = _row_from(u, rid, W_KEEP, rng)
                r["category"] = ""
                if k == 1:
                    r.update(fake.address(r.get("state") or None))
                w_rows.append(r); w_ent.append([e["eid"] for e in u])
        if rng.random() < x_share:
            for k in range(1 + int(rng.random() < x_repeat) + int(rng.random() < x_repeat / 3)):
                rid = f"{prefix}X{len(x_rows):06d}"
                x_rows.append(_row_from(u, rid, X_KEEP, rng)); x_ent.append([e["eid"] for e in u])
    while len(w_rows) < n_watchlist:                   # listed parties never seen in claims
        i = len(w_rows)
        e = _person_entity(fake, rng, f"{prefix}WO{i}", rng.random() < 0.5) if rng.random() < 0.85 else \
            _business_entity(fake, rng, f"{prefix}WO{i}")
        r = _row_from([e], f"{prefix}W{i:06d}", W_KEEP, rng)
        r["category"] = ""
        w_rows.append(r); w_ent.append([e["eid"]])
    W = pd.DataFrame(w_rows, columns=SCHEMA)
    X = pd.DataFrame(x_rows, columns=SCHEMA)
    W, _ = apply_noise(W, noise, fake, rng, scale=w_noise_scale, id_prefix="w")
    X, xlog = apply_noise(X, noise, fake, rng, scale=x_noise_scale, id_prefix="x")
    W["record_id"] = W["record_id"].str.replace("w:", "", regex=False)
    X["record_id"] = X["record_id"].str.replace("x:", "", regex=False)
    # claims and notes: shuffle the rows into claims of 1-5 rows, 1-3 notes each
    order = rng.permutation(len(X))
    ci, pos = 0, 0
    claim = np.empty(len(X), dtype=object); note = np.empty(len(X), dtype=object)
    while pos < len(order):
        k = int(rng.integers(1, 6)); idx = order[pos:pos + k]
        nn = int(rng.integers(1, 4))
        for j, t in enumerate(idx):
            claim[t] = f"{prefix}C{ci:06d}"; note[t] = f"{prefix}C{ci:06d}-N{j % nn}"
        ci += 1; pos += k
    X["claim_id"], X["note_id"] = claim, note
    # truth
    kind = {e["eid"]: e["kind"] for u in units for e in u}
    wi = defaultdict(list)
    for rid, ents in zip(W["record_id"], w_ent):
        for e in ents:
            wi[e].append(rid)
    truth = []
    for rid, ents in zip(X["record_id"], x_ent):
        for e in ents:
            part = "person" if kind.get(e) == "person" else "business"
            for wr in wi.get(e, []):
                truth.append((rid, part, wr))
    edges = edge_cases() if include_edges else []
    if edges:
        X = pd.concat([X, pd.DataFrame([r for c in edges for r in c["x"]], columns=SCHEMA)], ignore_index=True)
        W = pd.concat([W, pd.DataFrame([r for c in edges for r in c["w"]], columns=SCHEMA)], ignore_index=True)
        for c in edges:
            for ex in c["expect"]:
                if ex.get("w") and ex.get("true", ex.get("rank") == 1):
                    truth.append((ex["x"], ex["part"], ex["w"]))
    truth = pd.DataFrame(truth, columns=["x_record_id", "part", "w_record_id"]).drop_duplicates()
    X = X.fillna("").astype(str)
    W = W.fillna("").astype(str)
    return X, W, truth, edges, xlog


def make_scale_set(cfg, maps, ref):
    """The benchmark set: cfg.scale_watchlist watchlist rows and about cfg.scale_extracted
    extracted rows (not committed; written to data/)."""
    units = int(cfg.scale_extracted / 1.25)
    X, W, truth, _, _ = make_dataset(int(units * 0.83), int(units * 0.17), cfg.scale_watchlist, ref,
                                     maps["simulation_noise"], cfg.seed + 7, w_share=0.3,
                                     include_edges=False, prefix="Z")
    return X, W, truth

## LEIE exporter

Raw OIG LEIE (`UPDATED.csv`, 84,001 rows) → the input schema. The export keeps DOB and the
street address, so it lives only in gitignored `data/` (it is personal data about real
people). Mapping, as decided in PLAN.md open question 3:

| LEIE | Input column |
|---|---|
| LASTNAME, FIRSTNAME, MIDNAME | last_name, first_name, middle_name |
| BUSNAME | business_name |
| GENERAL | professional_license_type (drives the inferred category) |
| SPECIALTY | provider_specialty |
| NPI (all zeros = none) | provider_npi for a person, clinic_npi for a business |
| DOB (YYYYMMDD) | dob (ISO) |
| ADDRESS | street_number, street_direction, street_name, street_type, unit (parsed) |
| CITY, STATE, ZIP | city, state, zip |
| EXCLTYPE, EXCLDATE, REINDATE, WAIVERDATE, WVRSTATE, UPIN | x_ display columns, never compared |

`record_id` is goko's scheme (a hash of the record's identifying fields), so it is stable
across refreshes.

**Where Splink would do better.** Not applicable: this is data preparation.

In [18]:
import csv, io

LEIE_URL = "https://oig.hhs.gov/exclusions/downloadables/UPDATED.csv"


def _iso8(d):
    d = (d or "").strip()
    return f"{d[:4]}-{d[4:6]}-{d[6:]}" if len(d) == 8 and d.strip("0") else ""


def export_leie(raw_path, out_path=None):
    """Raw LEIE CSV -> input-schema frame (and CSV when out_path is given)."""
    raw = pd.read_csv(raw_path, dtype=str, keep_default_na=False, na_filter=False, encoding="utf-8",
                      encoding_errors="replace")
    raw = raw.apply(lambda s: s.str.strip())
    key = raw[["LASTNAME", "FIRSTNAME", "MIDNAME", "BUSNAME", "NPI", "CITY", "STATE", "EXCLTYPE",
               "EXCLDATE", "GENERAL", "SPECIALTY"]].apply(lambda s: s.str.upper()).agg("|".join, axis=1)
    rid = "leie:" + key.map(lambda k: hashlib.sha1(k.encode("utf-8")).hexdigest()[:12])
    n = rid.groupby(rid).cumcount()
    rid = np.where(n > 0, rid + "-" + (n + 1).astype(str), rid)
    person = (raw["LASTNAME"] != "") | (raw["FIRSTNAME"] != "")
    npi = raw["NPI"].where(raw["NPI"].str.strip("0") != "", "")
    parts = raw["ADDRESS"].map(parse_street_line)
    out = pd.DataFrame({c: "" for c in SCHEMA}, index=raw.index)
    out["record_id"] = rid
    out["first_name"], out["middle_name"], out["last_name"] = raw["FIRSTNAME"], raw["MIDNAME"], raw["LASTNAME"]
    out["business_name"] = raw["BUSNAME"]
    out["professional_license_type"] = raw["GENERAL"]
    out["provider_specialty"] = raw["SPECIALTY"]
    out["provider_npi"] = np.where(person, npi, "")
    out["clinic_npi"] = np.where(person, "", npi)
    out["dob"] = raw["DOB"].map(_iso8)
    out["street_number"] = parts.map(lambda t: t[0]); out["street_direction"] = parts.map(lambda t: t[1])
    out["street_name"] = parts.map(lambda t: t[2]); out["street_type"] = parts.map(lambda t: t[3])
    out["unit"] = parts.map(lambda t: t[4])
    out["city"], out["state"], out["zip"] = raw["CITY"], raw["STATE"], raw["ZIP"]
    for c in ("EXCLTYPE", "EXCLDATE", "REINDATE", "WAIVERDATE", "WVRSTATE", "UPIN"):
        out[f"x_{c.lower()}"] = raw[c] if c in raw else ""
    out["x_excldate"] = out["x_excldate"].map(_iso8)
    if out_path:
        out.to_csv(out_path, index=False, encoding="utf-8")
    return out


def leie_test_set(leie_rows, ref, noise, seed, n_copies, n_fictional):
    """Extracted rows for the LEIE run: noisy copies of n_copies LEIE rows (the full,
    pessimistic noise table) plus n_fictional parties not on the list; truth pairs for the
    copies. Copies lose the LEIE-only columns and get claim and note ids."""
    rng = np.random.default_rng(seed)
    pick = np.sort(rng.choice(len(leie_rows), size=min(n_copies, len(leie_rows)), replace=False))
    src = leie_rows.iloc[pick][SCHEMA].copy()
    fake = Fake(ref, rng)
    copies, log = apply_noise(src, noise, fake, rng, scale=1.0, id_prefix="lx")
    copies["professional_license_type"] = ""       # GENERAL is LEIE's field, not a claim form's
    copies["category"] = np.where(copies["first_name"].str.len() + copies["last_name"].str.len() > 0,
                                  "medical", "")
    fict, _, _, _, _ = make_dataset(n_fictional, n_fictional // 5, 0, ref, noise, seed + 1, x_share=1.0,
                                    w_share=0.0, include_edges=False, prefix="LF")
    fict = fict.iloc[:n_fictional]
    X = pd.concat([copies, fict], ignore_index=True)
    order = rng.permutation(len(X))
    X["claim_id"] = [f"LC{i // 3:06d}" for i in np.argsort(order)]
    X["note_id"] = X["claim_id"] + "-N0"
    person = (src["first_name"] != "") | (src["last_name"] != "")
    truth = pd.DataFrame({"x_record_id": copies["record_id"].to_numpy(),
                          "part": np.where(person.to_numpy(), "person", "business"),
                          "w_record_id": src["record_id"].to_numpy()})
    return X.fillna("").astype(str), truth, log

## 8-11 · Comparisons, u, m and the prior

**Comparisons (8).** The core's `compare_pairs` gives one level per field for any set of
pairs; the same code runs on candidates, random pairs, anchors, watchlist duplicates and
simulated pairs. Each part (person, business) has its own weights.

**u (9).** `random_pairs` random extracted × watchlist pairs per run (split between parts by
their share of extracted parties) through the same comparisons give the field-level u of
every level (Splink's method), and the rates of the name sub-parts. Exact levels use
value-specific u instead: names from the Census/SSA tables, organization words from NPPES
(or the watchlist's own share), identifiers, dates and addresses from how many hold the value.

**m (10)**, per field and level, in order of preference:

| Source | Pairs | Used for |
|---|---|---|
| 1a strict anchors | extracted pairs sharing a valid one-per-party identifier (SSN, NPI, DL; TIN for businesses) held under exactly one name; at most `anchor_pairs_per_value` pairs per value; same-note pairs excluded | every field except the names (the anchor conditions on the name) and except the anchoring field itself |
| 1b loose anchors | the same identifier types held by 2-20 extracted parties, no name condition; EM with u fixed and the non-name m fixed from 1a | name, middle, org name |
| 2 watchlist duplicates | watchlist pairs sharing an SSN / NPI (TIN / clinic NPI) held by 2-20 parties; persons also by exact name + DOB (then not for name or DOB) | fields still short of data |
| 3 simulation | `sim_records` watchlist rows and their noisy copies (noise table) | any field with < `n_min` informative pairs from 1-2 |
| 4 published | `mappings/published_m.csv` | the base every estimate is shrunk toward |

Each estimate is shrunk toward the next source with alpha = 5 pseudo-pairs; the manifest
records the chain, the pairs behind it and a 95% interval. An agreement level whose m falls
below its u is flagged and counts 0 bits.

**Prior (11)**, per group of the extracted party, over *all* pairs: pairs firing a strict
rule (single-holder identifier agrees with no veto; exact name + DOB; exact name + street;
exact org name + ZIP or + street) / how often a true match fires one of them (from the
estimated m and the fill rates) / all extracted × watchlist pairs of the group. A group with
no strict pair uses a pseudo-count and is flagged.

**Where Splink would do better.** Splink estimates u from up to 10^8 random pairs in
DuckDB and trains m with EM in several passes, each blocked on a different rule and with u
fixed, then averages the passes; here EM runs once, on the loose-anchor set. Splink's prior
comes from a deterministic-rules count with a user-given recall; here recall is computed from
the m estimates. Splink does not use anchors or simulation.

In [19]:
ANCHOR_TYPES = {"person": ["ssn", "npi", "dl"], "business": ["tin"]}
NAME_FREE = {"person": ["name", "middle"], "business": ["org"]}
WDUP_TYPES = {"person": ["ssn", "npi"], "business": ["tin", "cnpi"]}


def _pairs_within(groups, cap, rng, exclude_same=None):
    """All pairs within each group of positions (at most cap per group, deterministic)."""
    ls, rs = [], []
    for g in groups:
        g = np.asarray(g)
        i, j = np.triu_indices(len(g), k=1)
        a, b = g[i], g[j]
        if exclude_same is not None:
            keep = exclude_same[a] != exclude_same[b]
            a, b = a[keep], b[keep]
        if len(a) > cap:
            sel = np.sort(rng.choice(len(a), cap, replace=False))
            a, b = a[sel], b[sel]
        ls.append(a); rs.append(b)
    if not ls:
        return np.array([], np.int64), np.array([], np.int64)
    return np.concatenate(ls).astype(np.int64), np.concatenate(rs).astype(np.int64)


def anchor_pairs(fr, types, holders, name_condition, cap, rng, exclude_notes=True):
    """Pairs of parties within one frame sharing a value of `types`. name_condition=True keeps
    values held under exactly one name (strict); holders=(lo, hi) bounds the parties per value.
    Returns (l, r, anchor-field mask dict)."""
    notes = fr["note_id"].to_numpy(dtype=object) if exclude_notes else None
    if notes is not None:
        notes = np.where(notes == "", "row:" + fr["record_id"].to_numpy(dtype=object), notes)
    found = []
    for t in types:
        col = f"id_{t}"
        v = fr[col].to_numpy(dtype=object)
        d = pd.DataFrame({"pos": np.arange(len(fr)), "v": v, "n": fr["holder_key"].to_numpy(dtype=object)})
        d = d[d["v"] != ""]
        if not len(d):
            continue
        g = d.groupby("v")
        size = g["pos"].size()
        names = g["n"].nunique()
        ok = (size >= holders[0]) & (size <= holders[1])
        if name_condition:
            ok &= names == 1
        keep = set(size.index[ok])
        grp = [x["pos"].to_numpy() for v, x in d[d["v"].isin(keep)].groupby("v", sort=True)]
        l, r = _pairs_within(grp, cap, rng, notes)
        found.append(pd.DataFrame({"l": l, "r": r, "t": t}))
    if not found:
        return np.array([], np.int64), np.array([], np.int64), {}
    f = pd.concat(found, ignore_index=True)
    piv = f.assign(one=True).pivot_table(index=["l", "r"], columns="t", values="one", aggfunc="any",
                                         fill_value=False)
    l = piv.index.get_level_values(0).to_numpy()
    r = piv.index.get_level_values(1).to_numpy()
    return l, r, {t: piv[t].to_numpy(dtype=bool) for t in piv.columns}


def name_dob_pairs(fr, cap, rng):
    d = pd.DataFrame({"pos": np.arange(len(fr)), "k": fr["name_key"].to_numpy(dtype=object) + "|" +
                      fr["dob"].to_numpy(dtype=object)})
    d = d[(fr["dob"].to_numpy() != "") & (fr["first"].to_numpy() != "")]
    size = d.groupby("k")["pos"].size()
    keep = set(size.index[(size >= 2) & (size <= 20)])
    grp = [x["pos"].to_numpy() for _, x in d[d["k"].isin(keep)].groupby("k", sort=True)]
    return _pairs_within(grp, cap, rng)


def with_coparty(levels, left, right, part):
    if part == "person":
        levels["co_party"] = coparty_proxy_levels(left, right, levels["l"].to_numpy(), levels["r"].to_numpy())
    return levels


def estimate_parameters(pr, maps, ref, cfg, wdf, log=print):
    """u, m, weights per part. Returns dict with weights tables, components, source counts."""
    p = cfg.core
    rng = np.random.default_rng(cfg.seed)
    ctx = pr.ctx
    out = {"weights": {}, "u": {}, "sources": [], "components": None, "em": {}, "random_levels": {}}
    nx = {part: len(pr.X[part]) for part in ("person", "business")}
    tot = max(1, sum(nx.values()))
    # ---- u from random pairs ------------------------------------------------------------
    for part in ("person", "business"):
        t_u = time.time()
        X, W = pr.X[part], pr.W[part]
        n = int(min(cfg.random_pairs * nx[part] / tot, len(X) * len(W)))
        if n <= 0 or not len(X) or not len(W):
            out["u"][part] = estimate_u(pd.DataFrame(), PART_FIELDS[part], p)
            continue
        idx = rl.Index()
        idx.add(rl.index.Random(n, replace=True, random_state=cfg.seed))
        mi = idx.index(X[["party_id"]], W[["party_id"]])
        l, r = mi.get_level_values(0).to_numpy(), mi.get_level_values(1).to_numpy()
        # the frames are positional, so index values are positions
        lv = []
        step = 250_000
        for lo in range(0, len(l), step):
            lv.append(compare_pairs(X, W, l[lo:lo + step], r[lo:lo + step], part, ctx))
        lv = with_coparty(pd.concat(lv, ignore_index=True), X, W, part)
        out["u"][part] = estimate_u(lv, PART_FIELDS[part], p)
        out["random_levels"][part] = len(lv)
        if part == "person":
            out["components"] = name_components(lv)
        log(f"u[{part}]: {len(lv):,} random pairs ({time.time() - t_u:.0f}s)")
    ctx.components = out["components"] or {}
    pub = maps["published_m"]
    pub_arr = lambda f: np.array([float(pub[(pub["field"] == f) & (pub["level"] == lev)]["m"].iloc[0])
                                  for lev in FIELD_LEVELS[f]])
    t0 = time.time()
    # ---- simulation: noisy copies of watchlist rows ----------------------------------------
    k = min(cfg.sim_records, len(wdf))
    pick = np.sort(rng.choice(len(wdf), size=k, replace=False)) if k else np.array([], int)
    src = wdf.iloc[pick][SCHEMA].reset_index(drop=True)
    fake = Fake(ref, rng)
    copies, _ = apply_noise(src, maps["simulation_noise"], fake, rng, scale=1.0, id_prefix="sim")
    nick = ctx.nick
    _, cpar, _, _, _, _ = rows_to_parties(copies, "S", maps, nick, p, False)
    for c in ("id_ssn", "id_npi", "id_dl", "id_license", "id_tin", "id_cnpi", "id_email", "id_vin",
              "id_plate", "phone_own", "phone_row"):
        pass
    C = split_parts(cpar)
    sim_counts = {}
    for part in ("person", "business"):
        Cf, W = C[part], pr.W[part]
        wpos = pd.Series(np.arange(len(W)), index=W["record_id"].to_numpy())
        orig = Cf["record_id"].str.replace("sim:", "", regex=False)
        ok = orig.isin(wpos.index).to_numpy()
        l = np.flatnonzero(ok)
        r = wpos[orig[ok]].to_numpy() if ok.any() else np.array([], np.int64)
        # one watchlist record_id can have both parts; wpos is per part, so ids are unique
        lv = with_coparty(compare_pairs(Cf, W, l, r, part, ctx), Cf, W, part)
        sim_counts[part] = level_counts(lv, PART_FIELDS[part])
        out["sources"].append({"part": part, "source": "simulation", "pairs": len(lv)})
    log(f"simulation: {k:,} noisy copies compared ({time.time() - t0:.0f}s)")
    t0 = time.time()
    # ---- 1a strict anchors, 1b loose anchors (extracted x extracted) ---------------------
    # ---- 2 watchlist duplicates ---------------------------------------------------------
    for part in ("person", "business"):
        X, W = pr.X[part], pr.W[part]
        fields = PART_FIELDS[part]
        non_name = [f for f in fields if f not in NAME_FREE[part]]
        l, r, amask = anchor_pairs(X, ANCHOR_TYPES[part], (2, 10 ** 9), True, cfg.anchor_pairs_per_value, rng)
        lv1 = with_coparty(compare_pairs(X, X, l, r, part, ctx), X, X, part)
        c1a = level_counts(lv1, non_name, exclude=amask)
        out["sources"].append({"part": part, "source": "anchors_strict", "pairs": len(lv1)})
        l, r, dmask = anchor_pairs(W, WDUP_TYPES[part], (2, 20), False, cfg.anchor_pairs_per_value, rng,
                                   exclude_notes=False)
        lv2 = with_coparty(compare_pairs(W, W, l, r, part, ctx), W, W, part)
        c2 = level_counts(lv2, fields, exclude=dmask)
        n_wdup = len(lv2)
        if part == "person":
            l, r = name_dob_pairs(W, cfg.anchor_pairs_per_value, rng)
            lv2b = with_coparty(compare_pairs(W, W, l, r, part, ctx), W, W, part)
            c2b = level_counts(lv2b, [f for f in fields if f not in ("name", "dob")])
            for f, (cnt, n) in c2b.items():
                c0, n0 = c2.get(f, (np.zeros(len(FIELD_LEVELS[f])), 0.0))
                c2[f] = (c0 + cnt, n0 + n)
            n_wdup += len(lv2b)
        out["sources"].append({"part": part, "source": "watchlist_duplicates", "pairs": n_wdup})
        # preliminary m for the fixed (non-name) fields, then EM for the names
        rows = []
        for f in non_name:
            srcs = [("anchors_strict", *c1a.get(f, (None, 0.0))), ("watchlist_duplicates", *c2.get(f, (None, 0.0))),
                    ("simulation", *sim_counts[part].get(f, (None, 0.0)))]
            srcs = [(a, b if b is not None else np.zeros(len(FIELD_LEVELS[f])), c) for a, b, c in srcs]
            rows += combine_m(f, srcs, pub_arr(f), p)
        mfix = pd.DataFrame(rows)
        u_df = out["u"][part]
        l, r, lmask = anchor_pairs(X, ANCHOR_TYPES[part], cfg.loose_holders, False, cfg.anchor_pairs_per_value, rng)
        lv1b = with_coparty(compare_pairs(X, X, l, r, part, ctx), X, X, part)
        fixed = [f for f in non_name if f != "co_party"]
        m_fixed = {f: mfix[mfix["field"] == f].sort_values("level_code")["m"].to_numpy() for f in fixed}
        u_tab = {f: u_df[u_df["field"] == f].sort_values("level_code")["u"].to_numpy() for f in fields if f != "co_party"}
        em, lam, iters = em_fixed_u(lv1b, NAME_FREE[part], fixed, m_fixed, u_tab, p, exclude=lmask)
        out["em"][part] = {"pairs": len(lv1b), "lambda": lam, "iterations": iters}
        out["sources"].append({"part": part, "source": "anchors_loose_em", "pairs": len(lv1b)})
        for f in NAME_FREE[part]:
            cnt, n = (em[f][2], em[f][1]) if f in em else (np.zeros(len(FIELD_LEVELS[f])), 0.0)
            srcs = [("anchors_loose_em", cnt, n), ("watchlist_duplicates", *c2.get(f, (np.zeros(len(FIELD_LEVELS[f])), 0.0))),
                    ("simulation", *sim_counts[part].get(f, (np.zeros(len(FIELD_LEVELS[f])), 0.0)))]
            rows += combine_m(f, srcs, pub_arr(f), p)
        w = weights_table(rows, u_df, out["components"])
        w.insert(0, "part", part)
        out["weights"][part] = w
        log(f"m[{part}]: strict anchors {len(lv1):,}, loose anchors {len(lv1b):,} (EM lambda {lam:.3f}, "
            f"{iters} iterations; {time.time() - t0:.0f}s), watchlist duplicates {n_wdup:,}, simulated {out['sources'][0 if part == 'person' else 1]['pairs']:,}")
    return out


# ---- prior ------------------------------------------------------------------------------------
def strict_pairs(X, W, part, ctx):
    """Extracted x watchlist pairs firing a strict rule (not vetoed), as (l, r)."""
    found = []

    def join(kx, kw):
        a = pd.DataFrame({"k": kx, "l": np.arange(len(X))})
        b = pd.DataFrame({"k": kw, "r": np.arange(len(W))})
        a, b = a[a["k"] != ""], b[b["k"] != ""]
        m = a.merge(b, on="k")
        found.append(m[["l", "r"]])

    single = ctx.stats.single
    ids = ["ssn", "npi", "dl", "license", "email", "vin", "plate"] if part == "person" else ["tin", "cnpi", "email"]
    for t in ids:
        kx = X[f"id_{t}"].to_numpy(dtype=object)
        kx = np.where(pd.Series(kx).isin(single.get(t, set())).to_numpy(), kx, "")
        join(kx, W[f"id_{t}"].to_numpy(dtype=object))
    nk = lambda F: np.where((F["first"] != "") | (F["part"] == "business"), F["name_key"], "").astype(object)
    if part == "person":
        join(np.where(X["dob"] != "", nk(X) + "|" + X["dob"], ""), np.where(W["dob"] != "", nk(W) + "|" + W["dob"], ""))
    else:
        join(np.where(X["zip"] != "", nk(X) + "|" + X["zip"], ""), np.where(W["zip"] != "", nk(W) + "|" + W["zip"], ""))
    join(np.where(X["addr_street"] != "", nk(X) + "|" + X["addr_street"], ""),
         np.where(W["addr_street"] != "", nk(W) + "|" + W["addr_street"], ""))
    f = pd.concat(found, ignore_index=True).drop_duplicates()
    if not len(f):
        return f["l"].to_numpy(), f["r"].to_numpy()
    lv = compare_pairs(X, W, f["l"].to_numpy(), f["r"].to_numpy(), part, ctx)
    veto = np.zeros(len(lv), bool)
    for t in VETO_FIELDS:
        if f"veto_{t}" in lv:
            veto |= lv[f"veto_{t}"].to_numpy(dtype=bool)
    return f["l"].to_numpy()[~veto], f["r"].to_numpy()[~veto]


def fill_rates(Xg, W, part):
    sh = lambda F, c: float((F[c] != "").mean()) if len(F) else 0.0
    both = lambda c: sh(Xg, c) * sh(W, c)
    fill = {t: both(f"id_{t}") for t in ("ssn", "npi", "dl", "license", "email", "vin", "plate", "tin", "cnpi")
            if f"id_{t}" in Xg}
    if part == "person":
        full = lambda F: float(((F["first"] != "") & (F["last"] != "")).mean()) if len(F) else 0.0
        fill["name"] = full(Xg) * full(W)
        fill["dob"] = both("dob")
    else:
        fill["org"] = both("org_aliases")
        fill["zip"] = both("zip")
    fill["street"] = both("addr_street")
    return fill


def estimate_prior(pr, params_out, cfg, log=print):
    rows = []
    prior = {}
    for part in ("person", "business"):
        X, W = pr.X[part], pr.W[part]
        w = params_out["weights"][part]
        mlook = lambda f, lev: float(w[(w["field"] == f) & (w["level"] == lev)]["m"].iloc[0]) if ((w["field"] == f) & (w["level"] == lev)).any() else 0.0
        l, r = strict_pairs(X, W, part, pr.ctx)
        grp = X["prior_group"].to_numpy()
        for g in GROUPS:
            gm = grp == g
            if not gm.any():
                continue
            n_strict = int(gm[l].sum()) if len(l) else 0
            n_pairs = int(gm.sum()) * len(W)
            fill = fill_rates(X[gm], W, part)
            rec = strict_recall(fill, mlook)
            pv, flagged = prior_estimate(n_strict, n_pairs, rec, cfg.core)
            prior[(part, g)] = pv
            rows.append({"part": part, "group": g, "extracted_parties": int(gm.sum()),
                         "watchlist_parties": len(W), "all_pairs": n_pairs, "strict_pairs": n_strict,
                         "recall": rec, "prior": pv, "prior_logit_bits": math.log2(pv / (1 - pv)),
                         "flag": "pseudo-count: no strict pair or no recall" if flagged else ""})
    df = pd.DataFrame(rows)
    log(df[["part", "group", "extracted_parties", "strict_pairs", "recall", "prior", "flag"]].to_string(index=False))
    return prior, df

## 12 · Score

In chunks of extracted parties: candidates → comparisons → the core's Fellegi-Sunter scorer
(sum of log2(m/u) per field, value-specific u on the levels that have one, 0 bits when a field
is empty, agreement never below 0) → vetoes (p = 0, visible, with the reason) → basis.
Businesses are scored first; their identifier links at p >= 0.9 with no veto are the anchors
for the **co-party** pass on persons, which adds bits only where the names already agree
(one way: a co-party can strengthen a name, never make one, and name-only links never vouch
for each other).

A pair is **kept** when p >= `keep_p_floor`, or it is among its entity's top K (ties kept), or
it rests on an identifier, or it is vetoed. Pairs are never filtered by basis: a name-only
match is kept on the same terms as any other.

**Where Splink would do better.** Splink's `predict()` scores every blocked pair in SQL with
term-frequency adjustments and writes them all; the keep rule here exists because the output
is a workbook. Splink has no vetoes or basis classes: both are ours.

In [20]:
P_BINS = np.array([0, 1e-3, 0.01, 0.1, 0.5, 0.8, 0.9, 0.99, 1.0000001])


def _rank_within(l, p):
    """Rank of each pair within its entity (l), best first; ties share the rank ('min')."""
    d = pd.DataFrame({"l": l, "p": p})
    return d.groupby("l")["p"].rank(method="min", ascending=False).to_numpy()


def score_part(pr, part, weights, prior, cfg, anchors=None, log=print):
    """Score every candidate of one part. Returns (kept pairs frame, kept levels frame,
    per-entity summary, diagnostics)."""
    X, W = pr.X[part], pr.W[part]
    ctx = pr.ctx
    plan = CandidatePlan(X, W, part, ctx.rarity, cfg, ctx.dba_pairs)
    for rep in plan.report:
        log(f"  {part:<8} {rep['rule']:<22} pairs {rep['pairs_before_refine']:>12,}"
            + (f"  oversized keys {rep['oversized_keys']} -> dropped {rep['dropped_keys']} ({rep['dropped_pairs']:,} pairs)"
               if rep['oversized_keys'] or rep['dropped_keys'] else ""))
    grp_logit = {g: math.log2(prior[(part, g)] / (1 - prior[(part, g)])) for g in GROUPS if (part, g) in prior}
    x_logit = X["prior_group"].map(grp_logit).fillna(0.0).to_numpy(dtype=float)
    kept_s, kept_l, summ, all_keys = [], [], [], []
    hist = np.zeros((len(BASIS_ORDER), len(P_BINS) - 1), dtype=np.int64)
    n_scored = 0
    rule_counts = defaultdict(int)
    for lo in range(0, len(X), cfg.chunk_size):
        hi = min(len(X), lo + cfg.chunk_size)
        c = plan.chunk(lo, hi)
        if not len(c):
            continue
        l, r = c["l"].to_numpy(), c["r"].to_numpy()
        all_keys.append(l.astype(np.int64) * np.int64(1 << 32) + r)
        for name, bit in plan.bit.items():
            rule_counts[name] += int(((c["rules"].to_numpy() & bit) > 0).sum())
        lv = compare_pairs(X, W, l, r, part, ctx)
        pl = x_logit[l]
        if part == "person":
            tl, tr = X["tie_pos"].to_numpy()[l], W["tie_pos"].to_numpy()[r]
            lv["co_party"] = np.where((tl >= 0) & (tr >= 0), 1, EMPTY).astype(np.int8)
            sc = score_pairs(lv, part, weights, pl)
            anch = np.zeros(len(lv), bool)
            if anchors is not None and len(anchors):
                has = (tl >= 0) & (tr >= 0)
                anch[has] = np.isin(tl[has].astype(np.int64) * np.int64(1 << 32) + tr[has], anchors)
            if anch.any():
                lv, sc = apply_coparty(lv, sc, weights, pl, anch)
        else:
            sc = score_pairs(lv, part, weights, pl)
        n_scored += len(sc)
        p = sc["p"].to_numpy()
        basis = sc["basis"].to_numpy()
        bi = pd.Series(basis).map({b: i for i, b in enumerate(BASIS_ORDER)}).to_numpy()
        pbin = np.clip(np.searchsorted(P_BINS, p, side="right") - 1, 0, len(P_BINS) - 2)
        vetoed = sc["veto"].to_numpy() != ""
        np.add.at(hist, (bi[~vetoed], pbin[~vetoed]), 1)
        # rank among non-vetoed candidates of the entity
        pr_rank = np.where(vetoed, -1.0, p)
        rank = _rank_within(l, pr_rank)
        keep = (p >= cfg.keep_p_floor) | (rank <= cfg.top_k) | (basis == "identifier") | vetoed
        s = sc[keep].copy()
        s.insert(0, "rank", np.where(vetoed[keep], np.nan, rank[keep]))
        s.insert(0, "rules", plan.rule_names(c["rules"].to_numpy()[keep]))
        s.insert(0, "r", r[keep]); s.insert(0, "l", l[keep])
        s["prior_logit"] = pl[keep]
        kept_s.append(s.reset_index(drop=True))
        kept_l.append(lv[keep].reset_index(drop=True))
        cnt = pd.DataFrame({"l": l, "p": p, "veto": vetoed}).groupby("l").agg(
            candidates=("p", "size"), vetoed=("veto", "sum"))
        summ.append(cnt)
    cols_s = None
    S = pd.concat(kept_s, ignore_index=True) if kept_s else pd.DataFrame()
    L = pd.concat(kept_l, ignore_index=True) if kept_l else pd.DataFrame()
    E = pd.concat(summ) if summ else pd.DataFrame(columns=["candidates", "vetoed"])
    keys = np.concatenate(all_keys) if all_keys else np.array([], np.int64)
    diag = {"hist": hist, "scored": n_scored, "rule_counts": dict(rule_counts), "report": plan.report,
            "keys": keys}
    log(f"  {part}: {n_scored:,} pairs scored, {len(S):,} kept")
    return S, L, E, diag


def business_anchors(S, cfg):
    if not len(S):
        return np.array([], np.int64)
    m = (S["basis"] == "identifier") & (S["p"] >= cfg.core.coparty_min_p) & (S["veto"] == "")
    return np.unique(S.loc[m, "l"].to_numpy().astype(np.int64) * np.int64(1 << 32) + S.loc[m, "r"].to_numpy())


def score_all(pr, params_out, prior, cfg, log=print):
    res = {}
    Sb, Lb, Eb, Db = score_part(pr, "business", params_out["weights"]["business"], prior, cfg, log=log)
    anchors = business_anchors(Sb, cfg)
    log(f"  co-party anchors (business links on an identifier, p >= {cfg.core.coparty_min_p}): {len(anchors)}")
    Sp, Lp, Ep, Dp = score_part(pr, "person", params_out["weights"]["person"], prior, cfg, anchors=anchors, log=log)
    res["business"] = (Sb, Lb, Eb, Db)
    res["person"] = (Sp, Lp, Ep, Dp)
    return res

## 13 · Roll-up

**Entities**: one row per extracted party (record_id + part). Its probability is its single
best (non-vetoed) match; every other kept candidate is listed. `rests_on_name` is true when
the best match's basis is contextual or name_only: a filter, never a threshold. An extracted
party with no candidate is shown as such ("no candidate proposed"), not dropped.

**Claims**: per claim, rows, parties, the highest p and its entity, counts by basis x p band,
and name-only matches at p >= 0.5.

**Where Splink would do better.** Splink clusters pairwise predictions into entities
(connected components at a threshold); here the unit is one extracted party against the list,
so the roll-up is a best-match choice with the others listed, per DESIGN.md.

In [21]:
def pair_table(pr, part, S):
    """Kept pairs with identities, names and the ranking reason."""
    if not len(S):
        return pd.DataFrame()
    X, W = pr.X[part], pr.W[part]
    l, r = S["l"].to_numpy(), S["r"].to_numpy()
    out = pd.DataFrame({
        "pair_id": X["party_id"].to_numpy()[l] + "~" + W["party_id"].to_numpy()[r],
        "extracted_record_id": X["record_id"].to_numpy()[l], "part": part,
        "claim_id": X["claim_id"].to_numpy()[l], "extracted_name": X["display_name"].to_numpy()[l],
        "watchlist_record_id": W["record_id"].to_numpy()[r], "watchlist_name": W["display_name"].to_numpy()[r],
        "p": S["p"].to_numpy(), "bits": S["bits"].to_numpy(), "prior_bits": S["prior_logit"].to_numpy(),
        "basis": S["basis"].to_numpy(), "rests_on_name": np.isin(S["basis"].to_numpy(), ["contextual", "name_only"]),
        "veto": S["veto"].to_numpy(), "rank": S["rank"].to_numpy(), "proposed_by": S["rules"].to_numpy(),
        "_l": l, "_r": r})
    for f in PART_FIELDS[part]:
        out[f"bits_{f}"] = S[f"bits_{f}"].to_numpy()
    return out


def rollup_entities(pr, pairs, E_by_part, cfg):
    ent = []
    for part in ("person", "business"):
        X = pr.X[part]
        E = E_by_part[part]
        base = pd.DataFrame({
            "party_id": X["party_id"], "record_id": X["record_id"], "part": part, "claim_id": X["claim_id"],
            "note_id": X["note_id"], "name": X["display_name"],
            "category_given": X["category_given"], "category_inferred": X["category_inferred"],
            "category_rule": X["category_rule"], "category_used": X["category"],
            "category_mismatch": X["category_mismatch"], "prior_group": X["prior_group"]})
        base["candidates"] = E["candidates"].reindex(np.arange(len(X))).fillna(0).astype(int).to_numpy() if len(E) else 0
        base["vetoed_candidates"] = E["vetoed"].reindex(np.arange(len(X))).fillna(0).astype(int).to_numpy() if len(E) else 0
        P = pairs[pairs["part"] == part] if len(pairs) else pd.DataFrame()
        if len(P):
            ok = P[P["veto"] == ""].sort_values(["_l", "p", "bits", "watchlist_record_id"],
                                                ascending=[True, False, False, True], kind="stable")
            best = ok.groupby("_l").head(1).set_index("_l")
            non_name = ok[~ok["rests_on_name"] & (ok["basis"] != "none")].groupby("_l")["p"].max()
            pos = ok.groupby("_l").cumcount().to_numpy()
            sub = ok[(pos >= 1) & (pos <= cfg.top_k)]
            txt = (sub["watchlist_record_id"] + " (" + sub["p"].map(lambda v: f"{v:.3g}") + ", " + sub["basis"] + ")")
            others = txt.groupby(sub["_l"].to_numpy()).agg("; ".join)
            idx = np.arange(len(X))
            base["best_watchlist_record_id"] = best["watchlist_record_id"].reindex(idx).fillna("").to_numpy()
            base["best_watchlist_name"] = best["watchlist_name"].reindex(idx).fillna("").to_numpy()
            base["p"] = best["p"].reindex(idx).fillna(0.0).to_numpy()
            base["basis"] = best["basis"].reindex(idx).fillna("none").to_numpy()
            base["best_non_name_p"] = non_name.reindex(idx).fillna(0.0).to_numpy()
            base["other_candidates"] = others.reindex(idx).fillna("").to_numpy()
        else:
            base["best_watchlist_record_id"] = ""; base["best_watchlist_name"] = ""
            base["p"] = 0.0; base["basis"] = "none"; base["best_non_name_p"] = 0.0; base["other_candidates"] = ""
        base["rests_on_name"] = np.isin(base["basis"].to_numpy(), ["contextual", "name_only"])
        base["status"] = np.where(base["candidates"] == 0, "no candidate proposed",
                                  np.where(base["best_watchlist_record_id"] == "", "only vetoed candidates", "scored"))
        ent.append(base)
    return pd.concat(ent, ignore_index=True)


def p_band(p):
    return np.select([p >= 0.9, p >= 0.8, p >= 0.5, p >= 0.1], ["p>=0.9", "0.8-0.9", "0.5-0.8", "0.1-0.5"], "p<0.1")


def rollup_claims(entities, xrows_claims):
    e = entities.copy()
    e["band"] = p_band(e["p"].to_numpy())
    g = e.groupby("claim_id", sort=True)
    top = e.sort_values(["claim_id", "p"], ascending=[True, False], kind="stable").groupby("claim_id").head(1).set_index("claim_id")
    out = pd.DataFrame({"rows": xrows_claims.reindex(g.size().index).fillna(0).astype(int),
                        "parties": g.size(), "max_p": g["p"].max()})
    out["max_p_entity"] = top["party_id"].reindex(out.index)
    out["max_p_name"] = top["name"].reindex(out.index)
    out["max_p_basis"] = top["basis"].reindex(out.index)
    ct = pd.crosstab(e["claim_id"], e["basis"] + " " + e["band"])
    out = out.join(ct, how="left").fillna(0)
    out["name_only_at_p_0.5"] = e[(e["basis"] == "name_only") & (e["p"] >= 0.5)].groupby("claim_id").size().reindex(out.index).fillna(0).astype(int)
    return out.reset_index().rename(columns={"index": "claim_id"})

## 14 · Evidence

One row per kept pair and compared field that is not empty: both values as normalized,
the level, m with its source chain and pair count, the u actually used (value-specific or
field-level) with its source and count, and the bits. A pair's rows sum to its total bits
(checked in the self-tests); the prior is on the candidate row.

**Where Splink would do better.** Splink's waterfall chart shows the same breakdown
graphically, per pair, in the browser; this is its table form, filterable in Excel.

In [22]:
DISPLAY = {"name": lambda F: (F["first"] + " " + F["last"]).str.strip(), "middle": lambda F: F["middle"],
           "org": lambda F: F["org_aliases"].str.replace("|", " / ", regex=False), "dob": lambda F: F["dob"],
           "address": lambda F: F["raw_address"], "phone": lambda F: (F["phone_own"] + " " + F["phone_row"]).str.strip(),
           "spec_cat": lambda F: (F["specialty"] + " / " + F["category"]).str.strip(" /"),
           "co_party": lambda F: F["tie"]}


def evidence_for(pr, part, pairs_part, levels_part, S_part, weights):
    X, W = pr.X[part], pr.W[part]
    l, r = S_part["l"].to_numpy(), S_part["r"].to_numpy()
    vl, vr = {}, {}
    for f in PART_FIELDS[part]:
        if f in DISPLAY:
            vl[f] = DISPLAY[f](X).to_numpy(dtype=object)[l]
            vr[f] = DISPLAY[f](W).to_numpy(dtype=object)[r]
        elif f"id_{f}" in X:
            vl[f] = X[f"id_{f}"].to_numpy(dtype=object)[l]
            vr[f] = W[f"id_{f}"].to_numpy(dtype=object)[r]
    return evidence_rows(pairs_part["pair_id"].to_numpy(), levels_part, S_part, part, weights, vl, vr)


def sensitivity(entities_best, prior_df, cfg):
    """How many entities cross 0.5 / 0.8 / 0.9 if the prior or the recall behind it moved."""
    rows = []
    for _, pr_ in prior_df.iterrows():
        sub = entities_best[(entities_best["part"] == pr_["part"]) & (entities_best["prior_group"] == pr_["group"])]
        bits = sub["bits"].to_numpy()
        for mult in cfg.sensitivity_prior_mult:
            for rmult in cfg.sensitivity_recall:
                rec = min(1.0, pr_["recall"] * rmult) if pr_["recall"] > 0 else 0.0
                base = pr_["prior"] * mult * (pr_["recall"] / rec if rec > 0 else 1.0)
                base = min(base, 0.5)
                logit = math.log2(base / (1 - base)) + bits
                p = 1 / (1 + np.exp2(-logit))
                row = {"part": pr_["part"], "group": pr_["group"], "prior_multiplier": mult,
                       "recall_multiplier": rmult, "prior": base, "entities": len(sub)}
                for t in cfg.p_bands:
                    row[f"entities_p>={t}"] = int((p >= t).sum())
                rows.append(row)
    return pd.DataFrame(rows)

## 15 · Workbook and manifest

One Excel workbook, one sheet per table: **entities**, **candidates**, **evidence**,
**claims**, **manifest**. A table longer than Excel's row limit continues on `name (2)`,
`name (3)`, ... Every sheet has filters; `basis` and `rests_on_name` are there to filter on,
never thresholds. The manifest is also written as JSON, and every table as Parquet, beside the
workbook in gitignored `out/<dataset>/`.

The manifest holds: the run configuration, input and reference hashes, input reports,
unmapped category values, blocking counts, every m and u with its source, pair count and
interval, the name sub-part rates, the priors with their counts, recall and flags, the
sensitivity table, data-quality findings (invalid values, junk values, empty rows, category
disagreements), vetoes and the timing of each step.

**Where Splink would do better.** Splink writes predictions to a database table or Parquet
and its model JSON records the trained parameters; it has no workbook. Excel is this
package's requirement, with its row limit and write time.

In [23]:
def manifest_rows(section, df, key_cols, value_col, source_col=None, pairs_col=None, lo=None, hi=None, note_col=None):
    out = []
    for _, r in df.iterrows():
        out.append({"section": section, "key": "|".join(str(r[c]) for c in key_cols),
                    "value": r[value_col], "source": r[source_col] if source_col else "",
                    "pairs": r[pairs_col] if pairs_col else "",
                    "ci_low": r[lo] if lo else "", "ci_high": r[hi] if hi else "",
                    "note": r[note_col] if note_col else ""})
    return out


def _clean_cell(v):
    if v is None:
        return ""
    if isinstance(v, (float, np.floating)):
        if np.isnan(v):
            return ""
        return float(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    return v if isinstance(v, (int, float, str)) else str(v)


def _column_values(s):
    """A column as Python values xlsxwriter writes directly: NaN and '' become blanks."""
    a = s.to_numpy()
    if a.dtype.kind in "fc":
        return [None if v != v else float(v) for v in a.tolist()]
    if a.dtype.kind in "iu":
        return a.tolist()
    if a.dtype.kind == "b":
        return a.tolist()
    return [None if (v is None or v == "" or (isinstance(v, float) and v != v)) else
            (v if isinstance(v, (str, int, float, bool)) else str(v)) for v in a.tolist()]


def write_workbook(path, tables, row_limit, filters=None):
    """tables: ordered {sheet name: DataFrame}. Streams rows (xlsxwriter constant_memory).
    Returns {sheet: [sheet names written]}."""
    import xlsxwriter
    wb = xlsxwriter.Workbook(str(path), {"constant_memory": True, "strings_to_urls": False,
                                         "strings_to_formulas": False, "nan_inf_to_errors": True})
    head = wb.add_format({"bold": True, "bg_color": "#DDE3EA"})
    written = {}
    for name, df in tables.items():
        cols = [str(c) for c in df.columns]
        n = len(df)
        parts = max(1, math.ceil(n / row_limit))
        written[name] = []
        cols_py = [_column_values(df[c]) for c in df.columns]
        for k in range(parts):
            sname = name if k == 0 else f"{name} ({k + 1})"
            ws = wb.add_worksheet(sname[:31])
            written[name].append(sname)
            ws.write_row(0, 0, cols, head)
            lo, hi = k * row_limit, min(n, (k + 1) * row_limit)
            for i in range(lo, hi):
                ws.write_row(i - lo + 1, 0, [col[i] for col in cols_py])
            ws.autofilter(0, 0, max(1, hi - lo), max(0, len(cols) - 1))
            ws.freeze_panes(1, 0)
    wb.close()
    return written


def save_tables(out_dir, tables, manifest):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, df in tables.items():
        d = df.copy()
        for c in d.columns:
            if d[c].dtype == object:
                d[c] = d[c].map(lambda v: "" if v is None else (v if isinstance(v, str) else str(v)))
        d.to_parquet(out_dir / f"{name}.parquet", index=False)
    (out_dir / "manifest.json").write_text(json.dumps(manifest.to_dict("records"), indent=1, default=str),
                                           encoding="utf-8")

## 16 · Diagnostics and the manifest

**Where Splink would do better.** Splink's diagnostics are charts: match-weight
distributions, parameter-estimate comparisons across training passes, m/u per level, unlinkable
records, cluster studio. These are tables in the manifest and printed summaries.

In [24]:
def identifier_anchor_recall(pr, scored, part):
    """Pairs sharing a valid single-holder identifier: share proposed by at least one
    non-identifier rule (the name rules alone)."""
    X, W = pr.X[part], pr.W[part]
    S = scored[part][0]
    ids = ["ssn", "npi", "dl", "license", "email", "vin", "plate"] if part == "person" else ["tin", "cnpi", "email"]
    single = pr.ctx.stats.single
    found = []
    for t in ids:
        a = pd.DataFrame({"k": X[f"id_{t}"].to_numpy(dtype=object), "l": np.arange(len(X))})
        b = pd.DataFrame({"k": W[f"id_{t}"].to_numpy(dtype=object), "r": np.arange(len(W))})
        a = a[(a["k"] != "") & a["k"].isin(single.get(t, set()))]
        b = b[b["k"] != ""]
        found.append(a.merge(b, on="k")[["l", "r"]])
    A = pd.concat(found).drop_duplicates() if found else pd.DataFrame(columns=["l", "r"])
    if not len(A) or not len(S):
        return {"part": part, "identifier_anchor_pairs": len(A), "proposed_by_name_rules": 0, "recall": float("nan")}
    rules = pd.Series(S["rules"].to_numpy(), index=pd.MultiIndex.from_arrays([S["l"], S["r"]]))
    got = rules.reindex(pd.MultiIndex.from_frame(A)).fillna("")
    name_rule = got.map(lambda s: any(x and x not in IDENTIFIER_RULES for x in s.split(",")))
    return {"part": part, "identifier_anchor_pairs": len(A), "proposed_by_name_rules": int(name_rule.sum()),
            "recall": float(name_rule.mean())}


def truth_recall(pr, scored, pairs, truth, part):
    X, W = pr.X[part], pr.W[part]
    t = truth[truth["part"] == part]
    xi = pd.Series(np.arange(len(X)), index=X["record_id"].to_numpy())
    wi = pd.Series(np.arange(len(W)), index=W["record_id"].to_numpy())
    t = t[t["x_record_id"].isin(xi.index) & t["w_record_id"].isin(wi.index)]
    if not len(t):
        return {"part": part, "true_pairs": 0}
    keys = xi[t["x_record_id"]].to_numpy().astype(np.int64) * np.int64(1 << 32) + wi[t["w_record_id"]].to_numpy()
    proposed = np.isin(keys, scored[part][3]["keys"])
    P = pairs[pairs["part"] == part]
    kept_keys = P["_l"].to_numpy().astype(np.int64) * np.int64(1 << 32) + P["_r"].to_numpy()
    kept = np.isin(keys, kept_keys)
    kp = P.set_index(kept_keys)
    tb = kp.reindex(keys[kept])
    top1 = (tb["rank"] == 1).to_numpy()
    name_only = tb["rests_on_name"].to_numpy(dtype=bool)
    return {"part": part, "true_pairs": len(t), "proposed": int(proposed.sum()), "recall": float(proposed.mean()),
            "kept": int(kept.sum()), "kept_share": float(kept.mean()),
            "true_pairs_ranked_first": int(top1.sum()),
            "true_rests_on_name_kept": int(name_only.sum()),
            "true_by_basis": tb["basis"].value_counts().to_dict(),
            "true_p_ge_0.5": int((tb["p"] >= 0.5).sum()), "true_p_ge_0.9": int((tb["p"] >= 0.9).sum()),
            "missed_examples": list(t[~proposed]["x_record_id"].head(10))}


def diagnostics(pr, scored, pairs, cfg, truth_file=None, log=print):
    d = {"anchor_recall": [identifier_anchor_recall(pr, scored, part) for part in ("person", "business")]}
    for a in d["anchor_recall"]:
        log(f"blocking recall on identifier anchors [{a['part']}]: {a.get('proposed_by_name_rules', 0)} of "
            f"{a['identifier_anchor_pairs']} also proposed by a name rule ({a['recall']:.3f})")
    d["truth"] = []
    if truth_file is not None and Path(truth_file).exists():
        truth = pd.read_csv(truth_file, dtype=str, keep_default_na=False)
        d["truth"] = [truth_recall(pr, scored, pairs, truth, part) for part in ("person", "business")]
        for t in d["truth"]:
            if t.get("true_pairs"):
                log(f"truth [{t['part']}]: {t['true_pairs']} true pairs, proposed {t['recall']:.4f}, kept "
                    f"{t['kept_share']:.4f}, ranked first {t['true_pairs_ranked_first']}, p>=0.5 {t['true_p_ge_0.5']}, "
                    f"by basis {t['true_by_basis']}")
    hist = []
    for part in ("person", "business"):
        h = scored[part][3]["hist"]
        for i, b in enumerate(BASIS_ORDER):
            for j in range(len(P_BINS) - 1):
                if h[i, j]:
                    hist.append({"part": part, "basis": b, "p_from": P_BINS[j], "p_to": min(1.0, P_BINS[j + 1]),
                                 "pairs": int(h[i, j])})
    d["hist"] = pd.DataFrame(hist)
    if len(d["hist"]):
        log(d["hist"].pivot_table(index=["part", "basis"], columns="p_from", values="pairs", aggfunc="sum",
                                  fill_value=0).to_string())
    top = pairs[(pairs["basis"] == "name_only") & (pairs["veto"] == "")].sort_values("p", ascending=False).head(20)
    d["top_name_only"] = top
    log("top name-only matches:")
    log(top[["extracted_name", "watchlist_name", "watchlist_record_id", "p", "bits"]].to_string(index=False))
    d["scored_pairs"] = {part: scored[part][3]["scored"] for part in ("person", "business")}
    return d


def build_manifest(cfg, input_hash, ref, input_reports, pr, params, prior_df, sens, scored, diag, sw):
    rows = config_records(cfg)
    rows.append({"section": "version", "key": "matching_core", "value": CORE_VERSION, "source": "notebook"})
    rows += [{"section": "input", "key": f"{k}.sha256", "value": v, "source": "file"} for k, v in input_hash.items()]
    for rep in input_reports:
        for k in ("rows", "all_empty_columns", "added_columns", "passthrough_columns"):
            rows.append({"section": "input", "key": f"{rep['source']}.{k}", "value": json.dumps(rep[k]), "source": "load"})
    for _, r in ref["hash_checks"].iterrows():
        rows.append({"section": "reference", "key": r["file"], "value": r["actual"], "source": "SOURCES.md",
                     "note": r["status"]})
    rows.append({"section": "reference", "key": "rarity_source", "value": json.dumps(pr.reports["rarity_source"]),
                 "source": "Rarity"})
    for k in ("parties", "rows"):
        rows.append({"section": "breakdown", "key": k, "value": json.dumps(pr.reports[k]), "source": "breakdown"})
    for _, r in pr.reports["empty_rows_extracted"].iterrows():
        rows.append({"section": "data_quality", "key": "empty_row.extracted", "value": r["record_id"],
                     "source": "breakdown", "note": "no name, business name, TIN or clinic NPI: no party"})
    rows.append({"section": "data_quality", "key": "empty_rows.watchlist", "value": len(pr.reports["empty_rows_watchlist"]),
                 "source": "breakdown"})
    for _, r in pr.reports["invalid_values"].iterrows():
        rows.append({"section": "data_quality", "key": f"invalid.{r['type']}.{r['reason']}", "value": int(r["values"]),
                     "source": "normalize"})
    junk = pr.vi[pr.vi["junk"] & pr.vi["valid"]]
    for _, r in junk.iterrows():
        rows.append({"section": "data_quality", "key": f"junk.{r['type']}", "value": r["value"],
                     "source": "value index", "note": r["junk_reason"]})
    rows.append({"section": "data_quality", "key": "category_mismatch", "value": pr.reports["category_mismatch"],
                 "source": "category", "note": "given category disagrees with an inferred one"})
    if pr.reports["unmapped"] is not None:
        for _, r in pr.reports["unmapped"].iterrows():
            rows.append({"section": "unmapped", "key": f"{r['source_field']}:{r['value']}", "value": int(r["parties"]),
                         "source": "category_map.csv", "note": "-> unknown"})
    for part in ("person", "business"):
        for rep in scored[part][3]["report"]:
            rows.append({"section": "blocking", "key": f"{part}.{rep['rule']}", "value": rep["pairs_after_refine"],
                         "source": "key counts before indexing", "pairs": rep["pairs_before_refine"],
                         "note": (f"oversized keys {rep['oversized_keys']}, dropped {rep['dropped_keys']} "
                                  f"({rep['dropped_pairs']} pairs) {rep['dropped_examples']}").strip()})
        for k, v in scored[part][3]["rule_counts"].items():
            rows.append({"section": "blocking.proposed", "key": f"{part}.{k}", "value": v, "source": "indexing"})
        rows.append({"section": "blocking.proposed", "key": f"{part}.scored_pairs", "value": scored[part][3]["scored"],
                     "source": "indexing"})
    W = pd.concat([params["weights"]["person"], params["weights"]["business"]], ignore_index=True)
    for _, r in W.iterrows():
        k = f"{r['part']}.{r['field']}.{r['level']}"
        rows.append({"section": "m", "key": k, "value": r["m"], "source": r["m_chain"], "pairs": r["m_pairs"],
                     "ci_low": r["m_lo"], "ci_high": r["m_hi"], "note": r["flag"]})
        rows.append({"section": "u", "key": k, "value": r["u"], "source": r["u_source"], "pairs": r["u_pairs"],
                     "note": ("value-specific on this level: " + r["u_value_specific"]) if r["u_value_specific"] else ""})
    for k, v in (params["components"] or {}).items():
        rows.append({"section": "u.name_components", "key": k, "value": v, "source": "random pairs"})
    for s in params["sources"]:
        rows.append({"section": "m.sources", "key": f"{s['part']}.{s['source']}", "value": s["pairs"], "source": "pairs"})
    for part, e in params["em"].items():
        rows.append({"section": "m.em", "key": part, "value": json.dumps(e), "source": "EM, u fixed"})
    for _, r in prior_df.iterrows():
        rows.append({"section": "prior", "key": f"{r['part']}.{r['group']}", "value": r["prior"],
                     "source": f"strict pairs {r['strict_pairs']} / recall {r['recall']:.3f} / all pairs {r['all_pairs']}",
                     "pairs": r["strict_pairs"], "note": r["flag"]})
    for _, r in sens.iterrows():
        rows.append({"section": "sensitivity", "key": f"{r['part']}.{r['group']}.prior_x{r['prior_multiplier']}.recall_x{r['recall_multiplier']}",
                     "value": json.dumps({k: r[k] for k in r.index if k.startswith("entities")}), "source": f"prior {r['prior']:.3g}"})
    for a in diag["anchor_recall"]:
        rows.append({"section": "diagnostics", "key": f"identifier_anchor_recall.{a['part']}", "value": a["recall"],
                     "pairs": a["identifier_anchor_pairs"], "source": "name rules only"})
    for t in diag["truth"]:
        rows.append({"section": "diagnostics", "key": f"truth.{t['part']}", "value": json.dumps(t, default=str),
                     "source": "truth file (diagnostics only)"})
    vet = 0
    for part in ("person", "business"):
        S = scored[part][0]
        if len(S):
            for v, n in S.loc[S["veto"] != "", "veto"].value_counts().items():
                rows.append({"section": "vetoes", "key": f"{part}.{v}", "value": int(n), "source": "score"})
    for r in sw.rows:
        rows.append({"section": "timing", "key": r["step"], "value": r["seconds"], "source": "wall seconds",
                     "note": f"peak {r['peak_rss_gb']} GB"})
    m = pd.DataFrame(rows)
    for c in ("section", "key", "value", "source", "pairs", "ci_low", "ci_high", "note"):
        if c not in m:
            m[c] = ""
    m = m[["section", "key", "value", "source", "pairs", "ci_low", "ci_high", "note"]].fillna("")
    m["value"] = m["value"].map(lambda v: v if isinstance(v, (int, float, np.integer, np.floating)) else str(v))
    return m

### Edge-case check

Every hand-written case in `edge_cases()` against the run's candidates and entities: the
expected pair is visible, with the expected basis, veto, rank and levels, and the party carries
the expected category fields. Used by the diagnostics and by the self-tests.

In [25]:
def check_edge_cases(pr, pairs, levels_by_part, scored, entities, edges):
    """One row per expectation: ok / failed with the reason."""
    rows = []
    ent = entities.set_index(["record_id", "part"])
    empty = set(pr.reports["empty_rows_extracted"]["record_id"])
    for c in edges:
        for ex in c["expect"]:
            problems = []
            if ex.get("empty_row"):
                if ex["x"] not in empty:
                    problems.append("not reported as an empty row")
                rows.append({"case": c["case"], "desc": c["desc"], "x": ex["x"], "part": ex["part"], "w": "",
                             "ok": not problems, "problems": "; ".join(problems)})
                continue
            part = ex["part"]
            if ex.get("party"):
                if (ex["x"], part) not in ent.index:
                    problems.append("party missing")
                else:
                    e = ent.loc[(ex["x"], part)]
                    for k, v in ex["party"].items():
                        col = {"category": "category_used"}.get(k, k)
                        got = e[col]
                        if bool(got == v) is False:
                            problems.append(f"{k}={got!r}, expected {v!r}")
            if ex.get("w"):
                P = pairs[(pairs["part"] == part) & (pairs["extracted_record_id"] == ex["x"]) &
                          (pairs["watchlist_record_id"] == ex["w"])]
                if not len(P):
                    if not ex.get("if_visible"):      # a strong negative may fall below the keep rule
                        problems.append("pair not visible")
                else:
                    pr_ = P.iloc[0]
                    if "basis" in ex and pr_["basis"] != ex["basis"]:
                        problems.append(f"basis {pr_['basis']}, expected {ex['basis']}")
                    if "basis_not" in ex and pr_["basis"] == ex["basis_not"]:
                        problems.append(f"basis must not be {ex['basis_not']}")
                    if "veto" in ex and pr_["veto"] != ex["veto"]:
                        problems.append(f"veto {pr_['veto']!r}, expected {ex['veto']!r}")
                    if ex.get("veto") and pr_["p"] != 0:
                        problems.append("vetoed pair must have p = 0")
                    if ex.get("rank") and pr_["rank"] != ex["rank"]:
                        problems.append(f"rank {pr_['rank']}, expected {ex['rank']}")
                    if ex.get("levels"):
                        S, L = scored[part][0], scored[part][1]
                        m = (S["l"].to_numpy() == pr_["_l"]) & (S["r"].to_numpy() == pr_["_r"])
                        lv = L[m].iloc[0]
                        for f, want in ex["levels"].items():
                            code = int(lv[f]) if f in lv else EMPTY
                            got = "empty" if code < 0 else FIELD_LEVELS[f][code]
                            if got != want:
                                problems.append(f"{f} level {got}, expected {want}")
            rows.append({"case": c["case"], "desc": c["desc"], "x": ex["x"], "part": part, "w": ex.get("w", ""),
                         "ok": not problems, "problems": "; ".join(problems)})
    return pd.DataFrame(rows)

### The whole pipeline as one function

`run_pipeline` chains steps 2-16 on two validated input frames and returns every table. The
Run section below calls the steps one by one so each prints its own output; the self-tests
call this function on small inputs.

In [26]:
def run_pipeline(xdf, wdf, maps, ref, cfg, truth_file=None, log=None):
    log = log or (lambda *a, **k: None)
    pr = prepare(xdf, wdf, maps, ref, cfg, log=log)
    params = estimate_parameters(pr, maps, ref, cfg, wdf, log=log)
    prior, prior_df = estimate_prior(pr, params, cfg, log=log)
    scored = score_all(pr, params, prior, cfg, log=log)
    pairs = pd.concat([pair_table(pr, part, scored[part][0]) for part in ("business", "person")], ignore_index=True)
    entities = rollup_entities(pr, pairs, {part: scored[part][2] for part in ("person", "business")}, cfg)
    claims = rollup_claims(entities, xdf.groupby("claim_id").size())
    evidence = pd.concat([evidence_for(pr, part, pair_table(pr, part, scored[part][0]), scored[part][1],
                                       scored[part][0], params["weights"][part])
                          for part in ("business", "person") if len(scored[part][0])], ignore_index=True)
    diag = diagnostics(pr, scored, pairs, cfg, truth_file, log=log)
    return {"pr": pr, "params": params, "prior": prior, "prior_df": prior_df, "scored": scored, "pairs": pairs,
            "entities": entities, "claims": claims, "evidence": evidence, "diag": diag}

## Run

The cells below run the steps in order on the dataset chosen in Setup. Each prints what it
did; the timing and peak memory of every step go to the manifest.

In [27]:
CFG = make_config()
PATHS = CFG.paths
SW = Stopwatch()
print(f"dataset: {CFG.dataset}   root: {PATHS['root']}   out: {PATHS['out']}")
print(f"python {platform.python_version()}  pandas {pd.__version__}  numpy {np.__version__}  "
      f"recordlinkage {rl.__version__}  matching core v{CORE_VERSION}")
MAPS = load_mappings(PATHS["mappings"])
REF = load_reference(PATHS["reference"], CFG.core)
print(REF["hash_checks"][["file", "status"]].to_string(index=False))
SW.mark("setup: mappings and reference tables")

dataset: synthetic   root: C:\Users\yalov\oko0\.claude\worktrees\agent-a2023359df052efff\project_v0.2\record-linkage   out: C:\Users\yalov\oko0\.claude\worktrees\agent-a2023359df052efff\project_v0.2\record-linkage\out\synthetic
python 3.13.3  pandas 3.0.6  numpy 2.5.2  recordlinkage 0.16  matching core v1.0


                 file status
   first_names.csv.gz     ok
        nicknames.csv     ok
nicknames_LICENSE.txt     ok
    org_tokens.csv.gz     ok
  reference_meta.json     ok
      surnames.csv.gz     ok
[    0.5s  peak  0.23 GB] setup: mappings and reference tables


### Inputs for this run

`synthetic` and `scale` generate their files into `data/` (with a truth file beside them);
`leie` exports the raw LEIE into `data/leie_watchlist.csv` and builds the extracted test set;
`files` reads `RL_EXTRACTED` and `RL_WATCHLIST`. The truth file is read only by the
diagnostics and self-tests.

In [28]:
def dataset_files(cfg, maps, ref):
    d = cfg.paths["data"]
    ds = cfg.dataset
    if ds == "files":
        return Path(os.environ["RL_EXTRACTED"]), Path(os.environ["RL_WATCHLIST"]), None
    xf, wf, tf = d / f"{ds}_extracted.csv", d / f"{ds}_watchlist.csv", d / f"{ds}_truth.csv"
    if ds == "synthetic" and not (xf.exists() and wf.exists() and tf.exists()):
        X, W, truth, _, _ = make_dataset(cfg.synth_persons, cfg.synth_businesses, 5000, ref,
                                         maps["simulation_noise"], cfg.seed)
        X.to_csv(xf, index=False); W.to_csv(wf, index=False); truth.to_csv(tf, index=False)
    elif ds == "leie" and not (xf.exists() and wf.exists() and tf.exists()):
        raw = d / "LEIE_UPDATED.csv"
        if not raw.exists():
            import urllib.request
            req = urllib.request.Request(LEIE_URL, headers={"User-Agent": "record-linkage/1.0"})
            raw.write_bytes(urllib.request.urlopen(req, timeout=300).read())
        W = export_leie(raw, wf)
        X, truth, _ = leie_test_set(W, ref, maps["simulation_noise"], cfg.seed, cfg.leie_noisy_copies,
                                    cfg.leie_fictional)
        X.to_csv(xf, index=False); truth.to_csv(tf, index=False)
    elif ds == "scale" and not (xf.exists() and wf.exists() and tf.exists()):
        X, W, truth = make_scale_set(cfg, maps, ref)
        X.to_csv(xf, index=False); W.to_csv(wf, index=False); truth.to_csv(tf, index=False)
    return xf, wf, tf


XF, WF, TF = dataset_files(CFG, MAPS, REF)
SW.mark("inputs ready")
XDF, XREP = load_input(XF, "extracted")
WDF, WREP = load_input(WF, "watchlist")
INPUT_HASH = {"extracted": sha256_file(XF), "watchlist": sha256_file(WF)}
for rep in (XREP, WREP):
    print(f"{rep['source']}: {rep['rows']:,} rows; all-empty columns: {rep['all_empty_columns'] or 'none'}; "
          f"passthrough: {rep['passthrough_columns'] or 'none'}")
SW.mark("1 load and validate")

[    0.5s  peak  0.23 GB] inputs ready
extracted: 2,937 rows; all-empty columns: ['street_direction']; passthrough: none
watchlist: 5,047 rows; all-empty columns: ['claim_id', 'note_id', 'category', 'street_direction']; passthrough: none
[    0.7s  peak  0.24 GB] 1 load and validate


### Steps 2-6: normalize, breakdown, value index, category, rarity

In [29]:
PR = prepare(XDF, WDF, MAPS, REF, CFG)
print(f"category disagreements (given vs inferred): {PR.reports['category_mismatch']}")
if PR.reports["unmapped"] is not None and len(PR.reports["unmapped"]):
    print(f"unmapped watchlist specialty / licence values (-> unknown): {len(PR.reports['unmapped'])}")
    print(PR.reports["unmapped"].head(10).to_string(index=False))
SW.mark("2-6 normalize, breakdown, value index, category, rarity")

parties: {'extracted_person': 2527, 'extracted_business': 646, 'watchlist_person': 4318, 'watchlist_business': 850}  empty rows: extracted 1, watchlist 0
value index: 22,982 values, 21,985 single-holder, 69 junk; junk values blanked in 0 party fields
rarity: {'surnames': 'census2010', 'first_names': 'ssa1930-2005', 'org_tokens': 'nppes+reference_population'}; nicknames: 2827 rows; declared d/b/a pairs: 73
category disagreements (given vs inferred): 105
unmapped watchlist specialty / licence values (-> unknown): 3
source_field     value  parties
   specialty   NURSING      105
   specialty DENTISTRY       96
   specialty PHYSICIAN       78
[    3.2s  peak  0.28 GB] 2-6 normalize, breakdown, value index, category, rarity


### Steps 8-11: u, m, prior

In [30]:
PARAMS = estimate_parameters(PR, MAPS, REF, CFG, WDF)
SW.mark("8-10 u and m")
PRIOR, PRIOR_DF = estimate_prior(PR, PARAMS, CFG)
WEIGHTS = pd.concat([PARAMS["weights"]["person"], PARAMS["weights"]["business"]], ignore_index=True)
show = WEIGHTS[["part", "field", "level", "m", "m_source", "m_pairs", "u", "bits_field_level", "flag"]]
print(show.to_string(index=False, max_rows=200, float_format=lambda v: f"{v:.4g}"))
SW.mark("11 prior")

u[person]: 796,407 random pairs (14s)


u[business]: 203,592 random pairs (5s)


simulation: 5,047 noisy copies compared (3s)


m[person]: strict anchors 220, loose anchors 247 (EM lambda 1.000, 8 iterations; 1s), watchlist duplicates 117, simulated 4,318


m[business]: strict anchors 70, loose anchors 73 (EM lambda 1.000, 164 iterations; 1s), watchlist duplicates 14, simulated 850
[   25.4s  peak  0.79 GB] 8-10 u and m


    part        group  extracted_parties  strict_pairs   recall    prior flag
  person professional                796           449 0.629537 0.000208     
  person     business                 51            26 0.467013 0.000253     
  person      private               1065           420 0.472031 0.000193     
  person      unknown                615           219 0.473074 0.000174     
business professional                209           135 0.787417 0.000965     
business     business                437           150 0.538503 0.000750     
    part    field                   level         m             m_source  m_pairs         u  bits_field_level                         flag
  person      dob                   exact    0.8627       anchors_strict       95 0.0002064             12.03                             
  person      dob            swap_or_typo    0.1368       anchors_strict       95  0.001481              6.53                             
  person      dob              year_m

### Steps 7, 12: candidates and scoring (chunked)

In [31]:
SCORED = score_all(PR, PARAMS, PRIOR, CFG)
SW.mark("7, 12 candidates, comparisons, scoring")

  business tin                    pairs          112
  business cnpi                   pairs          127
  business email                  pairs           13
  business phone                  pairs           51
  business rare_word              pairs          890
  business lead_word_state        pairs          520
  business sn_sorted_name         pairs          429
  business street                 pairs          204


  business: 3,137 pairs scored, 2,654 kept
  co-party anchors (business links on an identifier, p >= 0.9): 216


  person   ssn                    pairs          118
  person   npi                    pairs          297
  person   dl                     pairs           63
  person   license                pairs           89
  person   email                  pairs           87
  person   vin                    pairs           25
  person   plate                  pairs           25
  person   phone                  pairs          288
  person   nysiis_initial         pairs        2,550
  person   nysiis_canon_initial   pairs        3,110
  person   nysiis_first_missing   pairs          175
  person   dob_initial            pairs          691
  person   swapped                pairs           22
  person   sn_last_first          pairs        1,431
  person   street                 pairs          666


  person: 10,911 pairs scored, 8,891 kept
[   27.6s  peak  0.79 GB] 7, 12 candidates, comparisons, scoring


### Steps 13-14: roll-ups and evidence

In [32]:
PAIRS = pd.concat([pair_table(PR, part, SCORED[part][0]) for part in ("business", "person")], ignore_index=True)
ENTITIES = rollup_entities(PR, PAIRS, {part: SCORED[part][2] for part in ("person", "business")}, CFG)
CLAIMS = rollup_claims(ENTITIES, XDF.groupby("claim_id").size())
EVIDENCE = pd.concat([evidence_for(PR, part, pair_table(PR, part, SCORED[part][0]), SCORED[part][1],
                                   SCORED[part][0], PARAMS["weights"][part])
                      for part in ("business", "person") if len(SCORED[part][0])], ignore_index=True)
_best = PAIRS[PAIRS["veto"] == ""].sort_values(["part", "_l", "p", "bits"], ascending=[True, True, False, False]) \
    .groupby(["part", "_l"]).head(1)
_best = _best.merge(ENTITIES[["party_id", "prior_group"]],
                    left_on=_best["pair_id"].str.split("~").str[0], right_on="party_id", how="left")
SENS = sensitivity(_best, PRIOR_DF, CFG)
print(ENTITIES.groupby(["part", "basis"]).agg(entities=("p", "size"), p_ge_05=("p", lambda s: int((s >= 0.5).sum())),
                                              p_ge_09=("p", lambda s: int((s >= 0.9).sum()))).to_string())
SW.mark("13-14 roll-ups and evidence")

                     entities  p_ge_05  p_ge_09
part     basis                                 
business address           86       80       79
         contextual        93       92       88
         identifier       204      201      201
         name_only         79       21       18
         none             184        0        0
person   address          364      357      353
         co_party           7        7        6
         contextual       247      211      209
         dob              241      240      240
         identifier       649      644      643
         name_only        422       54       47
         none             597        3        3
[   28.6s  peak  0.79 GB] 13-14 roll-ups and evidence


### Step 16: diagnostics

Blocking recall on identifier anchors (extracted x watchlist pairs sharing a valid
single-holder identifier: how many would the name rules alone have proposed?); on datasets
with a truth file, recall of the true pairs; the p histogram by basis; the 20 strongest
name-only matches.

In [33]:
DIAG = diagnostics(PR, SCORED, PAIRS, CFG, TF)
SW.mark("16 diagnostics")

blocking recall on identifier anchors [person]: 510 of 510 also proposed by a name rule (1.000)
blocking recall on identifier anchors [business]: 209 of 209 also proposed by a name rule (1.000)


truth [person]: 1720 true pairs, proposed 0.9901, kept 0.9901, ranked first 1551, p>=0.5 1680, by basis {'identifier': 705, 'address': 378, 'dob': 279, 'contextual': 249, 'name_only': 75, 'none': 10, 'co_party': 7}
truth [business]: 432 true pairs, proposed 1.0000, kept 1.0000, ranked first 407, p>=0.5 426, by basis {'identifier': 218, 'contextual': 111, 'address': 80, 'name_only': 22, 'none': 1}
p_from               0.000  0.001  0.010  0.100  0.500  0.800  0.900  0.990
part     basis                                                             
business address        11      7      0      1      0      1      0     80
         contextual      1      0      3     11      1      3      2    105
         identifier      0      1      0      2      0      0      1    215
         name_only     149     41     21      5      1      4      4     14
         none         1756      6      1      0      0      0      0      0
person   address         7      2      2      0      3      2      4

### Step 15: workbook and manifest

In [34]:
MANIFEST = build_manifest(CFG, INPUT_HASH, REF, (XREP, WREP), PR, PARAMS, PRIOR_DF, SENS, SCORED, DIAG, SW)
CAND_OUT = PAIRS.drop(columns=["_l", "_r"])
TABLES = {"entities": ENTITIES, "candidates": CAND_OUT,
          "evidence": EVIDENCE if CFG.evidence_in_workbook else EVIDENCE.iloc[:0],
          "claims": CLAIMS, "manifest": MANIFEST}
save_tables(PATHS["out"], {"entities": ENTITIES, "candidates": CAND_OUT, "evidence": EVIDENCE, "claims": CLAIMS,
                           "manifest": MANIFEST}, MANIFEST)
WB_PATH = PATHS["out"] / f"record_linkage_{CFG.dataset}.xlsx"
WRITTEN = write_workbook(WB_PATH, TABLES, CFG.excel_row_limit)
SW.mark("15 workbook")
print(f"workbook: {WB_PATH}  sheets: {sum(len(v) for v in WRITTEN.values())} "
      f"({', '.join(f'{k}: {len(v)}' for k, v in WRITTEN.items())})")
print(f"total {SW.rows[-1]['elapsed']:.1f}s, peak memory {SW.peak / 1e9:.2f} GB")
(PATHS["out"] / "timing.json").write_text(json.dumps(SW.rows, indent=1), encoding="utf-8")

[   37.5s  peak  0.79 GB] 15 workbook
workbook: C:\Users\yalov\oko0\.claude\worktrees\agent-a2023359df052efff\project_v0.2\record-linkage\out\synthetic\record_linkage_synthetic.xlsx  sheets: 5 (entities: 1, candidates: 1, evidence: 1, claims: 1, manifest: 1)
total 37.5s, peak memory 0.79 GB


1060

## 17 · Self-tests

`unittest` cases for every part of the package (normalizers, breakdown, category, candidates,
comparison levels, score invariants, parameters, output, the LEIE exporter), an end-to-end
run on a small synthetic set with its hand-written edge cases, and the hash check of the
matching core. They run on their own small inputs, so they are the same for every dataset.

**Where Splink would do better.** Splink has a large test suite across DuckDB, Spark and
SQLite backends; these tests cover this package only.

In [35]:
import unittest, tempfile, io

_T = {}


def _env():
    """Mappings, reference tables and a small configuration, loaded once."""
    if "cfg" not in _T:
        cfg = make_config("selftest", random_pairs=150_000, sim_records=4_000, chunk_size=700)
        _T["cfg"] = cfg
        _T["maps"] = load_mappings(cfg.paths["mappings"])
        _T["ref"] = load_reference(cfg.paths["reference"], cfg.core)
    return _T["cfg"], _T["maps"], _T["ref"]


def _small_run():
    """One end-to-end run on a small synthetic set with every edge case (cached)."""
    if "run" not in _T:
        cfg, maps, ref = _env()
        X, W, truth, edges, _ = make_dataset(400, 100, 1200, ref, maps["simulation_noise"], cfg.seed)
        X, _ = validate_input(X, "X"); W, _ = validate_input(W, "W")
        tf = cfg.paths["out"] / "selftest_truth.csv"
        truth.to_csv(tf, index=False)
        _T["run"] = run_pipeline(X, W, maps, ref, cfg, truth_file=tf)
        _T["inputs"] = (X, W, truth, edges)
    return _T["run"]


def _tframe(rows):
    df = pd.DataFrame([blank_row(r["record_id"], **{k: v for k, v in r.items() if k != "record_id"})
                       for r in rows], columns=SCHEMA)
    return validate_input(df, "t")[0]


def _parties(rows, source="X", watchlist=False):
    cfg, maps, ref = _env()
    nick = Nicknames(ref["nicknames"])
    return rows_to_parties(_tframe(rows), source, maps, nick, cfg.core, watchlist)


class TestNormalizers(unittest.TestCase):
    def test_ssn(self):
        self.assertEqual(norm_ssn("212-45-6781"), ("212456781", True, ""))
        for bad in ("000-12-3456", "666123456", "912345678", "123456789", "111111111", "12345"):
            self.assertFalse(norm_ssn(bad)[1], bad)

    def test_tin(self):
        self.assertTrue(norm_tin("45-1234567")[1])
        self.assertFalse(norm_tin("07-1234567")[1])
        self.assertFalse(norm_tin("000000000")[1])

    def test_npi_luhn(self):
        self.assertTrue(npi_luhn_ok("1234567893"))
        self.assertFalse(npi_luhn_ok("1234567890"))
        self.assertEqual(norm_npi("0000000000"), ("", False, ""))
        self.assertEqual(norm_npi("1234567890")[2], "check_digit")
        self.assertTrue(npi_luhn_ok(luhn_npi("123456789")))

    def test_phone_email(self):
        self.assertEqual(norm_phone("1 (718) 392-4411")[0], "7183924411")
        self.assertFalse(norm_phone("2125550123")[1])
        self.assertFalse(norm_phone("0123456789")[1])
        self.assertEqual(norm_email(" A.B@Mail.Test ")[0], "a.b@mail.test")
        self.assertFalse(norm_email("none@none.com")[1])
        self.assertFalse(norm_email("not an email")[1])

    def test_vin(self):
        self.assertTrue(norm_vin("1M8GDM9AXKP042788")[1])
        self.assertEqual(norm_vin("1M8GDM9AYKP042788")[2], "check_digit")
        self.assertEqual(norm_vin("1M8GDM9AXKP04278I")[2], "format")

    def test_dob(self):
        self.assertEqual(norm_dob("3/4/1961")[0], "1961-03-04")
        self.assertEqual(norm_dob("19610304")[0], "1961-03-04")
        self.assertEqual(norm_dob("1900-01-01")[2], "placeholder")
        self.assertEqual(norm_dob("02/30/1960")[2], "impossible")
        self.assertEqual(norm_dob("2090-01-01")[2], "range")

    def test_qualified_ids(self):
        self.assertEqual(norm_dl("d123-456", "New York")[0], "NY:D123456")
        self.assertEqual(norm_plate("abc 123", "")[0], ":ABC123")
        self.assertFalse(norm_license("00000", "NY")[1])

    def test_address(self):
        self.assertEqual(parse_street_line("2161 UNIVERSITY AVENUE W, STE 5"), ("2161", "", "UNIVERSITY", "AVE", "5"))
        self.assertEqual(parse_street_line("P O BOX 2161")[:3], ("2161", "", "PO BOX"))
        self.assertEqual(parse_street_line("222 N W 45TH AVE")[1], "NW")
        self.assertEqual(norm_street_parts("12", "", "Main Street", "", "Apt 4B"), ("12", "", "MAIN", "ST", "4B"))
        self.assertEqual(norm_state("new york"), "NY")
        self.assertEqual(norm_zip("2134"), "02134")
        self.assertEqual(norm_city("St. Louis"), "SAINT LOUIS")

    def test_person_names(self):
        self.assertEqual(clean_person("Dr. John", "a.", "Smith Jr."), ("JOHN", "A", "SMITH"))
        self.assertEqual(clean_person("", "", "Moy, Marvin"), ("MARVIN", "", "MOY"))
        self.assertEqual(clean_person("William A Weiner", "", ""), ("WILLIAM", "A", "WEINER"))
        self.assertEqual(clean_person("José", "", "O'Brien"), ("JOSE", "", "OBRIEN"))
        self.assertEqual(parse_person("WILLIAM A. WEINER, D.O.")["creds"], ["DO"])

    def test_org_aliases(self):
        self.assertEqual(org_aliases("Rutland Medical P.C. d/b/a Soul Radiology Medical Imaging"),
                         [["RUTLAND", "MEDICAL"], ["SOUL", "RADIOLOGY", "MEDICAL", "IMAGING"]])
        self.assertEqual(org_aliases("AMERICAN TRANSIT INS. CO."), [["AMERICAN", "TRANSIT", "INSURANCE"]])
        self.assertEqual(org_aliases("Smith & Jones LLP"), [["SMITH", "JONES"]])
        self.assertEqual(org_alias_string("Nexray Medical Imaging, P.C."), "NEXRAY MEDICAL IMAGING")

    def test_nicknames(self):
        _, _, ref = _env()
        n = Nicknames(ref["nicknames"])
        self.assertTrue(n.roots("Bill") & n.roots("William"))
        self.assertFalse(n.roots("Robert") & n.roots("William"))
        self.assertEqual(nysiis("Weiner"), nysiis("Wiener"))

    def test_near(self):
        self.assertTrue(_near("123456789", "123456780"))
        self.assertTrue(_near("123456789", "123457689"))
        self.assertFalse(_near("123456789", "123456798x"))
        self.assertFalse(_near("123456789", "123450780"))


class TestBreakdown(unittest.TestCase):
    def test_person_and_business_tied(self):
        rows, parties, details, ties, empty, _ = _parties([
            {"record_id": "r1", "first_name": "Ann", "last_name": "Lee", "business_name": "Lee Chiropractic PC",
             "tin": "45-1234567", "work_phone": "7183924411", "home_phone": "7183924412", "email": "ann@lee.test",
             "street_number": "5", "street_name": "Main", "street_type": "St", "zip": "11211", "state": "NY"}])
        self.assertEqual(sorted(parties["part"]), ["business", "person"])
        self.assertEqual(len(ties), 1)
        P = parties[parties["part"] == "person"].iloc[0]
        B = parties[parties["part"] == "business"].iloc[0]
        self.assertEqual(P["tie"], B["party_id"])
        self.assertEqual((P["id_email"], B["id_email"]), ("ann@lee.test", ""))
        self.assertEqual((P["phone_own"], P["phone_row"], B["phone_row"]), ("7183924412", "7183924411", "7183924411"))
        self.assertEqual(B["id_tin"], "451234567")
        self.assertEqual(P["addr_street"], B["addr_street"])
        own = details.set_index(["party_id", "type", "value"])["ownership"]
        self.assertEqual(own[(P["party_id"], "phone", "7183924411")], "row")
        self.assertEqual(own[(P["party_id"], "phone", "7183924412")], "own")

    def test_tin_only_and_empty(self):
        _, parties, _, _, empty, _ = _parties([{"record_id": "t", "tin": "451234567"},
                                               {"record_id": "e", "city": "Brooklyn"},
                                               {"record_id": "b", "business_name": "Acme Towing", "email": "x@acme.test"}])
        self.assertEqual(list(parties["record_id"]), ["t", "b"])
        self.assertEqual(list(empty["record_id"]), ["e"])
        self.assertEqual(parties.set_index("record_id").loc["b", "id_email"], "x@acme.test")

    def test_invalid_values_kept_visible_not_compared(self):
        _, parties, details, _, _, _ = _parties([{"record_id": "r", "first_name": "A", "last_name": "B",
                                                   "provider_npi": "1234567890", "ssn": "123456789"}])
        self.assertEqual(parties.iloc[0]["id_npi"], "")
        d = details.set_index("type")
        self.assertFalse(bool(d.loc["npi", "valid"]))
        self.assertEqual(d.loc["ssn", "reason"], "placeholder")

    def test_value_index(self):
        _, p1, d1, _, _, _ = _parties([{"record_id": "a", "first_name": "Ann", "last_name": "Lee", "ssn": "212456781"},
                                       {"record_id": "b", "first_name": "A", "last_name": "Lee", "ssn": "212456781"},
                                       {"record_id": "c", "first_name": "Bo", "last_name": "Kim", "ssn": "313456781"},
                                       {"record_id": "d", "first_name": "Cy", "last_name": "Poe", "ssn": "313456781"}])
        vi = mark_junk(value_index_fast(p1, d1), CoreParams(junk_holders=1)).set_index(["type", "value"])
        self.assertTrue(bool(vi.loc[("ssn", "212456781"), "single_holder"]))      # Ann Lee = A. Lee
        self.assertFalse(bool(vi.loc[("ssn", "313456781"), "single_holder"]))
        self.assertTrue(bool(vi.loc[("ssn", "313456781"), "junk"]))


class TestCategory(unittest.TestCase):
    def test_extracted_precedence(self):
        cfg, maps, _ = _env()
        _, p, _, _, _, _ = _parties([
            {"record_id": "npi", "category": "legal", "first_name": "A", "last_name": "B", "provider_npi": luhn_npi("123456789")},
            {"record_id": "giv", "category": "witness", "first_name": "C", "last_name": "D", "business_name": "D Auto Body"},
            {"record_id": "kw", "business_name": "Pemberton & Vasquez LLP"},
            {"record_id": "oth", "category": "other", "first_name": "E", "last_name": "F"},
            {"record_id": "lic", "first_name": "G", "last_name": "H", "professional_license_type": "DC"}])
        e = p.set_index(["record_id", "part"])
        self.assertEqual(e.loc[("npi", "person"), "category"], "medical")
        self.assertTrue(bool(e.loc[("npi", "person"), "category_mismatch"]))
        self.assertEqual(e.loc[("giv", "person"), "category"], "witness")          # given before keyword
        self.assertEqual(e.loc[("giv", "person"), "category_inferred"], "repair shop")
        self.assertEqual(e.loc[("giv", "person"), "prior_group"], "private")
        self.assertEqual(e.loc[("giv", "business"), "prior_group"], "business")
        self.assertEqual(e.loc[("kw", "business"), "category"], "legal")
        self.assertEqual(e.loc[("oth", "person"), "category"], "")                 # other = no information
        self.assertEqual(e.loc[("oth", "person"), "prior_group"], "unknown")
        self.assertEqual(e.loc[("lic", "person"), "cat_strength"], "strong")
        self.assertNotIn("witness", set(p["category_inferred"]))

    def test_watchlist_mapping_and_unmapped(self):
        _, p, _, _, _, unm = _parties([
            {"record_id": "w1", "first_name": "A", "last_name": "B", "provider_specialty": "CHIROPRACTIC"},
            {"record_id": "w2", "first_name": "C", "last_name": "D", "professional_license_type": "LAW PRACTICE"},
            {"record_id": "w3", "first_name": "E", "last_name": "F", "provider_specialty": "BASKET WEAVING"}],
            source="W", watchlist=True)
        self.assertEqual(list(p["category"]), ["medical", "legal", ""])
        self.assertIn("BASKET WEAVING", set(unm["value"]))


def _mini_ctx(xp, wp):
    cfg, maps, ref = _env()
    nick = Nicknames(ref["nicknames"])
    rar = Rarity(ref["surnames"], ref["first_names"], ref["org_tokens"], ref["meta"], cfg.core)
    stats = build_value_stats(xp, wp)
    dba = declared_dba_pairs(pd.concat([xp["org_aliases"], wp["org_aliases"]]))
    return CompareContext(cfg.core, rar, nick, stats, dba,
                          {"first_nick_or_close": 0.01, "first_initial": 0.05, "first_empty": 0.05,
                           "first_differs": 0.9, "last_close": 0.002})


def _pair_levels(xrow, wrow, part="person"):
    _, xp, _, _, _, _ = _parties([xrow])
    _, wp, _, _, _, _ = _parties([wrow], source="W", watchlist=True)
    X, W = split_parts(xp)[part], split_parts(wp)[part]
    ctx = _mini_ctx(xp, wp)
    lv = compare_pairs(X, W, [0], [0], part, ctx)
    return lv.iloc[0], ctx


def _lev(lv, f):
    c = int(lv[f])
    return "empty" if c < 0 else FIELD_LEVELS[f][c]


class TestComparisons(unittest.TestCase):
    def test_name_levels(self):
        cases = [(("John", "Smith"), ("John", "Smith"), "exact"),
                 (("Bill", "Szczepanski"), ("William", "Szczepanski"), "first_nick_or_close"),
                 (("Maria", "Garcia-Villanueva"), ("Maria", "Garcia"), "last_close_first_agrees"),
                 (("R", "Featherstone"), ("Rupert", "Featherstone"), "initial_agrees"),
                 (("Chidi", "Nwachukwu"), ("Nwachukwu", "Chidi"), "swapped"),
                 (("", "Webb"), ("Henry", "Webb"), "first_empty"),
                 (("Alan", "Webb"), ("Henry", "Webb"), "first_differs"),
                 (("Alan", "Webb"), ("Henry", "Moss"), "else")]
        for (f1, l1), (f2, l2), want in cases:
            row = {"record_id": "x", "first_name": f1, "last_name": l1}
            if not l1:
                row["business_name"] = "placeholder biz"    # keep a party with no surname
                row["first_name"] = ""
                row["middle_name"] = ""
            lv, _ = _pair_levels(row, {"record_id": "w", "first_name": f2, "last_name": l2})
            got = _lev(lv, "name") if l1 else "empty"
            self.assertEqual(got, want, (f1, l1, f2, l2))

    def test_dob_levels(self):
        for a, b, want in (("1975-04-09", "1975-04-09", "exact"), ("1975-04-09", "1975-09-04", "swap_or_typo"),
                           ("1975-04-09", "1975-04-08", "swap_or_typo"), ("1975-04-09", "1975-04-21", "year_month"),
                           ("1975-04-09", "1975-11-21", "year"), ("1975-04-09", "1980-11-21", "differs"),
                           ("1975-04-09", "1900-01-01", "empty")):
            lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "dob": a},
                                 {"record_id": "w", "first_name": "A", "last_name": "B", "dob": b})
            self.assertEqual(_lev(lv, "dob"), want, (a, b))

    def test_address_levels(self):
        base = {"street_number": "5", "street_name": "Main", "street_type": "St", "unit": "2", "city": "Brooklyn",
                "state": "NY", "zip": "11211"}
        for change, want in (({}, "exact"), ({"unit": "3"}, "street"), ({"street_number": "7"}, "zip"),
                             ({"street_number": "7", "zip": "11222"}, "city_state"),
                             ({"street_number": "7", "zip": "11222", "city": "Albany"}, "state"),
                             ({"street_number": "7", "zip": "90026", "city": "Los Angeles", "state": "CA"}, "differs")):
            lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", **base},
                                 {"record_id": "w", "first_name": "A", "last_name": "B", **{**base, **change}})
            self.assertEqual(_lev(lv, "address"), want, change)

    def test_identifier_levels_and_veto(self):
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "ssn": "212456781"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "ssn": "212456781"})
        self.assertEqual(_lev(lv, "ssn"), "exact")
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "ssn": "212456781"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "ssn": "212456718"})
        self.assertEqual(_lev(lv, "ssn"), "near")
        self.assertFalse(bool(lv["veto_ssn"]))
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "ssn": "212456781"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "ssn": "313999222"})
        self.assertEqual(_lev(lv, "ssn"), "differs")
        self.assertTrue(bool(lv["veto_ssn"]))
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "driver_license_number": "D1234567", "driver_license_state": "NY"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "driver_license_number": "K9876543", "driver_license_state": "NJ"})
        self.assertFalse(bool(lv["veto_dl"]))                                       # two states: no veto
        lv, _ = _pair_levels({"record_id": "x", "business_name": "A Corp", "clinic_npi": luhn_npi("111111112")},
                             {"record_id": "w", "business_name": "A Corp", "clinic_npi": luhn_npi("222222223")}, "business")
        self.assertEqual(_lev(lv, "cnpi"), "differs")
        self.assertFalse(bool(lv["veto_cnpi"]))                                     # clinic NPI never vetoes

    def test_phone_levels(self):
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "home_phone": "7183924411"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "home_phone": "7183924411"})
        self.assertEqual(_lev(lv, "phone"), "exact_owned_single")
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "work_phone": "7183924411"},
                             {"record_id": "w", "first_name": "C", "last_name": "D", "work_phone": "7183924411"})
        self.assertEqual(_lev(lv, "phone"), "exact_shared")

    def test_org_levels(self):
        cases = [("Lakeshore Med Ctr", "Lakeshore Medical Center Inc", "exact"),
                 ("Kestrel Imaging LLC d/b/a Harborview Radiology", "Harborview Radiology", "exact"),
                 ("Pinecrest Orthopedic", "Pinecrest Orthopedic Rehabilitation", "short_form"),
                 ("Allport Indemnity Company", "Allport Casualty Surety Company", "sibling"),
                 ("Quixotic Holdings", "Unrelated Towing", "none")]
        for a, b, want in cases:
            lv, _ = _pair_levels({"record_id": "x", "business_name": a}, {"record_id": "w", "business_name": b}, "business")
            self.assertEqual(_lev(lv, "org"), want, (a, b))

    def test_spec_cat(self):
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "provider_specialty": "Chiropractor", "provider_npi": luhn_npi("123456789")},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "provider_specialty": "CHIROPRACTIC"})
        self.assertEqual(_lev(lv, "spec_cat"), "specialty")


class TestCandidates(unittest.TestCase):
    def setUp(self):
        run = _small_run()
        self.pr = run["pr"]
        self.cfg = _env()[0]

    def test_chunked_equals_unchunked(self):
        for part in ("person", "business"):
            X, W = self.pr.X[part], self.pr.W[part]
            plan = CandidatePlan(X, W, part, self.pr.ctx.rarity, self.cfg, self.pr.ctx.dba_pairs)
            one = plan.chunk(0, len(X)).sort_values(["l", "r"]).reset_index(drop=True)
            many = pd.concat([plan.chunk(lo, min(len(X), lo + 97)) for lo in range(0, len(X), 97)])
            many = many.sort_values(["l", "r"]).reset_index(drop=True)
            self.assertTrue(one.equals(many), part)

    def test_each_rule_proposes_its_target(self):
        X = _tframe([{"record_id": "x1", "first_name": "Bill", "last_name": "Kowalski"},
                    {"record_id": "x6", "first_name": "Rita", "last_name": "Oldname", "dob": "1970-01-02"},
                    {"record_id": "x2", "first_name": "Ann", "last_name": "Zyx", "ssn": "212456781"},
                    {"record_id": "x3", "first_name": "Mo", "last_name": "Qwe", "home_phone": "7183924411"},
                    {"record_id": "x4", "first_name": "Chidi", "last_name": "Nwachukwu"},
                    {"record_id": "x5", "last_name": "Plover", "dob": "1981-01-01"}])
        W = _tframe([{"record_id": "w1", "first_name": "William", "last_name": "Kowalski"},
                    {"record_id": "w6", "first_name": "Rita", "last_name": "Newname", "dob": "1970-01-02"},
                    {"record_id": "w2", "first_name": "Zed", "last_name": "Other", "ssn": "212456781"},
                    {"record_id": "w3", "first_name": "Zed", "last_name": "Else", "home_phone": "7183924411"},
                    {"record_id": "w4", "first_name": "Nwachukwu", "last_name": "Chidi"},
                    {"record_id": "w5", "first_name": "Ruth", "last_name": "Plover"}])
        cfg, maps, ref = _env()
        pr = prepare(X, W, maps, ref, cfg, log=lambda *a: None)
        plan = CandidatePlan(pr.X["person"], pr.W["person"], "person", pr.ctx.rarity, cfg)
        c = plan.chunk(0, len(pr.X["person"]))
        got = {(pr.X["person"]["record_id"][l], pr.W["person"]["record_id"][r]): set(n.split(","))
               for l, r, n in zip(c["l"], c["r"], plan.rule_names(c["rules"]))}
        self.assertIn("nysiis_canon_initial", got[("x1", "w1")])
        self.assertNotIn("nysiis_initial", got[("x1", "w1")])
        self.assertIn("dob_initial", got[("x6", "w6")])
        self.assertIn("ssn", got[("x2", "w2")])
        self.assertIn("phone", got[("x3", "w3")])
        self.assertIn("swapped", got[("x4", "w4")])
        self.assertIn("nysiis_first_missing", got[("x5", "w5")])

    def test_cross_file_same_part_only(self):
        S = _small_run()["scored"]
        for part in ("person", "business"):
            s = S[part][0]
            self.assertTrue((s["l"] < len(self.pr.X[part])).all() and (s["r"] < len(self.pr.W[part])).all())

    def test_strict_rules_subset_of_blocking(self):
        cfg = self.cfg
        for part in ("person", "business"):
            X, W = self.pr.X[part], self.pr.W[part]
            l, r = strict_pairs(X, W, part, self.pr.ctx)
            keys = self.pr and _small_run()["scored"][part][3]["keys"]
            k = l.astype(np.int64) * np.int64(1 << 32) + r
            self.assertTrue(np.isin(k, keys).all(), part)


class TestScoring(unittest.TestCase):
    def setUp(self):
        self.run = _small_run()

    def test_empty_fields_are_zero_bits(self):
        for part in ("person", "business"):
            S, L = self.run["scored"][part][0], self.run["scored"][part][1]
            for f in COMPARED_FIELDS[part]:
                self.assertTrue((S.loc[L[f].to_numpy() < 0, f"bits_{f}"] == 0).all(), (part, f))

    def test_evidence_sums_to_total(self):
        ev = self.run["evidence"].groupby("pair_id")["bits"].sum()
        pairs = self.run["pairs"].set_index("pair_id")["bits"]
        diff = (pairs.reindex(ev.index) - ev).abs()
        self.assertLess(float(diff.max()), 1e-6)
        zero = pairs[~pairs.index.isin(ev.index)]
        self.assertTrue((zero.abs() < 1e-9).all())

    def test_agreement_never_lowers_p(self):
        part = "person"
        S, L = self.run["scored"][part][0], self.run["scored"][part][1]
        w = self.run["params"]["weights"][part]
        lv = L.copy()
        base = score_pairs(lv, part, w, S["prior_logit"].to_numpy())
        for f in ("ssn", "npi", "email", "dob", "name"):
            lv2 = lv.copy()
            empty = lv2[f].to_numpy() < 0
            lv2.loc[empty, f] = 0
            lv2.loc[empty, f"uv_{f}"] = 1e-6
            more = score_pairs(lv2, part, w, S["prior_logit"].to_numpy())
            ok = base["veto"].to_numpy() == ""
            self.assertTrue((more["p"].to_numpy()[ok] >= base["p"].to_numpy()[ok] - 1e-12).all(), f)

    def test_veto_visible_with_p_zero(self):
        P = self.run["pairs"]
        v = P[P["veto"] != ""]
        self.assertGreater(len(v), 0)
        self.assertTrue((v["p"] == 0).all())

    def test_name_only_never_identifier(self):
        part = "person"
        S, L = self.run["scored"][part][0], self.run["scored"][part][1]
        idn = S["basis"].to_numpy() == "identifier"
        any_id = np.zeros(len(L), bool)
        for f in IDENTIFIER_FIELDS:
            if f in L:
                any_id |= (L[f].to_numpy() == 0) & (S[f"bits_{f}"].to_numpy() > 0)
        self.assertTrue((any_id[idn]).all())
        self.assertTrue((S.loc[~any_id, "basis"] != "identifier").all())

    def test_coparty_needs_anchor_and_name(self):
        S, L = self.run["scored"]["person"][0], self.run["scored"]["person"][1]
        cp = S["bits_co_party"].to_numpy() > 0
        self.assertTrue((S["bits_name"].to_numpy()[cp] > 0).all())
        self.assertTrue((L["co_party"].to_numpy()[cp] == 0).all())
        anchors = business_anchors(self.run["scored"]["business"][0], _env()[0])
        X, W = self.run["pr"].X["person"], self.run["pr"].W["person"]
        tl, tr = X["tie_pos"].to_numpy()[S["l"].to_numpy()[cp]], W["tie_pos"].to_numpy()[S["r"].to_numpy()[cp]]
        self.assertTrue(np.isin(tl.astype(np.int64) * np.int64(1 << 32) + tr, anchors).all())

    def test_deterministic(self):
        cfg, maps, ref = _env()
        X, W, truth, edges = _T["inputs"]
        again = run_pipeline(X, W, maps, ref, cfg)
        a = self.run["pairs"].drop(columns=["_l", "_r"]).reset_index(drop=True)
        b = again["pairs"].drop(columns=["_l", "_r"]).reset_index(drop=True)
        pd.testing.assert_frame_equal(a, b)

    def test_edge_cases(self):
        X, W, truth, edges = _T["inputs"]
        r = check_edge_cases(self.run["pr"], self.run["pairs"], None, self.run["scored"], self.run["entities"], edges)
        self.assertTrue(r["ok"].all(), r[~r["ok"]].to_string())

    def test_every_true_name_only_pair_visible(self):
        t = [x for x in self.run["diag"]["truth"] if x.get("true_pairs")]
        for x in t:
            self.assertEqual(x["proposed"], x["kept"], x["part"])

    def test_entities_one_row_per_party(self):
        e = self.run["entities"]
        self.assertTrue(e["party_id"].is_unique)
        n = sum(len(self.run["pr"].X[p]) for p in ("person", "business"))
        self.assertEqual(len(e), n)


class TestParameters(unittest.TestCase):
    def test_u_closed_form(self):
        lv = pd.DataFrame({"email": np.array([0] * 3 + [1] * 97 + [-1] * 50, dtype=np.int8)})
        u = estimate_u(lv, ["email"], CoreParams(u_pseudo=0.0)).set_index("level")["u"]
        self.assertAlmostEqual(u["exact"], 0.03)
        self.assertAlmostEqual(u["differs"], 0.97)

    def test_anchor_field_excluded(self):
        lv = pd.DataFrame({"ssn": np.array([0, 0, 0, 2], dtype=np.int8), "dob": np.array([0, 0, 4, 4], dtype=np.int8)})
        c = level_counts(lv, ["ssn", "dob"], exclude={"ssn": np.array([True, True, True, False])})
        self.assertEqual(c["ssn"][1], 1.0)
        self.assertEqual(c["dob"][1], 4.0)

    def test_em_recovers_m(self):
        rng = np.random.default_rng(3)
        n, lam = 20000, 0.3
        m = {"a": np.array([0.8, 0.15, 0.05]), "b": np.array([0.7, 0.3]), "c": np.array([0.9, 0.1])}
        u = {"a": np.array([0.01, 0.09, 0.9]), "b": np.array([0.05, 0.95]), "c": np.array([0.1, 0.9])}
        match = rng.random(n) < lam
        data = {}
        for f in m:
            k = len(m[f])
            data[f] = np.where(match, rng.choice(k, n, p=m[f]), rng.choice(k, n, p=u[f])).astype(np.int8)
        lv = pd.DataFrame(data)
        saved = {f: FIELD_LEVELS.get(f) for f in m}
        FIELD_LEVELS.update({"a": ["x", "y", "z"], "b": ["x", "y"], "c": ["x", "y"]})
        try:
            est, lam_hat, _ = em_fixed_u(lv, ["a", "b"], ["c"], {"c": m["c"]}, u, CoreParams())
        finally:
            for f, v in saved.items():
                if v is None:
                    FIELD_LEVELS.pop(f, None)
        self.assertAlmostEqual(lam_hat, lam, delta=0.02)
        self.assertLess(np.abs(est["a"][0] - m["a"]).max(), 0.03)
        self.assertLess(np.abs(est["b"][0] - m["b"]).max(), 0.03)

    def test_combine_chain(self):
        p = CoreParams(alpha=5, n_min=50)
        rows = combine_m("email", [("anchors_strict", np.array([8.0, 2.0]), 10.0),
                                   ("simulation", np.array([900.0, 100.0]), 1000.0)], [0.7, 0.3], p)
        m = rows[0]["m"]
        sim = (900 + 5 * 0.7) / (1000 + 5)
        self.assertAlmostEqual(m, (8 + 5 * sim) / 15)
        rows = combine_m("email", [("anchors_strict", np.array([80.0, 20.0]), 100.0),
                                   ("simulation", np.array([0.0, 1000.0]), 1000.0)], [0.7, 0.3], p)
        self.assertAlmostEqual(rows[0]["m"], (80 + 5 * 0.7) / 105)       # simulation skipped: enough pairs
        self.assertEqual(rows[0]["m_chain"], "anchors_strict(100)")

    def test_prior_formula(self):
        p = CoreParams()
        self.assertAlmostEqual(prior_estimate(10, 1_000_000, 0.5, p)[0], 2e-5)
        v, flagged = prior_estimate(0, 1_000_000, 0.5, p)
        self.assertTrue(flagged)
        self.assertAlmostEqual(v, 1e-6)
        self.assertAlmostEqual(strict_recall({"ssn": 0.5}, lambda f, l: 0.9), 0.45)

    def test_manifest_has_every_parameter(self):
        run = _small_run()
        W = pd.concat([run["params"]["weights"]["person"], run["params"]["weights"]["business"]])
        self.assertTrue((W["m_chain"] != "").all() and W["m_pairs"].notna().all())
        self.assertTrue((W["u_source"].fillna("") != "").all())
        n_levels = sum(len(FIELD_LEVELS[f]) for p in ("person", "business") for f in PART_FIELDS[p])
        self.assertEqual(len(W), n_levels)
        self.assertTrue((run["prior_df"]["prior"] > 0).all())


class TestOutput(unittest.TestCase):
    def test_continuation_sheets(self):
        import openpyxl
        with tempfile.TemporaryDirectory() as d:
            path = Path(d) / "t.xlsx"
            df = pd.DataFrame({"a": np.arange(25), "b": ["x"] * 25, "c": [True] * 25, "d": [np.nan] * 25})
            written = write_workbook(path, {"cand": df, "manifest": df.head(2)}, row_limit=10)
            self.assertEqual(written["cand"], ["cand", "cand (2)", "cand (3)"])
            wb = openpyxl.load_workbook(path, read_only=True)
            self.assertEqual(wb.sheetnames, ["cand", "cand (2)", "cand (3)", "manifest"])
            rows = list(wb["cand (3)"].iter_rows(values_only=True))
            self.assertEqual(rows[0], ("a", "b", "c", "d"))
            self.assertEqual(len(rows), 6)
            wb.close()

    def test_manifest_sections(self):
        cfg, maps, ref = _env()
        run = _small_run()
        sw = Stopwatch(); sw.mark("t")
        m = build_manifest(cfg, {"x": "0"}, ref, ({"source": "x", "rows": 1, "all_empty_columns": [], "added_columns": [],
                                                   "passthrough_columns": []},), run["pr"], run["params"],
                           run["prior_df"], sensitivity(pd.DataFrame({"part": [], "prior_group": [], "bits": []}),
                                                        run["prior_df"], cfg), run["scored"], run["diag"], sw)
        for s in ("config", "config.core", "reference", "m", "u", "prior", "blocking", "data_quality", "timing"):
            self.assertIn(s, set(m["section"]), s)


class TestLeieExport(unittest.TestCase):
    def test_export(self):
        raw = ('LASTNAME,FIRSTNAME,MIDNAME,BUSNAME,GENERAL,SPECIALTY,UPIN,NPI,DOB,ADDRESS,CITY,STATE,ZIP,EXCLTYPE,EXCLDATE,REINDATE,WAIVERDATE,WVRSTATE\n'
               '"DOE","JANE","Q","","PHYSICIAN (MD, DO)","FAMILY PRACTICE","","1234567893","19600102","12 W MAIN ST, APT 4","ALBANY","NY","12207","1128b4","20200101","00000000","00000000",""\n'
               '"","","","ACME DME, INC","DME COMPANY","DME - GENERAL","","1234567893","","P O BOX 7","TROY","NY","12180","1128a1","20190101","00000000","00000000",""\n')
        with tempfile.TemporaryDirectory() as d:
            p = Path(d) / "raw.csv"
            p.write_text(raw, encoding="utf-8")
            out = export_leie(p)
        self.assertEqual(list(out.columns[:len(SCHEMA)]), SCHEMA)
        a, b = out.iloc[0], out.iloc[1]
        self.assertEqual((a["provider_npi"], a["clinic_npi"], b["provider_npi"], b["clinic_npi"]),
                         ("1234567893", "", "", "1234567893"))
        self.assertEqual(a["dob"], "1960-01-02")
        self.assertEqual((a["street_number"], a["street_direction"], a["street_name"], a["street_type"], a["unit"]),
                         ("12", "W", "MAIN", "ST", "4"))
        self.assertEqual(a["professional_license_type"], "PHYSICIAN (MD, DO)")
        self.assertEqual(a["x_excldate"], "2020-01-01")
        self.assertTrue(out["record_id"].is_unique and out["record_id"].str.startswith("leie:").all())


class TestInputs(unittest.TestCase):
    def test_validation(self):
        with self.assertRaises(InputError):
            validate_input(pd.DataFrame({"record_id": ["a", "a"]}), "t")
        with self.assertRaises(InputError):
            validate_input(pd.DataFrame({"first_name": ["a"]}), "t")
        df, rep = validate_input(pd.DataFrame({"record_id": ["a"], "odd": ["1"]}), "t")
        self.assertIn("x_odd", df.columns)
        self.assertIn("ssn", rep["all_empty_columns"])

    def test_mappings_valid(self):
        cfg, maps, ref = _env()
        self.assertEqual(set(ref["hash_checks"]["status"]), {"ok"})
        self.assertGreater(len(maps["category_map"]), 280)


CORE_SHA256 = "f51d969c9f7efe34c4b2abbd416542003d79308ab12694a09ce24852591a2c48"


NL, CR = chr(10), chr(13)


def notebook_core_hash(nb_path):
    """sha256 over the code cells between the 'Matching core v' heading and its end marker,
    exactly as recorded when the notebook was assembled."""
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    inside, parts = False, []
    for c in nb["cells"]:
        src = "".join(c["source"])
        if c["cell_type"] == "markdown" and src.startswith("## Matching core v"):
            inside = True
            continue
        if c["cell_type"] == "markdown" and src.strip().startswith("*End of the matching core.*"):
            break
        if inside and c["cell_type"] == "code":
            parts.append(NL.join(l.rstrip() for l in src.replace(CR + NL, NL).split(NL)))
    return hashlib.sha256((NL + "# ---- cell ----" + NL).join(parts).encode("utf-8")).hexdigest()


class TestCoreHash(unittest.TestCase):
    def test_core_section_matches_recorded_hash(self):
        nbp = _env()[0].paths["notebook"]
        if not nbp.exists() or CORE_SHA256.startswith("<"):
            self.skipTest("runs inside the assembled notebook")
        self.assertEqual(notebook_core_hash(nbp), CORE_SHA256,
                         f"the matching core v{CORE_VERSION} changed: bump the version, re-record the hash, copy to [A]")


def run_selftests(verbosity=1):
    suite = unittest.TestSuite()
    loader = unittest.TestLoader()
    for cls in (TestNormalizers, TestBreakdown, TestCategory, TestComparisons, TestCandidates, TestScoring,
                TestParameters, TestOutput, TestLeieExport, TestInputs, TestCoreHash):
        suite.addTests(loader.loadTestsFromTestCase(cls))
    stream = io.StringIO()
    res = unittest.TextTestRunner(stream=stream, verbosity=verbosity).run(suite)
    return res, stream.getvalue()

In [36]:
SELFTEST, SELFTEST_TEXT = run_selftests(verbosity=1)
print(SELFTEST_TEXT[-3000:])
print(f"self-tests: {SELFTEST.testsRun} run, {len(SELFTEST.failures)} failures, {len(SELFTEST.errors)} errors, "
      f"{len(SELFTEST.skipped)} skipped; matching core v{CORE_VERSION} sha256 {CORE_SHA256}")
assert SELFTEST.wasSuccessful(), "self-tests failed"

[    0.0s  peak  0.79 GB] t
...................................................
----------------------------------------------------------------------
Ran 51 tests in 37.851s

OK

self-tests: 51 run, 0 failures, 0 errors, 0 skipped; matching core v1.0 sha256 f51d969c9f7efe34c4b2abbd416542003d79308ab12694a09ce24852591a2c48


## Acceptance (PLAN.md section 5)

Each criterion measured on this run where the dataset allows it: the edge cases and blocking
recall on `synthetic`, the LEIE targets on `leie`, the time and memory targets on `scale`.
Nothing here changes a result; it reports.

In [37]:
def acceptance_table(cfg, diag, sw, pairs, scored, selftest):
    rows = []
    add = lambda c, target, value, met: rows.append({"criterion": c, "target": target, "value": value,
                                                    "met": "yes" if met else ("n/a" if met is None else "NO")})
    add("self-tests pass", "all", f"{selftest.testsRun} run, {len(selftest.failures) + len(selftest.errors)} failing",
        selftest.wasSuccessful())
    total = sw.rows[-1]["elapsed"] if sw.rows else float("nan")
    gen = next((r["elapsed"] for r in sw.rows if r["step"] == "inputs ready"), 0.0)
    total = total - gen          # making the test inputs is not part of a run
    add("input generation / export (not counted)", "-", f"{gen:.0f} s", None)
    truth = {t["part"]: t for t in diag["truth"] if t.get("true_pairs")}
    tp = sum(t["true_pairs"] for t in truth.values())
    prop = sum(t["proposed"] for t in truth.values())
    kept = sum(t["kept"] for t in truth.values())
    rec = prop / tp if tp else float("nan")
    if cfg.dataset == "synthetic":
        add("pipeline time (notebook check < 2 min incl. self-tests: see run_notebook_check.py)", "< 120 s",
            f"{total:.0f} s pipeline", total < 120)
        add("blocking recall on true synthetic pairs", ">= 0.99", f"{rec:.4f} ({prop}/{tp})", rec >= 0.99)
        from_edges = check_edge_cases(PR, PAIRS, None, SCORED, ENTITIES, edge_cases())
        add("every hand-written case: expected basis, veto, top rank", "all",
            f"{int(from_edges['ok'].sum())}/{len(from_edges)}", bool(from_edges["ok"].all()))
    if cfg.dataset == "leie":
        add("LEIE run time", "< 300 s", f"{total:.0f} s", total < 300)
        add("LEIE true pairs proposed", ">= 0.98", f"{rec:.4f} ({prop}/{tp})", rec >= 0.98)
    if cfg.dataset == "scale":
        add("scale run time", "< 3600 s", f"{total:.0f} s", total < 3600)
        add("scale peak memory", "< 10 GB", f"{sw.peak / 1e9:.2f} GB", sw.peak < 10e9)
        add("scale true pairs proposed (diagnostic)", "-", f"{rec:.4f} ({prop}/{tp})", None)
    if tp:
        add("every true pair proposed is visible (kept), incl. name-only", "kept = proposed", f"{kept}/{prop}", kept == prop)
    W_ = pd.concat([PARAMS["weights"]["person"], PARAMS["weights"]["business"]])
    add("every parameter in the manifest with source and count", "all m/u rows",
        f"{len(W_)} level rows, {int((W_['m_chain'] == '').sum())} without m source",
        bool((W_["m_chain"] != "").all() and (W_["u_source"].fillna("") != "").all()))
    add("scored pairs", "-", f"{sum(diag['scored_pairs'].values()):,}", None)
    return pd.DataFrame(rows)


ACCEPTANCE = acceptance_table(CFG, DIAG, SW, PAIRS, SCORED, SELFTEST)
print(ACCEPTANCE.to_string(index=False))
(PATHS["out"] / "acceptance.json").write_text(ACCEPTANCE.to_json(orient="records", indent=1), encoding="utf-8")

                                                                         criterion          target                             value met
                                                                   self-tests pass             all                 51 run, 0 failing yes
                                           input generation / export (not counted)               -                               1 s n/a
pipeline time (notebook check < 2 min incl. self-tests: see run_notebook_check.py)         < 120 s                     37 s pipeline yes
                                           blocking recall on true synthetic pairs         >= 0.99                0.9921 (2135/2152) yes
                           every hand-written case: expected basis, veto, top rank             all                             53/53 yes
                       every true pair proposed is visible (kept), incl. name-only kept = proposed                         2135/2135 yes
                             every parame

1060

## 18 · Review sample (later phase: not built)

This section is a skeleton. The plan's milestone M6 adds:

1. **Blind stratified sample.** From the candidates sheet, strata by basis x p band x prior group,
   a fixed number per stratum, drawn with the run seed; the reviewer's sheet shows both records
   side by side without p, basis or bits.
2. **Labels read back** from `review/` (gitignored) with openpyxl: match / not a match / cannot
   tell, with a reason.
3. **Metrics**: precision per stratum and overall with the stratum weights, recall against the
   labelled matches that the candidates reached, and a calibration table (predicted p vs
   observed share) with explicit denominators (AGENTS rule 12).

Labels are never read by the pipeline (AGENTS rule 13): they measure it, they do not tune it.

**Where Splink would do better.** Splink can take labels as a table and plot ROC and
precision-recall curves and threshold selection charts from them directly.
